# DATA-DRIVEN 1D MECHANICAL EARTH MODEL
## INCREMENT 3 — DEVIATION SURVEY & DEPTH FRAMEWORK

**Author:** Mikael Elgo
**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D MEM. This notebook is a portfolio and educational workflow. It is **not** calibrated for operational drilling, well design, casing design, or real-world mud-weight decisions.

**Locked foundation:** Increment 2.1.1 (`p2mem` 0.2.1) — LAS ingestion and per-file curve-contract resolution for the four approved wells — has passed independent technical review (158 tests, 4/4 real wells loading, zero ingestion errors/warnings, 20/20 notebook cells matching packaged source) and is treated as **LOCKED**. `p2mem/units.py`, `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, and `config/las_curve_contracts.yml` are **not modified** in this notebook.

**Corrective patch note (v3.1):** an independent audit of the original Increment 3 (`p2mem` 0.3.0) found four defects, all corrected in this `p2mem` 0.3.1 revision without any change to scope, equations, or the locked foundation: (1) the four deviation-survey filenames were incorrectly keyed with underscores substituted for spaces (`Poseidon_2_dev.txt` instead of the real `Poseidon 2_dev.txt`), which would fail to resolve against the actual files in Google Drive — corrected everywhere (contract keys, this notebook's `DEV_FILES` mapping, tests, outputs); (2) exported CSV/JSON deliverables could embed a full, environment-dependent build path in a diagnostic `context`/`error_message` field — every exported field now carries a basename only; (3) the supplied `DLS` column's degrees-per-30-metres normalization, while scientifically defensible, was not disclosed as an inference the way the pre-existing MD-unit inference was — a new, independently-verified `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` warning is now raised, with the actual computed discrepancy, for every successfully loaded file; (4) the dogleg-angle computation used `arccos`, which is ill-conditioned near a zero dogleg and produced a spurious ~1e-6-degree value for two stations with identical inclination/azimuth — replaced with a numerically stable vector (`arctan2`-based) formulation that reports exactly `0.0` in that case. See `INCREMENT_03_1_MANIFEST.md` for the full audit and re-verification record. The real four-well data, station counts, tolerances, and the unresolved Proteus 1ST2 trajectory discrepancy are all **unchanged** by this patch.

**Packaging patch note (v3.1.1):** a real Google Colab `Run all` from a fresh runtime (not just static notebook/source parity checking) exposed a working-directory execution-order defect: Colab starts in `/content`, not `PROJECT_ROOT`, and the previous revision of this notebook did not enter `PROJECT_ROOT` until a `%cd` cell well after the first relative `%%writefile` cell had already failed. Step 2b below now enters and verifies `PROJECT_ROOT` immediately after the locked-foundation check and strictly before any relative write. This is a packaging/notebook-execution-order correction only — no scientific or computational code, contract, test, output, or figure changed, and `p2mem` remains version 0.3.1. See `INCREMENT_03_1_1_MANIFEST.md` for the full audit and re-verification record.

### Technical Objective

Add a validated deviation-survey and depth-reference layer on top of the locked LAS layer, for the same four wells (Poseidon 2, Boreas 1, Poseidon North 1, Proteus 1ST2):

1. Strict, auditable ingestion of each well's Petrel deviation-survey (well-trace) text file, with an explicit per-file contract.
2. Survey-station quality control (monotonic MD, physically valid inclination, no silent repair).
3. An explicit, transparent implementation of the standard minimum-curvature trajectory method.
4. An independent comparison of that computed trajectory against the Petrel-supplied source trajectory (TVD, X, Y, Z, DX, DY) — reported, never hidden, never forced to agree.
5. An explicit depth-reference framework (MD, TVD, TVDSS, sign/datum conventions) and an auditable, per-well policy for which trajectory (Petrel-supplied or independently computed) is used downstream.
6. Mapping of the already-validated Increment 2.1.1 LAS `MD_m` arrays onto TVD and TVDSS, with no silent extrapolation beyond surveyed coverage.

**Explicitly NOT implemented in this increment:** checkshot ingestion, time-depth conversion, formation-top correction, petrophysical interpretation, gamma-ray normalization, shale-volume calculation, lithology classification, normal-compaction-trend fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, wellbore-stability calculations, or operational mud-weight recommendations. Those belong to later, explicitly gated increments.

### Theory and Physical Basis

**Minimum curvature.** For two consecutive survey stations 1 and 2 with inclination *I* (from vertical) and azimuth *A* (clockwise from north), the dogleg angle β between their tangent directions is:

$$\cos\beta = \cos I_1 \cos I_2 + \sin I_1 \sin I_2 \cos(A_2 - A_1)$$

This remains the definition `p2mem.trajectory` implements, but (as of this v3.1 corrective patch) β is not computed by evaluating this right-hand side and taking `arccos` of it — that formulation is ill-conditioned as β → 0 (its derivative diverges there), producing a spurious ~1e-6-degree dogleg for two stations with numerically identical inclination and azimuth. Instead, β is computed as the angle between the two stations' 3-D tangent unit vectors $u=(\cos I, \sin I\cos A, \sin I \sin A)$ via $\beta = \operatorname{atan2}(\lVert u_1 \times u_2\rVert,\, u_1\cdot u_2)$ — mathematically identical for every β in $[0,\pi]$, but numerically well-conditioned everywhere, including exactly at β = 0.

Minimum curvature fits a single constant-curvature circular arc between the two station directions (as opposed to the tangential or average-angle methods), using the ratio factor:

$$RF = \frac{2}{\beta}\tan\left(\frac{\beta}{2}\right) \xrightarrow{\beta \to 0} 1$$

so that the displacement over an interval of measured-depth length ΔMD is:

$$\Delta TVD = \frac{\Delta MD}{2}(\cos I_1 + \cos I_2)\,RF \qquad
\Delta N = \frac{\Delta MD}{2}(\sin I_1\cos A_1 + \sin I_2\cos A_2)\,RF \qquad
\Delta E = \frac{\Delta MD}{2}(\sin I_1\sin A_1 + \sin I_2\sin A_2)\,RF$$

and dogleg severity, in the oilfield-conventional degrees per 30 m, is $DLS_{deg/30m} = (\beta_{deg} / \Delta MD) \times 30$.

**Depth reference.** For these files, MD and TVD are both referenced to zero at the well datum (rotary table) and increase downward; the datum elevation itself is referenced to mean sea level (MSL), positive upward. The Petrel-supplied elevation coordinate is $Z_m = DatumElevation_m - TVD_m$ (positive upward), so:

$$TVDSS_m = TVD_m - DatumElevation_m = -Z_m$$

Both the minimum-curvature computation and the depth-reference sign conventions above are implemented in `p2mem/trajectory.py` and `p2mem/depth_mapping.py` respectively, and are independently verified in `tests/test_trajectory.py` and `tests/test_depth_mapping.py` against closed-form and cross-checked (not merely self-referential) analytical cases before this notebook touches any real project file.

> **QUALITY-CONTROL NOTE:** this notebook reports BOTH the Petrel-supplied source trajectory and this project's independently computed minimum-curvature trajectory for every well, side by side, with an explicit residual comparison. Where they disagree beyond a declared tolerance, that disagreement is reported as a visible finding — never silently corrected, hidden, or used to justify loosening every well's tolerance after the fact.

### Input Data and Contracts

#### Step 1 — Mount Google Drive

**Technical objective:** re-attach the persistent project folder from prior increments.

**Inputs/outputs:** no project inputs are read here; this only establishes the `/content/drive` mount point.

**Failure behavior:** if the authorization prompt is declined, every subsequent cell that reads/writes under `/content/drive/MyDrive/...` fails with a clear "no such file or directory" error — there is no silent fallback to local VM storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Step 2 — Verify the Increment 2.1.1 foundation is present and locked

**Technical objective:** confirm that the locked LAS-ingestion layer (`p2mem/io/las.py`, `p2mem/models.py`, `config/las_curve_contracts.yml`) and its real four-well outputs (`outputs/02_las_inventory/`) already exist in this Drive project folder, BEFORE this notebook adds anything on top. Increment 3 is explicitly instructed not to modify these files unless an actual blocking defect is demonstrated — none was found or is claimed here.

**Inputs:** the existing `/content/drive/MyDrive/Poseidon_1D_MEM/` folder.
**Outputs:** a pass/fail confirmation printed to the cell output; no files are written by this cell.

**Failure behavior:** raises `RuntimeError` naming exactly which expected file is missing, and instructs the user to run the Increment 2.1.1 notebook first. This is a hard gate, not a warning.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/Poseidon_1D_MEM"

_required_locked_files = [
    "pyproject.toml",
    os.path.join("p2mem", "__init__.py"),
    os.path.join("p2mem", "units.py"),
    os.path.join("p2mem", "models.py"),
    os.path.join("p2mem", "io", "las.py"),
    os.path.join("p2mem", "io", "inventory.py"),
    os.path.join("config", "las_curve_contracts.yml"),
]
_missing = [f for f in _required_locked_files if not os.path.exists(os.path.join(PROJECT_ROOT, f))]
if _missing:
    raise RuntimeError(
        "Locked Increment 2.1.1 foundation is missing file(s): "
        + ", ".join(_missing)
        + ". Run the Increment 2.1.1 notebook first; Increment 3 does not "
        "reconstruct the locked LAS-ingestion layer from memory."
    )
print("Locked Increment 2.1.1 foundation confirmed present:")
for f in _required_locked_files:
    print("  -", f)

#### Step 2b — Enter the project root before any relative file write

**Increment 3.1.1 correction:** a real Google Colab `Run all` from a fresh runtime exposed an execution-order defect not caught by static notebook/source parity checking. Colab's working directory starts as `/content`, not `PROJECT_ROOT`. Every `%%writefile` cell below (Step 6 onward) writes a **relative** path (e.g. `p2mem/__init__.py`), and the previous revision of this notebook did not change the working directory to `PROJECT_ROOT` until a later `%cd` cell (originally in Step 8, just before the editable install) - well after those relative writes had already been attempted. On a genuinely fresh runtime this fails immediately with `FileNotFoundError: [Errno 2] No such file or directory: 'p2mem/__init__.py'`, requiring the user to manually insert an `os.chdir(PROJECT_ROOT)` and rerun. This cell makes that manual step unnecessary by performing the working-directory change here - immediately after `PROJECT_ROOT` is defined and the locked foundation is confirmed present (Step 2), and strictly before the first relative `%%writefile` cell (Step 6).

**Technical objective:** change into `PROJECT_ROOT`, then verify (not assume) that the change succeeded and that the approved `p2mem` foundation is actually present there, before any relative write is attempted.

**Failure behavior:** raises `RuntimeError` if the working directory does not actually resolve to `PROJECT_ROOT` after the `chdir` call, or if `p2mem` is not a directory at that location. Neither case is silently papered over - in particular, this cell never creates an empty substitute `p2mem` directory to make the check pass; a missing foundation is always a hard stop, directing the user to extract the approved Increment package first.

**Note on this design vs. the later `%cd` cell in Step 8:** that cell (`%cd /content/drive/MyDrive/Poseidon_1D_MEM`, immediately before `pip install -q -e .`) is retained unchanged as a defensive, explicit second confirmation - it is redundant given this cell, but harmless (re-entering a directory the kernel is already in), and removing it is not necessary for correctness. It is not, and must never be, the first working-directory change in this notebook.

In [ ]:
os.chdir(PROJECT_ROOT)

if Path.cwd().resolve() != Path(PROJECT_ROOT).resolve():
    raise RuntimeError(
        f"Failed to enter the project root. "
        f"Expected {PROJECT_ROOT}, actual working directory: {Path.cwd()}"
    )

if not Path("p2mem").is_dir():
    raise RuntimeError(
        f"Required p2mem directory is missing under {PROJECT_ROOT}. "
        "Extract the approved Increment package before running this notebook."
    )

print("Working directory confirmed:", Path.cwd())

#### Step 3 — Create the Increment 3 directory additions

**Technical objective:** create the new directories this increment needs, without touching any existing Increment 1.1/2.1.1 directory.

**Expected result:** `data/raw/deviation/`, `p2mem` (existing), `outputs/03_deviation_depth/` and `outputs/03_deviation_depth/figures/` exist.

In [ ]:
for d in ["data/raw/deviation", "outputs/03_deviation_depth", "outputs/03_deviation_depth/figures"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print("Increment 3 directories ready.")

#### Step 4 — Verify the four raw deviation-survey filenames are present (exact names only), and their SHA-256

**Technical objective:** require the exact approved filenames — `Poseidon 2_dev.txt`, `Boreas 1_dev.txt`, `Poseidon North 1_dev.txt`, `Proteus 1ST2_dev.txt` — under `data/raw/deviation/`, and stop cleanly, naming every missing file, if any are absent. No substitute well (Poseidon 1, Kronos 1, Pharos 1, or any other) is ever accepted in place of these four.

**Increment 3.1 correction:** these four filenames contain literal spaces, matching the files exactly as they exist in Google Drive. Increment 3 originally substituted underscores for the spaces in this cell, which would have caused every one of these lookups to fail against the real files. The `DEV_FILES` dict keys below (`Poseidon_2`, `Boreas_1`, `Poseidon_North_1`, `Proteus_1ST2`) remain underscored — they are internal, Python-identifier-friendly well keys, never used as filenames — while every dict **value** is the exact, literal, space-containing filename.

**Assumptions:** the user has uploaded these four files to `/content/drive/MyDrive/Poseidon_1D_MEM/data/raw/deviation/` before running this cell (this notebook cannot fetch them itself — they are private project inputs, not published data).

**Failure behavior:** raises `RuntimeError` listing every missing filename by its exact expected name. Never silently proceeds with fewer than four wells, and never substitutes a different well's file.

**Expected result:** all four files found, and their SHA-256 hashes printed for the user's own independent verification (not yet compared against a stored value — that happens after ingestion, as an ERROR-blocking contract check).

In [ ]:
import hashlib

DEV_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "deviation")
DEV_FILES = {
    "Poseidon_2": "Poseidon 2_dev.txt",
    "Boreas_1": "Boreas 1_dev.txt",
    "Poseidon_North_1": "Poseidon North 1_dev.txt",
    "Proteus_1ST2": "Proteus 1ST2_dev.txt",
}

_missing_dev = [fn for fn in DEV_FILES.values() if not os.path.exists(os.path.join(DEV_DIR, fn))]
if _missing_dev:
    raise RuntimeError(
        "Missing required deviation-survey file(s) under "
        f"{DEV_DIR}: {_missing_dev}. Increment 3 requires exactly these four "
        "approved wells' files, under these exact names - no other well is "
        "accepted as a substitute."
    )

print("All four approved deviation-survey files found. SHA-256:")
for key, fn in DEV_FILES.items():
    p = os.path.join(DEV_DIR, fn)
    print(f"  {key} ({fn}): {hashlib.sha256(open(p, 'rb').read()).hexdigest()}")

#### Step 5 — Install dependencies

**Technical objective:** install exactly the packages this increment's code and notebook display/plotting cells need. `matplotlib` is new in this notebook (QC figures) but is NOT added as a runtime dependency of the installable `p2mem` package itself (see `pyproject.toml` - only NumPy and PyYAML are runtime dependencies).

In [ ]:
!pip install -q numpy pyyaml pytest pandas matplotlib

### Input Data and Contracts

#### Step 6 — Write the Increment 3 package files

**Technical objective:** write every new/updated source file for this increment, verbatim from the tested files on disk (this build script never hand-retypes code into notebook cells - every `%%writefile` body below is read directly from the file that was actually run through `pytest`).

**New modules:** `p2mem/deviation_models.py` (typed dataclasses), `p2mem/trajectory.py` (minimum-curvature engine), `p2mem/depth_mapping.py` (MD-to-TVD/TVDSS interpolation), `p2mem/io/deviation.py` (Petrel deviation-file parser and contract resolver), `p2mem/io/deviation_inventory.py` (deterministic output-table builders).

**Updated (version/documentation only):** `pyproject.toml`, `p2mem/__init__.py`, `README.md` — package version bumped to 0.3.1 (Increment 3.1 corrective patch); the locked LAS-layer files (`p2mem/units.py`, `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, `config/las_curve_contracts.yml`) are intentionally NOT rewritten here.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68.0"]
build-backend = "setuptools.build_meta"

[project]
name = "p2mem"
version = "0.3.1"
description = "Screening-level 1D Mechanical Earth Model workflow for Poseidon 2 (Tier C, uncalibrated / educational)."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "All Rights Reserved. Copyright (c) 2026 Mikael Elgo. This is a personal portfolio project; no license is granted for reuse, redistribution, or commercial use without the author's explicit written permission." }
authors = [
    { name = "Mikael Elgo" }
]
keywords = ["geomechanics", "mechanical-earth-model", "pore-pressure", "wellbore-stability", "portfolio-project"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Programming Language :: Python :: 3",
    "Intended Audience :: Science/Research",
    "Topic :: Scientific/Engineering",
    "License :: Other/Proprietary License",
]

# Runtime dependencies are deliberately minimal. No unit-handling libraries
# (e.g. Pint) are used: unit conversions are implemented explicitly in
# p2mem.units so that every conversion factor is visible, documented, and
# testable rather than delegated to a third-party unit registry. PyYAML is
# added in Increment 2 for exactly one purpose: parsing the human-authored,
# human-reviewable per-file LAS curve contracts in
# config/las_curve_contracts.yml - a plain-text, diffable format was judged
# preferable to a hand-rolled config parser or a hard-coded Python dict.
dependencies = [
    "numpy>=1.24",
    "pyyaml>=6.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
]

[tool.setuptools.packages.find]
include = ["p2mem*"]

[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]


In [ ]:
%%writefile p2mem/__init__.py
"""
p2mem - Poseidon 2 1D Mechanical Earth Model workflow package.

Project classification: Tier C - Screening-Level / Uncalibrated Educational
1D Mechanical Earth Model (see project design review, Rev 1). Nothing in
this package should be presented as a calibrated, operational, or
field-validated result unless an explicit independent calibration record
is attached to that specific output.

This package is under incremental, gated construction.

* Increment 1 / 1.1 delivered the project skeleton and the unit-control
  system (``p2mem.units``).
* Increment 2 added an auditable LAS-ingestion layer with explicit
  per-file curve contracts (``p2mem.io.las``, ``p2mem.io.inventory``,
  ``p2mem.models``) for the four approved wells (Poseidon 2, Boreas 1,
  Poseidon North 1, Proteus 1ST2). It performs LAS parsing, curve-identity
  resolution, NULL-sentinel handling, and factual inventory generation
  ONLY - no deviation-survey processing, MD-to-TVD/TVDSS transformation,
  checkshot processing, formation-top correction, petrophysical
  interpretation, or any later-phase geomechanical calculation.
* Increment 2.1 / 2.1.1 are corrective patches to Increment 2, applied
  after independent technical audits, WITHOUT changing scope or the
  underlying LAS-parsing/curve-resolution architecture (which both audits
  found sound). 2.1 corrected: canonical array naming (every array is now
  explicitly unit-suffixed, e.g. ``VP_m_s`` rather than ``DTCO``, so a
  name can never be mistaken for the wrong physical quantity or unit);
  the measured-depth curve is now located via an explicit contract role
  rather than by matching a canonical name spelled "DEPT"; several
  file-identity checks (filename, SHA-256, WELL, VERS, WRAP, NULL) that
  were not previously blocking now are; curve-coverage statistics now
  report raw AND canonical values with explicit units; and per-well
  batch failures are now typed (``p2mem.models.IngestionFailure``)
  instead of bare caught exceptions. 2.1.1 corrected a packaging-only gap
  (three notebook ``%%writefile`` cells that had drifted from their
  packaged source files). See ``INCREMENT_02_v2.1_MANIFEST.md`` and
  ``INCREMENT_02_v2.1.1_MANIFEST.md`` for the full audits and
  corrected-file checksums.
* Increment 3 adds Petrel deviation-survey ingestion with explicit
  per-file contracts (``p2mem.io.deviation``), a standard minimum-
  curvature trajectory engine with a numerically stable ratio-factor
  limit (``p2mem.trajectory``), an explicit MD-referenced/TVD-referenced/
  TVDSS depth-reference framework and MD-to-TVD/TVDSS interpolation with
  no silent extrapolation (``p2mem.depth_mapping``), and typed dataclasses
  for all of the above (``p2mem.deviation_models``) - for the same four
  approved wells. It independently reproduces the Petrel-supplied
  trajectory to millimetre scale for three of the four wells and
  discloses (rather than resolves) a real, larger trajectory-
  reconstruction discrepancy found in Proteus 1ST2's deeper section - see
  ``INCREMENT_03_MANIFEST.md``. It performs deviation-survey ingestion,
  trajectory validation, and depth mapping ONLY - no checkshot
  processing, formation-top correction, petrophysical interpretation, or
  any later-phase geomechanical calculation.
* Increment 3.1 is a corrective patch to Increment 3, applied after an
  independent technical audit, WITHOUT changing scope, equations, real
  well data, or the locked LAS/Increment-2.1.1 foundation. It corrected
  four defects: (1) the four deviation-survey source filenames are the
  exact, literal names as they exist in Google Drive, which contain
  spaces (e.g. ``"Poseidon 2_dev.txt"``) - Increment 3 had incorrectly
  substituted underscores in the contract keys, notebook mapping, and
  tests, which would have failed to resolve against the real files;
  internal well keys (e.g. ``Poseidon_2``) remain underscored and are
  unaffected; (2) every exported CSV/JSON/manifest field is now
  guaranteed to carry a basename only, never a full environment-dependent
  build path (runtime-only diagnostic objects may still retain one);
  (3) the previously undisclosed inference that the supplied ``DLS``
  column is normalized as degrees per 30 metres is now explicitly flagged
  with a new, independently per-file-verified
  ``DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M`` WARNING (mirroring the
  pre-existing MD-unit-inference warning); (4) the dogleg angle between
  successive stations is now computed with a numerically stable
  ``arctan2(||cross||, dot)`` vector formulation (``p2mem.trajectory``)
  instead of ``arccos``, which was ill-conditioned near a zero dogleg and
  previously reported a spurious ~1e-6-degree value for two stations with
  identical inclination/azimuth. The real four-well data, station counts,
  tolerances, and the unresolved Proteus 1ST2 trajectory discrepancy are
  all unchanged by this patch. See ``INCREMENT_03_1_MANIFEST.md`` for the
  full audit and re-verification record.

Subsequent increments (checkshot/time-depth processing, formation-top
correction, petrophysics, pore pressure, elastic properties, strength,
stress, and wellbore-stability screening) are added one validated phase
at a time and are intentionally absent from this version - importing them
will fail until they exist.
"""

__version__ = "0.3.1"

# Fixed project-wide assurance tier. Referenced by later modules (reporting,
# plotting) so that every generated output can stamp its own classification
# without each module re-declaring the string. This value must not be
# changed without a documented calibration event (e.g. a verified RFT/MDT,
# LOT/XLOT, or core-calibrated log tie) recorded in the method-and-citation
# register.
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

__all__ = ["__version__", "ASSURANCE_TIER"]


In [ ]:
%%writefile README.md
# Poseidon 2 — 1D Mechanical Earth Model

**Author:** Mikael Elgo

**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D Mechanical Earth Model (MEM)

> **This project is screening-level, uncalibrated, and educational in nature. It is NOT validated against independent field measurements (no confirmed RFT/MDT pressure points, LOT/XLOT tests, or core-calibrated log ties are currently incorporated), and it must NOT be used for operational drilling, well-design, or any real-world decision-making. It exists to demonstrate a technically defensible, transparent, modular geomechanics workflow — not to produce field-ready predictions.**

---

## Purpose and technical scope

This repository implements a modular, reproducible 1D Mechanical Earth Model workflow for the Poseidon 2 well, built from well logs, deviation surveys, checkshot data, formation tops, and Vp/Vs data supplied for the project. The intended end-to-end scope (delivered incrementally, one validated phase at a time) covers:

- data quality control and depth alignment across LAS logs, deviation surveys, and checkshot data
- pore-pressure prediction (Eaton-family methods, contingent on a defensible normal compaction trend)
- elastic properties (dynamic Vp/Vs-derived Poisson's ratio, and density-dependent moduli where density coverage permits)
- rock-strength estimation
- vertical-stress (overburden) modelling
- horizontal-stress and wellbore-stability screening (Kirsch elastic wall-stress equations with Mohr–Coulomb/Mogi–Coulomb failure criteria)
- uncertainty treatment via deterministic low/base/high scenarios and one-at-a-time sensitivity (tornado) analysis, rather than unsupported probabilistic distributions

Every empirical or correlation-based relationship used anywhere in this project (Eaton, Bowers, Gardner, Castagna, etc.) is required to have a recorded source, stated units, applicability range, and calibration status in the project's method-and-citation register *before* it is implemented in code. Nothing is fabricated or assumed silently: missing measurements, missing calibration points, and unavailable data are always reported as unavailable rather than filled in.

This is a personal portfolio project intended to demonstrate scientific rigor, reproducibility, and honest handling of data limitations — not a commercial or operational deliverable.

## Current implementation status

**Increment 3 / 3.1 (this release, v0.3.1): Deviation-Survey Ingestion, Minimum-Curvature Validation, and MD–TVD–TVDSS Depth Framework.** Builds on the LOCKED Increment 2.1.1 LAS-ingestion layer (unmodified - see below) by adding Petrel deviation-survey parsing, an explicit per-file survey contract, a standard minimum-curvature trajectory engine, an explicit depth-reference (MD/TVD/TVDSS) framework, and MD-to-TVD/TVDSS mapping of the existing LAS `MD_m` arrays. See `INCREMENT_03_MANIFEST.md` for the Increment 3 technical design and real four-well integration results, and `INCREMENT_03_1_MANIFEST.md` for the Increment 3.1 corrective patch (four audit findings: source-filenames-with-spaces, absolute-path leakage, DLS-normalization disclosure, dogleg numerical stability — see "Increment 3.1 update" below). The Proteus 1ST2 trajectory-discrepancy finding is disclosed, not resolved, in either release.

Locked from Increment 2.1.1 (unmodified in Increment 3 unless a blocking defect is documented - none was found):
- `p2mem/units.py` — an explicit, NumPy-based unit-conversion layer (no external unit-registry dependency such as Pint) implementing 21 public conversion functions between oilfield and SI-internal units. Unchanged since Increment 1.1. See the module docstring and `tests/test_units.py`.
- `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, `config/las_curve_contracts.yml` — the auditable LAS 2.0 parser, per-file curve-contract resolver, and inventory builders for the four approved wells, corrected and independently re-verified through Increment 2.1.1 (158 tests passing, 4/4 real wells loading with zero ingestion errors). See `INCREMENT_02_v2.1.1_MANIFEST.md`.

New in Increment 3:
- `p2mem/deviation_models.py` — typed, frozen dataclasses for every deviation-survey/trajectory/depth-mapping result object (header info, per-file contract, raw station data, minimum-curvature result, trajectory-validation residuals, depth-basis selection, typed batch failures, LAS depth-mapping result), mirroring the LAS layer's design philosophy. Every source (`_source_`) array is kept explicitly separate from every independently computed (`_mc_`) array — never overwritten, never mixed.
- `p2mem/io/deviation.py` — an auditable Petrel deviation-survey (well-trace) parser and per-file contract resolver for the same four wells, keyed by their exact, literal source filenames (which contain spaces, e.g. `"Poseidon 2_dev.txt"` — corrected in Increment 3.1; see below). Extracts and preserves the full header block (well/survey identity, wellhead X/Y, datum and its MSL reference, coordinate-reference-system text, declared angle/depth/coordinate conventions) and the exact 11-column station table, with file-identity checks (filename, SHA-256, well/survey identifier, wellhead/datum values, coordinate system, column order, station count, MD coverage) all enforced as blocking `ERROR`s before any trajectory computation is attempted. Also raises two disclosure `WARNING`s for every successfully loaded file: the pre-existing `MD_UNIT_NOT_EXPLICITLY_DECLARED`, and the Increment 3.1 `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` (the supplied `DLS` column's degrees-per-30-m normalization is inferred, not header-declared, and is independently verified per file against a recomputation from that file's own inclination/azimuth).
- `p2mem/trajectory.py` — the standard minimum-curvature method (a numerically stable `arctan2(||cross||, dot)` dogleg-angle formulation — Increment 3.1 correction, see below — a ratio factor with an explicit Taylor-series limit as the dogleg approaches zero, TVD/northing/easting displacement, dogleg severity in degrees per 30 m), implemented explicitly and transparently with no third-party survey-computation library.
- `p2mem/depth_mapping.py` — MD-to-TVD/TVDSS interpolation of the locked LAS `MD_m` array against the explicitly selected depth-trajectory basis, using a documented, deterministic piecewise-linear station interpolation (never a per-sample minimum-curvature recomputation) that never extrapolates silently.
- `p2mem/io/deviation_inventory.py` — deterministic, metadata-only inventory-table builders for the Increment 3 outputs (file inventory, trajectory-validation summary, depth-reference register, LAS depth-mapping summary, ingestion issues, JSON manifest) — never raw per-sample station or LAS arrays, and (Increment 3.1 correction) never a full environment-dependent build path, only a basename.
- `config/deviation_survey_contracts.yml` — the human-authored, human-reviewable per-file deviation-survey contract for each of the four wells, keyed by the exact literal source filename (`"Poseidon 2_dev.txt"`, `"Boreas 1_dev.txt"`, `"Poseidon North 1_dev.txt"`, `"Proteus 1ST2_dev.txt"` — corrected in Increment 3.1), including an explicit, uniformly applied `depth_basis_policy` (`petrel_source_trace`, the conservative default given the Proteus 1ST2 finding below) and residual-comparison tolerances declared once and applied identically to every well (never tuned per well to force a pass/fail outcome).
- **Key real-data finding (disclosed, not resolved):** independent minimum-curvature reconstruction of Poseidon 2, Boreas 1, and Poseidon North 1 agrees with their Petrel-supplied TVD to approximately millimetre scale. Proteus 1ST2 shows a materially larger discrepancy (~0.19 m TVD, ~1.6 m easting at maximum) concentrated in its deeper section (below ~MD 4200 m), even though its own supplied dogleg-severity column is internally consistent with an independent recomputation from its own inclination/azimuth at every station. This is reported as a visible trajectory-validation `WARNING`, not corrected, hidden, or used to justify loosening every well's tolerance — see `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation and the evidence pattern observed. Unaffected by the Increment 3.1 patch.
- Still not implemented: checkshot ingestion, time-depth conversion, formation-top correction, petrophysical interpretation, gamma-ray normalization, shale-volume calculation, lithology classification, normal-compaction-trend fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those are explicitly out of scope for this increment and are added one gated increment at a time in later releases.

## Installation

Requires Python 3.9 or later.

```bash
# from the project root (the directory containing pyproject.toml)
pip install -e .
```

This installs the `p2mem` package in editable mode along with its runtime dependencies: NumPy (`numpy>=1.24`) and, as of Increment 2, PyYAML (`pyyaml>=6.0`) — used for parsing the human-authored curve contracts in both `config/las_curve_contracts.yml` and (new in Increment 3) `config/deviation_survey_contracts.yml`. No new runtime dependency was added in Increment 3: the minimum-curvature engine and depth-mapping interpolation use only NumPy. Matplotlib and pandas are used only for notebook display and QC-figure generation (`run_integration_03.py`, the Increment 3 notebook) — never imported by the installable `p2mem` package itself. To also install the test dependency:

```bash
pip install -e ".[dev]"
```

## Running the tests

```bash
pytest -v
```

The suite in `tests/test_units.py` validates `p2mem/units.py` (unchanged since Increment 1.1) against analytical reference values, round-trip consistency, scalar/array inputs, NaN preservation, and rejection of invalid/nonphysical/ambiguous inputs. The suite in `tests/test_las.py` validates `p2mem/io/las.py` (locked since Increment 2.1.1) against small synthetic LAS fixtures. The suites in `tests/test_trajectory.py`, `tests/test_deviation.py`, and `tests/test_depth_mapping.py` (new in Increment 3; extended in Increment 3.1 with dogleg numerical-stability, DLS-normalization-disclosure, and real-filename-with-spaces regression/negative tests) validate the minimum-curvature engine, the Petrel deviation-survey parser/contract resolver, and the MD-to-TVD/TVDSS mapping respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data. `tests/test_deviation_inventory.py` (new in Increment 3.1) validates that no exported inventory/issues/manifest row, for a successful or a failed well, ever embeds a full environment-dependent build path. None of these suites require the four private/raw project LAS or deviation files, so the full suite runs the same way for anyone who clones this repository. Run the command above and read the reported pass/fail count directly — this document does not assert a fixed expected count, since that must always be read from the actual `pytest` output for the code currently on disk. Real four-well integration validation (which DOES require the raw LAS and deviation files, not included in this repository) is a separate notebook run — see `02_LAS_Ingestion_and_Curve_Contracts.ipynb` and `03_Deviation_Survey_and_Depth_Framework.ipynb`.

## Directory structure

```
Poseidon_1D_MEM/
├── README.md
├── pyproject.toml
├── p2mem/
│   ├── __init__.py
│   ├── models.py
│   ├── units.py
│   ├── deviation_models.py
│   ├── trajectory.py
│   ├── depth_mapping.py
│   └── io/
│       ├── __init__.py
│       ├── las.py
│       ├── inventory.py
│       ├── deviation.py
│       └── deviation_inventory.py
├── tests/
│   ├── test_units.py
│   ├── test_las.py
│   ├── test_trajectory.py
│   ├── test_deviation.py
│   ├── test_depth_mapping.py
│   ├── test_deviation_inventory.py
│   └── fixtures/         (small synthetic LAS + deviation-survey files; no project raw data)
├── config/
│   ├── las_curve_contracts.yml
│   └── deviation_survey_contracts.yml
├── data/
│   └── raw/
│       ├── logs/         (the four raw LAS files - NOT included in this repository; immutable inputs)
│       └── deviation/    (the four raw deviation-survey files, exact filenames contain spaces, e.g. "Poseidon 2_dev.txt" - NOT included in this repository; immutable inputs)
├── notebooks/  (reserved for later increments)
└── outputs/
    ├── 02_las_inventory/       (Increment 2.1.1 real four-well run: CSV/JSON metadata only, no raw log samples)
    └── 03_deviation_depth/     (Increment 3 real four-well run: CSV/JSON metadata + QC figures, no raw station/log samples)
```

`notebooks/` is created empty by the project-setup notebook cell and is not yet populated in-repo (the increment notebooks themselves are delivered as top-level files, e.g. `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, and are meant to be run from Google Drive per their own directory-setup cells).

## Scientific limitations

These limitations are specific to the Poseidon 2 dataset and this project's current increment, and are carried forward here so they are visible outside the conversation in which they were identified:

- **RHOB (bulk density) coverage in Poseidon 2 ends at approximately 5,296.85 m MD.** Sonic and other curves continue deeper, so Vp/Vs and dynamic Poisson's ratio remain computable below that depth, but density-dependent properties (Young's modulus, shear modulus, bulk modulus, acoustic impedance, shear impedance) are unavailable below it unless density is explicitly estimated and flagged as such — never silently substituted.
- **No reliable shale-based normal compaction trend (NCT) exists from Poseidon 2 alone.** A provisional, transferred candidate NCT identified in offset well Poseidon North 1 is a *candidate*, not a validated trend, and must not be presented as calibrated.
- **Independently measured Vp/Vs quality flags:** approximately 3.38% of Poseidon 2 Vp/Vs values fall below 1.5, and approximately 0.53% fall below the physical validity cutoff of √2 (≈1.4142) required for a non-negative dynamic Poisson's ratio.
- **No independent calibration data (RFT/MDT pressure points, LOT/XLOT tests, or core data) has been supplied or incorporated.** Any pore-pressure or stress output in later increments must be presented as a bounded or theoretical estimate, not a validated field prediction.
- **Empirical/correlation equations are not implemented until their governing equation, units, applicability range, and calibration status are recorded in the project's method-and-citation register.** Several candidate methods remain in "pending" status and are intentionally absent from the codebase for that reason, not because they were overlooked.
- Additional open items (offset-well GR/ECGR scale adjudication, missing formation tops for one offset well) are tracked in the project's design-review documentation and gate specific later phases (lithology and pore-pressure), not this increment.
- **Increment 2 update:** LAS ingestion independently reconfirms (does not newly discover, and does not act on) two previously-flagged anomalies from the Rev 1 design review: Boreas 1's ECGR curve (canonical name `ECGR_api`) ranges from approximately −0.0001 to 519.18 API (vs. roughly 5–205 API for the other three wells' GR-family curves) with 96.98% valid coverage; and Proteus 1ST2's LAS log file places its neutron-porosity curve (canonical name `NPHI_pct`) at column position 5 rather than the last position (8) used by the other three wells. Both are reported as ingestion facts (see `outputs/02_las_inventory/`); neither is rescaled, reinterpreted, or otherwise acted on by this increment.
- **Increment 2.1 / 2.1.1 update:** corrective patches addressing independent audits' naming, reporting, contract-validation, and notebook/source-synchronization findings — see `INCREMENT_02_v2.1_MANIFEST.md` and `INCREMENT_02_v2.1.1_MANIFEST.md`. No new scientific finding was made in either patch; the two anomalies above are unaffected and remain open items for a later, explicitly-scoped increment.
- **Increment 3 update:** deviation-survey ingestion independently reconfirms the Petrel-supplied trajectory for Poseidon 2, Boreas 1, and Poseidon North 1 to approximately millimetre scale via minimum curvature, and additionally DISCOVERS (not merely reconfirms) a real trajectory-reconstruction discrepancy in Proteus 1ST2's deeper section (~0.19 m TVD, ~1.6 m easting at maximum, concentrated below ~MD 4200 m) — see "Current implementation status" above and `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation. This is disclosed as an open item, not corrected or hidden; downstream MD-to-TVD/TVDSS mapping for Proteus 1ST2 conservatively uses the Petrel-supplied source trajectory (not the disagreeing minimum-curvature trajectory) as a result.
- **Increment 3.1 update:** a corrective patch addressing an independent audit's findings on source-filename handling, output environment-independence, an undisclosed normalization inference, and dogleg-angle numerical conditioning — see `INCREMENT_03_1_MANIFEST.md` for the full audit and re-verification record. No new scientific finding was made in this patch; the Proteus 1ST2 discrepancy above is unaffected, remains disclosed exactly as before, and was neither corrected nor concealed. The only numerical changes are at the floating-point noise floor of the diagnostic `dogleg_deg`/`dls_deg_per_30m`/TVD-and-offset-residual fields (at most ~9×10⁻¹³ m for TVD, ~7×10⁻¹⁵ m for easting/northing, across all four real wells), with zero change to any well's PASS/WARNING status.

## Screening-level statement

**This 1D Mechanical Earth Model is a screening-level, uncalibrated, educational work product.** It has not been validated against independent field measurements and does not carry the assurance level required for drilling engineering, well design, casing/mud-weight selection, or any other operational decision. Any numerical result produced by this codebase should be read as illustrative of a defensible methodology applied to the available data, not as a certified or field-ready prediction.


#### `p2mem/deviation_models.py` — typed dataclasses for the deviation-survey/trajectory/depth-mapping layer

In [ ]:
%%writefile p2mem/deviation_models.py
"""
p2mem.deviation_models - Typed, documented dataclasses for the
deviation-survey ingestion, minimum-curvature trajectory, and depth-mapping
layer (Increment 3).

Design rationale
-----------------
This mirrors the design philosophy already established in `p2mem.models`
for the LAS-ingestion layer: every ingestion/computation result is a
frozen `dataclasses.dataclass`, never an undocumented tuple, and every
raw/source quantity is kept in a separately named array from any derived
or computed quantity (see the "source vs. computed" naming pattern below).
No third-party schema library is introduced.

This module is data structure only - it performs no I/O, no parsing, no
numerical computation. See:
    `p2mem.io.deviation`   - Petrel deviation-file parsing and per-file
                              contract resolution.
    `p2mem.trajectory`     - minimum-curvature computation
                              (`MinimumCurvatureResult` is defined there,
                              alongside the numerical functions that
                              produce it, and is reused unmodified here).
    `p2mem.depth_mapping`  - MD-to-TVD/TVDSS interpolation.

Source-versus-computed naming
-------------------------------
Every quantity that came directly from the Petrel deviation file is named
with an explicit `_source_` component (e.g. `TVD_source_m`,
`DX_source_m`), and every quantity computed independently by this
project's minimum-curvature implementation is named with an explicit
`_mc_` component (e.g. `TVD_mc_m`, `EASTING_offset_mc_m`). A `_source_`
array is never overwritten by a `_mc_` computation, and the two are never
silently mixed - see `p2mem.trajectory` and `p2mem.io.deviation` for the
computation and residual-comparison logic that keeps them explicitly
separate.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple

import numpy as np

from p2mem.trajectory import MinimumCurvatureResult

__all__ = [
    "DeviationHeaderInfo",
    "DeviationFileContract",
    "DeviationStationData",
    "DeviationIngestionIssue",
    "TrajectoryValidationResult",
    "DepthBasisSelection",
    "DeviationWellResult",
    "DeviationIngestionFailure",
    "LasDepthMappingResult",
]

# Recognized values for `DeviationFileContract.depth_basis_policy` (see
# `p2mem.io.deviation` for where this is applied).
DEPTH_BASIS_PETREL_SOURCE = "petrel_source_trace"
DEPTH_BASIS_MINIMUM_CURVATURE = "minimum_curvature_computed"
VALID_DEPTH_BASIS_POLICIES = (DEPTH_BASIS_PETREL_SOURCE, DEPTH_BASIS_MINIMUM_CURVATURE)

# Recognized values for a residual-comparison "status" field.
STATUS_PASS = "PASS"
STATUS_WARNING = "WARNING"
STATUS_FAIL = "FAIL"


# ---------------------------------------------------------------------------
# Header / provenance
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class DeviationHeaderInfo:
    """
    Everything read from a Petrel deviation-survey text file's leading
    comment-header block, plus file-identity metadata, WITHOUT reading or
    parsing the station data rows.

    Every field here is a LITERAL statement parsed out of the file's own
    header lines (or file-system/hash identity) - never inferred, assumed,
    or filled in. `depth_reference_statement`, `angle_unit_statement`,
    `dx_dy_statement`, and `z_statement` preserve the file's own declared
    convention text verbatim (or a normalized-but-traceable summary of
    it), so a later reader never has to trust an unstated assumption about
    sign, datum, or units.
    """

    source_path: str
    source_filename: str
    sha256: str
    well_name: str
    survey_name: str
    wellhead_x_m: float
    wellhead_y_m: float
    datum_elevation_m: float
    datum_reference: str
    well_type: str
    coordinate_reference_system: str
    depth_reference_statement: str
    angle_unit_statement: str
    dx_dy_statement: str
    z_statement: str
    column_names: Tuple[str, ...]
    header_line_count: int
    data_line_offset: int


# ---------------------------------------------------------------------------
# Contract (what config/deviation_survey_contracts.yml says a given file
# SHOULD be)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class DeviationFileContract:
    """
    The complete expected identity, structure, and validation policy for
    one specific deviation-survey source file, as declared in
    `config/deviation_survey_contracts.yml`.

    `header_tolerance_m` bounds how far a file's actual wellhead X/Y and
    datum elevation may differ from the contract's declared expected
    values before being treated as a mismatch (guards against a
    reasonable floating-point/rounding difference while still catching a
    genuinely wrong file).

    `residual_tolerance_tvd_m` / `residual_tolerance_horizontal_m` are the
    PASS thresholds for the independent minimum-curvature-vs-Petrel-source
    trajectory comparison; `residual_fail_threshold_m` is the outer bound
    beyond which a residual is treated as FAIL (indicating a likely
    parsing/contract error) rather than WARNING (a real, documented
    trajectory-reconstruction discrepancy - see `TrajectoryValidationResult`
    and the Increment 3 manifest's discussion of the Proteus 1ST2 finding).
    These three tolerances are deliberately declared identically across
    all four wells in the shipped contract (not tuned per well to force a
    particular pass/fail outcome) - see the contract file's own header
    comment for the rationale.

    `azimuth_reference_for_grid_coordinates` names which of the file's two
    azimuth columns (`AZIM_TN` or `AZIM_GN`) is used for the
    minimum-curvature northing/easting computation - always `"AZIM_GN"`
    for these Petrel files, since the accompanying X/Y/DX/DY columns are
    stated to be grid coordinates, but this is declared explicitly in the
    contract (never hard-coded silently) so a future file using a
    different reference is not silently mishandled.

    `depth_basis_policy` selects, per well, which trajectory
    (`DEPTH_BASIS_PETREL_SOURCE` or `DEPTH_BASIS_MINIMUM_CURVATURE`) is
    used as the downstream MD-to-TVD/TVDSS mapping basis - see
    `DepthBasisSelection`.
    """

    source_filename: str
    expected_sha256: str
    expected_well_identifier: str
    expected_survey_identifier: str
    expected_coordinate_reference_system: str
    expected_wellhead_x_m: float
    expected_wellhead_y_m: float
    expected_datum_m: float
    expected_datum_reference: str
    expected_column_count: int
    expected_column_order: Tuple[str, ...]
    expected_units: Dict[str, str]
    expected_station_count: int
    expected_md_min_m: float
    expected_md_max_m: float
    azimuth_reference_for_grid_coordinates: str
    source_depth_convention: str
    header_tolerance_m: float
    residual_tolerance_tvd_m: float
    residual_tolerance_horizontal_m: float
    residual_fail_threshold_m: float
    depth_basis_policy: str
    notes: str


# ---------------------------------------------------------------------------
# Raw station data (literal, source-preserving)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class DeviationStationData:
    """
    The literal, source-preserving numeric station arrays for one
    deviation-survey file, in original file (increasing-MD) row order.
    Every array has the same length (the station count).

    These are the EXACT values parsed from the file - no NULL/sentinel
    substitution is applicable here (Petrel deviation files, unlike LAS
    files, have no declared NULL-value convention; a missing/invalid
    numeric token is a structural parsing failure, not a sentinel to
    substitute - see `p2mem.io.deviation`), and no unit conversion is
    applied (all quantities are already in the file's declared units:
    metres for MD/X/Y/Z/TVD/DX/DY, degrees for AZIM_TN/INCL/AZIM_GN, and
    degrees-per-30-metres for the file's own supplied DLS column).
    """

    MD_source_m: np.ndarray
    X_source_m: np.ndarray
    Y_source_m: np.ndarray
    Z_source_m: np.ndarray
    TVD_source_m: np.ndarray
    DX_source_m: np.ndarray
    DY_source_m: np.ndarray
    AZIM_TN_source_deg: np.ndarray
    INCL_source_deg: np.ndarray
    DLS_source_deg_per_30m: np.ndarray
    AZIM_GN_source_deg: np.ndarray


@dataclass(frozen=True)
class DeviationIngestionIssue:
    """
    One fact worth reporting about a deviation-survey ingestion attempt:
    either an ERROR (blocks a successful load) or a WARNING (recorded and
    surfaced, but does not by itself stop ingestion). Mirrors
    `p2mem.models.IngestionIssue` from the LAS-ingestion layer, redefined
    locally here so this module stays independent of the locked LAS
    layer's data structures.
    """

    severity: str  # "ERROR" or "WARNING"
    code: str
    message: str
    context: str


# ---------------------------------------------------------------------------
# Trajectory validation (source-vs-computed comparison)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class TrajectoryValidationResult:
    """
    The complete, typed result of comparing the independently computed
    minimum-curvature trajectory against the Petrel-supplied source
    trajectory for one well.

    Every residual metric is reported for THREE separate axes (TVD,
    easting-offset-vs-DX, northing-offset-vs-DY), never pooled into one
    combined number, plus three internal source-consistency checks
    (X ~= X_wellhead + DX, Y ~= Y_wellhead + DY, Z ~= Datum - TVD) that
    verify the Petrel-supplied columns are mutually consistent with each
    other, independent of the minimum-curvature computation.

    `overall_status` is the worst (PASS < WARNING < FAIL) of all six
    per-axis statuses. A WARNING here does not mean ingestion failed -
    see `DeviationWellResult.contract_status`, which is tracked
    separately - it means this specific, disclosed trajectory-consistency
    check did not meet its PASS tolerance and must be reported, not
    hidden.

    `origin_initialization_note` documents, for this specific well, that
    the minimum-curvature trajectory was initialized from that well's own
    first-station source TVD/DX/DY (see `p2mem.trajectory
    .MinimumCurvatureResult` and `p2mem.io.deviation
    .compute_well_trajectory`) - never a hard-coded (0, 0, 0) origin.
    """

    well_key: str
    comparison_basis: str

    tvd_max_abs_residual_m: float
    tvd_mean_residual_m: float
    tvd_rmse_m: float
    tvd_endpoint_residual_m: float
    tvd_tolerance_m: float
    tvd_status: str

    easting_max_abs_residual_m: float
    easting_mean_residual_m: float
    easting_rmse_m: float
    easting_endpoint_residual_m: float
    easting_tolerance_m: float
    easting_status: str

    northing_max_abs_residual_m: float
    northing_mean_residual_m: float
    northing_rmse_m: float
    northing_endpoint_residual_m: float
    northing_tolerance_m: float
    northing_status: str

    x_consistency_max_abs_residual_m: float
    x_consistency_status: str
    y_consistency_max_abs_residual_m: float
    y_consistency_status: str
    z_consistency_max_abs_residual_m: float
    z_consistency_status: str

    overall_status: str
    origin_initialization_note: str


@dataclass(frozen=True)
class DepthBasisSelection:
    """
    The explicit, auditable record of which trajectory (Petrel-supplied
    source, or independently computed minimum-curvature) was selected as
    THIS well's downstream MD-to-TVD/TVDSS mapping basis, and why. Never
    inferred implicitly from `TrajectoryValidationResult.overall_status` -
    always a recorded, per-well decision (see
    `config/deviation_survey_contracts.yml:depth_basis_policy` and
    `p2mem.io.deviation`).
    """

    well_key: str
    selected_basis: str  # one of VALID_DEPTH_BASIS_POLICIES
    rationale: str


# ---------------------------------------------------------------------------
# Combined per-well result
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class DeviationWellResult:
    """
    The complete, typed result of successfully loading, contract-
    resolving, and trajectory-validating one deviation-survey file.

    `contract_status` reflects ONLY file-identity/structural contract
    resolution ("PASSED" when no ERROR-severity `issues` entry exists;
    "FAILED" is never returned here - a contract failure raises
    `p2mem.io.deviation.DeviationContractError` instead, mirroring the
    LAS-layer convention that a successfully RETURNED result is always a
    passed one). It is deliberately independent of
    `validation.overall_status`: a well can have `contract_status ==
    "PASSED"` (the file and contract are valid) while
    `validation.overall_status == "WARNING"` (its computed and supplied
    trajectories disagree by more than the pass tolerance) - see the
    Proteus 1ST2 finding in the Increment 3 manifest. Ingestion success
    and trajectory agreement are two different questions, and this
    dataclass keeps their answers in two different fields on purpose.
    """

    header: DeviationHeaderInfo
    contract: DeviationFileContract
    raw: DeviationStationData
    mc: MinimumCurvatureResult
    validation: TrajectoryValidationResult
    depth_basis: DepthBasisSelection
    issues: Tuple[DeviationIngestionIssue, ...] = field(default_factory=tuple)
    contract_status: str = "PASSED"


@dataclass(frozen=True)
class DeviationIngestionFailure:
    """
    A typed, structured record of why one well's deviation-survey file
    failed to load during a batch (`p2mem.io.deviation.load_deviation_surveys`).
    Mirrors `p2mem.models.IngestionFailure` from the LAS-ingestion layer.

    `error_type` is one of "file_not_found", "parsing_failure",
    "contract_failure", or "trajectory_failure" (a
    `p2mem.trajectory.TrajectoryComputationError` raised while computing
    the minimum-curvature trajectory for an otherwise contract-valid
    file - kept distinct from "contract_failure" since it identifies a
    different stage of the pipeline).
    """

    well_key: str
    source_path: str
    error_type: str
    message: str
    exception: BaseException


# ---------------------------------------------------------------------------
# LAS MD -> TVD/TVDSS mapping (p2mem.depth_mapping)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class LasDepthMappingResult:
    """
    The complete, typed result of mapping one well's Increment-2.1.1-
    canonical LAS `MD_m` array onto TVD and TVDSS, using the explicitly
    selected depth basis for that well.

    `las_md_source_m` is the ORIGINAL, unmodified LAS canonical `MD_m`
    array (never overwritten - see `p2mem.depth_mapping`).
    `tvd_mapped_m` / `tvdss_mapped_m` are the interpolated results, one
    value per LAS sample, in the same order.

    `interpolation_method` names the deterministic method used (this
    project's implementation: piecewise-linear interpolation of the
    validated, selected-basis station trajectory - explicitly NOT a
    per-sample minimum-curvature recomputation; see
    `p2mem.depth_mapping` module docstring for the rationale).

    `n_extrapolated` counts LAS samples that would have required
    extrapolation beyond the survey's station MD coverage; by policy this
    is always 0 for a successful mapping result (extrapolation is
    rejected - see `p2mem.depth_mapping.DepthMappingError`) and is
    reported here only as a confirming, self-describing field.
    """

    well_key: str
    depth_basis_used: str
    interpolation_method: str
    las_md_source_m: np.ndarray
    tvd_mapped_m: np.ndarray
    tvdss_mapped_m: np.ndarray
    n_samples: int
    survey_md_min_m: float
    survey_md_max_m: float
    las_md_min_m: float
    las_md_max_m: float
    coverage_margin_lower_m: float
    coverage_margin_upper_m: float
    n_extrapolated: int
    datum_elevation_m: float


#### `p2mem/trajectory.py` — minimum-curvature trajectory engine

Implements the dogleg-angle, ratio-factor, and displacement formulas from the Theory section above, with an explicit Taylor-series limit for the ratio factor as the dogleg approaches zero (never a raw 0/0 division), and rejects nonphysical station data (inclination outside [0, 180] degrees, non-finite values, non-increasing MD) rather than silently repairing it.

In [ ]:
%%writefile p2mem/trajectory.py
"""
p2mem.trajectory - Minimum-curvature well-trajectory computation (Increment 3).

Scope
-----
This module implements ONLY the standard minimum-curvature method for
converting a sequence of (MD, inclination, azimuth) directional-survey
stations into a 3D trajectory (TVD, northing offset, easting offset), plus
the numerically stable ratio-factor limit, dogleg-severity computation, and
residual comparison against an independently supplied ("source") trajectory
(e.g. the Petrel-computed X/Y/Z/TVD columns in a deviation-survey file).

Nothing here reads a file, applies a contract, or decides which trajectory
("Petrel-supplied" vs "minimum-curvature-computed") should be used
downstream - that policy decision belongs to `p2mem.depth_mapping` and the
per-well `deviation_survey_contracts.yml` configuration. This module is
pure numerical computation only.

Governing theory
-----------------
For two consecutive survey stations 1 and 2 with inclination I (measured
from vertical) and azimuth A (measured clockwise from north, in the same
azimuth reference used for the accompanying grid coordinates), the dogleg
angle `beta` (the angle, in 3D space, between the two stations'
tangent-direction unit vectors) satisfies:

    cos(beta) = cos(I1) cos(I2) + sin(I1) sin(I2) cos(A2 - A1)

This is the standard textbook statement of the dogleg-angle relationship,
and remains the definition this module implements. However, `beta` itself
is NOT computed by evaluating this right-hand side and then taking
`arccos` of it (see "Numerical stability of the dogleg angle" below for
why) - it is instead computed via an equivalent, numerically stable
vector formulation that agrees with this formula everywhere but does not
share `arccos`'s ill-conditioning near `beta == 0`.

The minimum-curvature method then fits a single circular arc, of constant
curvature, between the two station directions - as opposed to the (less
accurate) tangential or average-angle methods, which assume straight-line
or simple-average behavior between stations.

The ratio factor:

    RF = (2 / beta) * tan(beta / 2)

converts the straight-line ("tangential") displacement into the
arc-corrected minimum-curvature displacement. As beta -> 0 (no direction
change - a perfectly straight hold section), RF has the well-defined limit
RF -> 1 (a straight line is a degenerate circular arc of infinite radius).
This module evaluates that limit via a Taylor-series expansion for very
small beta, rather than relying on floating-point division/tan() behavior
at or near beta == 0, which otherwise produces 0/0 (NaN) exactly at
beta == 0 and can lose precision for beta approaching it.

Displacement over an interval of measured-depth length dMD is then:

    dTVD      = (dMD / 2) (cos I1 + cos I2) RF
    dNorthing = (dMD / 2) (sin I1 cos A1 + sin I2 cos A2) RF
    dEasting  = (dMD / 2) (sin I1 sin A1 + sin I2 sin A2) RF

and dogleg severity, expressed in the oilfield-conventional degrees per 30
metres, is:

    DLS_deg_per_30m = (beta_deg / dMD) * 30

Numerical stability of the dogleg angle (Increment 3.1 correction)
---------------------------------------------------------------------
Increment 3 originally computed `beta` as `arccos(clip(cos_beta, -1, 1))`,
evaluating the right-hand side of the formula above directly. `arccos` is
ill-conditioned as its argument approaches +1 (`beta -> 0`): its
derivative diverges there, so an ordinary ~1e-16 floating-point residual
in `cos_beta` surfaced as a spurious dogleg of order 1e-8 radians (~1e-6
degrees) even for two stations with EXACTLY identical inclination and
azimuth - a perfectly straight hold section that should report `beta ==
0.0` exactly. This was reported (not hidden) at the time, with a widened
test tolerance and a docstring note - but a widened tolerance treats the
symptom, not the cause, and this corrective patch replaces the
ill-conditioned formulation instead.

`dogleg_angle_rad` now computes `beta` as the angle between the two
stations' 3-D tangent unit vectors via `arctan2(|u1 x u2|, u1 . u2)`
rather than `arccos(u1 . u2)`. This is mathematically the identical angle
for every `beta` in `[0, pi]` (both the cross-product magnitude and the
dot product are continuous, well-conditioned functions of `u1`/`u2` even
as the vectors converge), but unlike `arccos`, `arctan2` has a bounded
derivative everywhere on this domain, including exactly at `beta == 0`:
two identical unit vectors have an EXACT (floating-point-zero) cross
product, so `arctan2(0.0, ~1.0)` returns exactly `0.0`, with no residual
noise floor. A genuinely tiny nonzero dogleg (e.g. 1e-6 degrees of real
curvature) remains numerically well-resolved, since `arctan2` near
`y = 0` does not suffer the vanishing-derivative problem `arccos` has
near `x = 1`. See `tests/test_trajectory.py` for a direct comparison of
old-versus-new behavior on identical, tiny, and moderate doglegs.

Angle handling
--------------
All angle trigonometry is performed in radians. Degree<->radian conversion
uses the already-locked, tested `p2mem.units.degrees_to_radians` /
`radians_to_degrees` functions (Increment 1.1) rather than reimplementing
degree/radian conversion here, so the exact same validated conversion is
used project-wide.

Azimuth is measured mod 360 degrees; the dogleg formula above uses
`cos(A2 - A1)`, which is inherently periodic in the azimuth difference, so
a wraparound case (e.g. A1 = 359 deg, A2 = 1 deg, an actual difference of
only 2 degrees) is handled correctly WITHOUT any explicit modulo-360
normalization of the input azimuths themselves - `cos` already treats
359 deg and -1 deg identically.

Azimuth is physically undefined at zero inclination (a vertical borehole
has no horizontal direction to reference an azimuth against). This is not
special-cased in `dogleg_angle_rad`: when I1 == 0, `sin(I1) == 0`, so the
`sin(I1) sin(I2) cos(A2 - A1)` cross-term vanishes identically regardless
of what azimuth value the survey happens to record for that station (real
files commonly record 0.0 or an arbitrary placeholder azimuth at a
vertical station) - so no artificial "azimuth discontinuity" dogleg is
ever generated purely from a vertical station's azimuth value. See
`test_trajectory.py::test_azimuth_undefined_at_zero_inclination_does_not_
inflate_dogleg`.

Failure handling
-----------------
`TrajectoryComputationError` is raised for nonphysical inputs (inclination
outside [0, 180] degrees, non-finite MD/inclination/azimuth, or a
non-increasing MD sequence) - this module never silently clips, drops, or
"repairs" a station; the caller (`p2mem.io.deviation`) is responsible for
station-level QC before trajectory computation is attempted, and this
module performs its own defensive validation as a second, independent
gate.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from p2mem.units import degrees_to_radians, radians_to_degrees

__all__ = [
    "TrajectoryComputationError",
    "MinimumCurvatureResult",
    "dogleg_angle_rad",
    "ratio_factor",
    "minimum_curvature_intervals",
    "compute_minimum_curvature_trajectory",
]

# Below this dogleg angle (radians), the ratio factor is evaluated via its
# Taylor-series limit (RF = 1 + beta^2/12) rather than the direct
# 2/beta * tan(beta/2) expression, which is an exact 0/0 (NaN) form at
# beta == 0 and is unnecessary to evaluate directly at all once the
# quadratic term is already many orders of magnitude below float64
# precision (beta ~ 1e-6 rad gives a Taylor correction of ~8e-14, i.e.
# already below double-precision resolution of the leading 1.0 term).
# This threshold is independent of, and unaffected by, the Increment 3.1
# dogleg-angle numerical-stability correction (see module docstring):
# beta == 0.0 (identical stations) and any genuinely tiny beta are both
# still routed to this Taylor branch exactly as before.
_SMALL_DOGLEG_THRESHOLD_RAD = 1.0e-9


class TrajectoryComputationError(ValueError):
    """
    Raised when minimum-curvature trajectory computation is asked to
    operate on nonphysical or structurally invalid station data:
    inclination outside [0, 180] degrees, a non-finite MD/inclination/
    azimuth value, or a measured-depth sequence that is not strictly
    increasing. Never silently clipped, dropped, or repaired.
    """


@dataclass(frozen=True)
class MinimumCurvatureResult:
    """
    The complete, typed result of a minimum-curvature trajectory
    computation over a full station sequence of length n.

    All arrays have length n (one value per station). The first station
    (index 0) has no preceding interval, so its `dogleg_deg` and
    `dls_deg_per_30m` are defined as exactly 0.0 by convention (consistent
    with how the real Petrel deviation files themselves record DLS = 0.0
    at their first station) - not NaN, since a "no computation possible
    here" placeholder value of 0.0 for an angle/severity index at the very
    top of the well is the same convention the source data already uses.

    `tvd_mc_m`, `northing_offset_mc_m`, and `easting_offset_mc_m` are
    CUMULATIVE, computed by initializing station 0 from the caller-
    supplied `tvd_origin_m` / `northing_origin_m` / `easting_origin_m`
    (see `compute_minimum_curvature_trajectory`) and then accumulating the
    per-interval minimum-curvature deltas station-by-station. This
    initialization choice - using the survey's own first-station source
    offsets as the tie-on point, rather than assuming a hard-coded (0, 0,
    0) origin - is deliberate: real files (e.g. Proteus 1ST2 in this
    project) can carry a tiny nonzero first-station MD/TVD/offset
    (~-7.63e-7 m) that must be preserved as the literal starting
    condition, not silently zeroed.
    """

    dogleg_deg: np.ndarray
    dls_deg_per_30m: np.ndarray
    tvd_mc_m: np.ndarray
    northing_offset_mc_m: np.ndarray
    easting_offset_mc_m: np.ndarray


def _validate_stations(
    md_m: np.ndarray, incl_deg: np.ndarray, azim_deg: np.ndarray
) -> None:
    if not (md_m.shape == incl_deg.shape == azim_deg.shape):
        raise TrajectoryComputationError(
            f"minimum-curvature input arrays must share one shape; got "
            f"md_m={md_m.shape}, incl_deg={incl_deg.shape}, azim_deg={azim_deg.shape}"
        )
    if md_m.ndim != 1 or md_m.size < 1:
        raise TrajectoryComputationError(
            "minimum-curvature input must be a 1-D array of at least one station"
        )
    for name, arr in (("md_m", md_m), ("incl_deg", incl_deg), ("azim_deg", azim_deg)):
        if not np.all(np.isfinite(arr)):
            bad = np.where(~np.isfinite(arr))[0].tolist()
            raise TrajectoryComputationError(
                f"{name} contains non-finite value(s) at station index(es) {bad}; "
                f"minimum-curvature computation requires every station finite"
            )
    if md_m.size > 1 and not np.all(np.diff(md_m) > 0.0):
        bad = np.where(np.diff(md_m) <= 0.0)[0].tolist()
        raise TrajectoryComputationError(
            f"MD must be strictly increasing for minimum-curvature computation; "
            f"non-increasing step(s) found starting at interval index(es) {bad}"
        )
    if np.any((incl_deg < 0.0) | (incl_deg > 180.0)):
        bad = np.where((incl_deg < 0.0) | (incl_deg > 180.0))[0].tolist()
        raise TrajectoryComputationError(
            f"inclination outside the physically valid range [0, 180] degrees "
            f"at station index(es) {bad}: {incl_deg[bad].tolist()}"
        )


def dogleg_angle_rad(
    incl1_deg: np.ndarray,
    incl2_deg: np.ndarray,
    azim1_deg: np.ndarray,
    azim2_deg: np.ndarray,
) -> np.ndarray:
    """
    Compute the dogleg angle beta (radians) between consecutive stations,
    mathematically defined by:

        cos(beta) = cos(I1) cos(I2) + sin(I1) sin(I2) cos(A2 - A1)

    Inputs are in degrees (converted internally via
    `p2mem.units.degrees_to_radians`); output is in radians, always in
    [0, pi].

    Numerically stable implementation (Increment 3.1 correction): rather
    than evaluating the right-hand side above and taking `arccos` of it
    (ill-conditioned as `beta -> 0` - see the module docstring's
    "Numerical stability of the dogleg angle" section for the full
    rationale and the defect this replaced), `beta` is computed as the
    angle between each station pair's 3-D tangent unit vectors
    `u = (cos I, sin I cos A, sin I sin A)` via

        beta = arctan2(|u1 x u2|, u1 . u2)

    This is the identical angle for every beta in [0, pi] (the dot
    product above expands to exactly `cos(I1)cos(I2) +
    sin(I1)sin(I2)cos(A2-A1)` by the standard trig product-to-sum
    identity), but has a bounded derivative everywhere on this domain,
    including exactly at beta == 0: two numerically identical tangent
    vectors have an exactly-zero cross product, so identical (I, A)
    station pairs now report beta == 0.0 to full floating-point
    precision, with no residual noise floor, while genuinely tiny nonzero
    doglegs remain accurately resolved (no clipping of the dot product is
    needed here, unlike the old `arccos` formulation, since `arctan2` is
    well-behaved for any finite argument pair).
    """
    i1 = degrees_to_radians(np.asarray(incl1_deg, dtype=np.float64))
    i2 = degrees_to_radians(np.asarray(incl2_deg, dtype=np.float64))
    a1 = degrees_to_radians(np.asarray(azim1_deg, dtype=np.float64))
    a2 = degrees_to_radians(np.asarray(azim2_deg, dtype=np.float64))

    u1 = np.stack([np.cos(i1), np.sin(i1) * np.cos(a1), np.sin(i1) * np.sin(a1)], axis=-1)
    u2 = np.stack([np.cos(i2), np.sin(i2) * np.cos(a2), np.sin(i2) * np.sin(a2)], axis=-1)

    cross_norm = np.linalg.norm(np.cross(u1, u2), axis=-1)
    dot = np.sum(u1 * u2, axis=-1)
    return np.arctan2(cross_norm, dot)


def ratio_factor(beta_rad: np.ndarray) -> np.ndarray:
    """
    Compute the minimum-curvature ratio factor RF = (2/beta) tan(beta/2),
    using the numerically stable limit RF -> 1 + beta^2/12 for
    beta < `_SMALL_DOGLEG_THRESHOLD_RAD` (including beta == 0 exactly,
    which the direct formula cannot evaluate: 2/0 * tan(0) is a 0/0 form).

    The Taylor series RF = 1 + beta^2/12 + O(beta^4) follows from
    tan(x) = x + x^3/3 + O(x^5) with x = beta/2:

        (2/beta) tan(beta/2) = (2/beta) (beta/2 + (beta/2)^3/3 + ...)
                              = 1 + beta^2/12 + O(beta^4)
    """
    beta = np.asarray(beta_rad, dtype=np.float64)
    small = np.abs(beta) < _SMALL_DOGLEG_THRESHOLD_RAD
    with np.errstate(divide="ignore", invalid="ignore"):
        direct = (2.0 / beta) * np.tan(beta / 2.0)
    taylor = 1.0 + (beta**2) / 12.0
    return np.where(small, taylor, direct)


def minimum_curvature_intervals(
    md_m: np.ndarray,
    incl_deg: np.ndarray,
    azim_deg: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute per-interval dogleg angle, ratio factor, and minimum-curvature
    displacement deltas (dTVD, dNorthing, dEasting) for every consecutive
    station pair in a station sequence of length n.

    Returns (beta_deg, rf, delta_tvd_m, delta_northing_m, delta_easting_m),
    each of length (n - 1): index k corresponds to the interval from
    station k to station k+1.

    `azim_deg` must already be the single azimuth reference the caller
    intends to use for the grid-coordinate (northing/easting) computation
    (i.e. AZIM_GN for these Petrel files, per the project's deviation
    contracts) - this function does not know about, and never mixes,
    multiple azimuth references; that separation is the caller's
    responsibility (see `p2mem.io.deviation`).
    """
    md_m = np.asarray(md_m, dtype=np.float64)
    incl_deg = np.asarray(incl_deg, dtype=np.float64)
    azim_deg = np.asarray(azim_deg, dtype=np.float64)
    _validate_stations(md_m, incl_deg, azim_deg)

    i1_deg, i2_deg = incl_deg[:-1], incl_deg[1:]
    a1_deg, a2_deg = azim_deg[:-1], azim_deg[1:]
    delta_md = md_m[1:] - md_m[:-1]

    beta_rad = dogleg_angle_rad(i1_deg, i2_deg, a1_deg, a2_deg)
    rf = ratio_factor(beta_rad)

    i1 = degrees_to_radians(i1_deg)
    i2 = degrees_to_radians(i2_deg)
    a1 = degrees_to_radians(a1_deg)
    a2 = degrees_to_radians(a2_deg)

    half_md = delta_md / 2.0
    delta_tvd = half_md * (np.cos(i1) + np.cos(i2)) * rf
    delta_northing = half_md * (np.sin(i1) * np.cos(a1) + np.sin(i2) * np.cos(a2)) * rf
    delta_easting = half_md * (np.sin(i1) * np.sin(a1) + np.sin(i2) * np.sin(a2)) * rf

    beta_deg = radians_to_degrees(beta_rad)
    return beta_deg, rf, delta_tvd, delta_northing, delta_easting


def compute_minimum_curvature_trajectory(
    md_m: np.ndarray,
    incl_deg: np.ndarray,
    azim_deg: np.ndarray,
    tvd_origin_m: float,
    northing_origin_m: float,
    easting_origin_m: float,
) -> MinimumCurvatureResult:
    """
    Compute the full cumulative minimum-curvature trajectory (TVD,
    northing offset, easting offset) for a station sequence of length n,
    initialized at station 0 from the caller-supplied origin values
    (typically that well's own first-station source TVD/DY/DX, so the
    independent trajectory starts from exactly the same tie-on point as
    the source survey - see `MinimumCurvatureResult` docstring).

    Dogleg severity is reported in the oilfield-conventional
    degrees-per-30-metres: DLS_deg_per_30m = (beta_deg / delta_MD) * 30.
    A zero-length interval (delta_MD == 0) cannot occur here because
    `_validate_stations` requires MD strictly increasing.
    """
    md_m = np.asarray(md_m, dtype=np.float64)
    incl_deg = np.asarray(incl_deg, dtype=np.float64)
    azim_deg = np.asarray(azim_deg, dtype=np.float64)
    _validate_stations(md_m, incl_deg, azim_deg)
    n = md_m.size

    dogleg_deg_full = np.zeros(n, dtype=np.float64)
    dls_full = np.zeros(n, dtype=np.float64)
    tvd_mc = np.empty(n, dtype=np.float64)
    northing_mc = np.empty(n, dtype=np.float64)
    easting_mc = np.empty(n, dtype=np.float64)

    tvd_mc[0] = tvd_origin_m
    northing_mc[0] = northing_origin_m
    easting_mc[0] = easting_origin_m

    if n > 1:
        beta_deg, _rf, d_tvd, d_north, d_east = minimum_curvature_intervals(
            md_m, incl_deg, azim_deg
        )
        delta_md = md_m[1:] - md_m[:-1]
        dls_intervals = (beta_deg / delta_md) * 30.0

        dogleg_deg_full[1:] = beta_deg
        dls_full[1:] = dls_intervals
        tvd_mc[1:] = tvd_origin_m + np.cumsum(d_tvd)
        northing_mc[1:] = northing_origin_m + np.cumsum(d_north)
        easting_mc[1:] = easting_origin_m + np.cumsum(d_east)

    return MinimumCurvatureResult(
        dogleg_deg=dogleg_deg_full,
        dls_deg_per_30m=dls_full,
        tvd_mc_m=tvd_mc,
        northing_offset_mc_m=northing_mc,
        easting_offset_mc_m=easting_mc,
    )


#### `p2mem/io/deviation.py` — Petrel deviation-survey parser and per-file contract resolver

Mirrors the locked LAS layer's design: structural parsing (can the file even be tokenized) is kept separate from contract resolution (does this specific file's header/data agree with `config/deviation_survey_contracts.yml`), and every file-identity check (filename, SHA-256, well/survey identifier, wellhead/datum, coordinate system, column order, station count, MD coverage) is a blocking `ERROR`, never a silent pass.

In [ ]:
%%writefile p2mem/io/deviation.py
"""
p2mem.io.deviation - Auditable Petrel deviation-survey (well-trace) parser
and per-file curve-contract resolution (Increment 3).

Why this module exists
------------------------
The four approved deviation-survey files (Poseidon 2, Boreas 1, Poseidon
North 1, Proteus 1ST2) are plain-text Petrel well-trace exports with a
fixed-format, human-readable header block (well name, survey name,
wellhead coordinates, datum, coordinate-reference-system statement, and
sign/unit conventions) followed by a fixed 11-column station table (MD, X,
Y, Z, TVD, DX, DY, AZIM_TN, INCL, DLS, AZIM_GN). This module parses that
format strictly: every header field, and the exact column order, is
extracted and preserved; nothing is inferred, renamed, or filled in
without being explicitly reported.

Mirroring the design already established for the LAS-ingestion layer
(`p2mem.io.las`, locked as of Increment 2.1.1), this module separates
STRUCTURAL PARSING (can the file even be tokenized: is the header
present, are all 11 columns present with unique names, does every row
have the right width and numeric tokens) from CONTRACT RESOLUTION (does
this specific file's actual header/data agree with what
`config/deviation_survey_contracts.yml` says it SHOULD be: filename,
SHA-256, well/survey identity, wellhead coordinates, datum, coordinate
system, column order, station count, MD coverage). A structural defect is
a `DeviationParsingError`; a contract mismatch is a
`DeviationContractError`. Both are always ERROR-severity and always
blocking - this module never guesses its way past either kind of problem.

Header parsing note (MD unit)
-------------------------------
Every file explicitly declares X/Y/DX/DY/Z/TVD units as metres (the
"WELL HEAD X-COORDINATE: ... (m)" lines and the "DX DY ARE GIVEN IN GRID
NORTH IN m-UNITS" / "DEPTH (Z, tvd_z) GIVEN IN m-UNITS" statements), and
declares angles in degrees ("ANGLES ARE GIVEN IN DEGREES") - covering
AZIM_TN, INCL, and AZIM_GN. The MD column's own unit is NOT covered by any
of those explicit statements (no header line says "MD is in metres").
Rather than silently assuming metres, this module records that gap as a
disclosed WARNING-severity `DeviationIngestionIssue`
(`MD_UNIT_NOT_EXPLICITLY_DECLARED`) on every successful load, and treats
MD as metres only because its numeric magnitude is self-consistent with
the file's own explicitly-metric TVD/Z columns (a station's MD and TVD
are always of the same order of magnitude in these near-vertical-to-
moderate-inclination wells) and with the already-validated, explicitly
metric `MD_m` canonical curve from the locked Increment 2.1.1 LAS layer
for the same four wells. This is an inference, not a fabrication, and it
is reported, not hidden.
"""

from __future__ import annotations

import hashlib
import math
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import yaml

from p2mem.deviation_models import (
    DEPTH_BASIS_PETREL_SOURCE,
    STATUS_FAIL,
    STATUS_PASS,
    STATUS_WARNING,
    VALID_DEPTH_BASIS_POLICIES,
    DeviationFileContract,
    DeviationHeaderInfo,
    DeviationIngestionFailure,
    DeviationIngestionIssue,
    DeviationStationData,
    DeviationWellResult,
    DepthBasisSelection,
    TrajectoryValidationResult,
)
from p2mem.trajectory import (
    TrajectoryComputationError,
    compute_minimum_curvature_trajectory,
)

__all__ = [
    "DeviationFileNotFoundError",
    "DeviationParsingError",
    "DeviationContractDefinitionError",
    "DeviationContractError",
    "REQUIRED_COLUMN_NAMES",
    "MD_UNIT_NOT_EXPLICITLY_DECLARED_CODE",
    "DLS_NORMALIZATION_INFERRED_CODE",
    "load_deviation_contract_config",
    "parse_deviation_header",
    "read_deviation_stations",
    "resolve_deviation_contract",
    "compute_well_trajectory_validation",
    "load_deviation_file",
    "load_deviation_surveys",
]

# Issue codes shared between production code and tests, so a test can
# reference the exact code without duplicating the literal string.
MD_UNIT_NOT_EXPLICITLY_DECLARED_CODE = "MD_UNIT_NOT_EXPLICITLY_DECLARED"
DLS_NORMALIZATION_INFERRED_CODE = "DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M"


# ---------------------------------------------------------------------------
# Exceptions
# ---------------------------------------------------------------------------
class DeviationFileNotFoundError(FileNotFoundError):
    """The deviation-survey file path given to `load_deviation_file` does not exist."""


class DeviationParsingError(ValueError):
    """
    A structural defect in the deviation-survey text file itself: a
    missing/unrecognized header banner, a missing required header field,
    a missing/duplicated required column, a malformed numeric token, a
    non-finite (NaN/Inf) literal token, or a data row whose column count
    does not match the header's declared column count. Raised before any
    contract is consulted.
    """


class DeviationContractDefinitionError(ValueError):
    """
    `config/deviation_survey_contracts.yml` itself is malformed: a
    duplicate top-level file key, a missing required field, an invalid
    type, an unsupported convention value (e.g. an azimuth reference or
    depth-basis policy this module does not implement), or an internally
    inconsistent/incompatible set of expected values (e.g. a declared
    column count that does not match the declared column-order list
    length, or a residual fail-threshold at or below its own pass
    tolerance). Raised at contract-load time, before any deviation file is
    opened.
    """


class DeviationContractError(RuntimeError):
    """
    A specific file's actual parsed header or station data does not agree
    with its `DeviationFileContract`: wrong filename, SHA-256 mismatch,
    well/survey identity mismatch, wellhead/datum mismatch beyond
    tolerance, coordinate-reference-system mismatch, wrong column order,
    station-count mismatch, or MD-coverage mismatch. Carries `.issues`
    (the full tuple of `DeviationIngestionIssue`, ERROR and WARNING alike)
    for callers that want the complete detail, not just the first failure.
    """

    def __init__(self, message: str, issues: Tuple[DeviationIngestionIssue, ...]):
        super().__init__(message)
        self.issues = issues


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
# The fixed 11-column Petrel well-trace schema this parser implements.
# Order here is NOT the required file order (that is a per-file contract
# question - see `expected_column_order`); this is simply the set of
# canonical field names every file must supply exactly once each.
REQUIRED_COLUMN_NAMES: Tuple[str, ...] = (
    "MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN",
)

# Maps a source column name to the `DeviationStationData` field name it
# populates.
_COLUMN_TO_FIELD = {
    "MD": "MD_source_m",
    "X": "X_source_m",
    "Y": "Y_source_m",
    "Z": "Z_source_m",
    "TVD": "TVD_source_m",
    "DX": "DX_source_m",
    "DY": "DY_source_m",
    "AZIM_TN": "AZIM_TN_source_deg",
    "INCL": "INCL_source_deg",
    "DLS": "DLS_source_deg_per_30m",
    "AZIM_GN": "AZIM_GN_source_deg",
}

_SEPARATOR_RE = re.compile(r"^#=+$")
_WELL_NAME_RE = re.compile(r"^#\s*WELL NAME:\s*(.+?)\s*$")
_SURVEY_RE = re.compile(r"^#\s*DEFINITIVE SURVEY:\s*(.+?)\s*$")
_WELLHEAD_X_RE = re.compile(r"^#\s*WELL HEAD X-COORDINATE:\s*([\-0-9.eE]+)\s*\(([^)]*)\)\s*$")
_WELLHEAD_Y_RE = re.compile(r"^#\s*WELL HEAD Y-COORDINATE:\s*([\-0-9.eE]+)\s*\(([^)]*)\)\s*$")
_DATUM_RE = re.compile(r"^#\s*WELL DATUM\s*\(([^)]*)\)\s*:\s*([\-0-9.eE]+)\s*\(([^)]*)\)\s*$")
_WELL_TYPE_RE = re.compile(r"^#\s*WELL TYPE:\s*(.+?)\s*$")
_DEPTH_REF_RE = re.compile(r"^#\s*(MD AND TVD ARE REFERENCED.+)$")
_ANGLE_UNIT_RE = re.compile(r"^#\s*ANGLES ARE GIVEN IN\s*(.+?)\s*$")
_CRS_RE = re.compile(r"^#\s*XYZ TRACE IS GIVEN IN COORDINATE SYSTEM\s*(.+?)\s*$")
_AZIM_TN_DESC_RE = re.compile(r"^#\s*AZIM_TN:")
_AZIM_GN_DESC_RE = re.compile(r"^#\s*AZIM_GN:")
_DXDY_RE = re.compile(r"^#\s*(DX DY ARE GIVEN IN.+)$")
_Z_RE = re.compile(r"^#\s*(DEPTH \(Z,.+)$")
_BANNER_RE = re.compile(r"^#\s*WELL TRACE FROM PETREL")


# ---------------------------------------------------------------------------
# Contract configuration (config/deviation_survey_contracts.yml)
# ---------------------------------------------------------------------------
class _NoDuplicateKeySafeLoader(yaml.SafeLoader):
    """
    A `yaml.SafeLoader` subclass that raises on a duplicate mapping key
    instead of silently keeping only the last occurrence (PyYAML's
    default `SafeLoader` behavior). Used only for
    `config/deviation_survey_contracts.yml` so that a duplicate top-level
    file key (or a duplicate field within one file's entry) is a loud
    contract-authoring error, never a silent overwrite.
    """


def _construct_mapping_no_duplicates(loader: yaml.SafeLoader, node, deep: bool = False):
    mapping: Dict = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise DeviationContractDefinitionError(
                f"Duplicate key {key!r} found while parsing deviation contract YAML "
                f"(line {key_node.start_mark.line + 1}); duplicate contract keys are "
                f"rejected rather than silently keeping only the last one."
            )
        value = loader.construct_object(value_node, deep=deep)
        mapping[key] = value
    return mapping


_NoDuplicateKeySafeLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, _construct_mapping_no_duplicates
)

_REQUIRED_FILE_FIELDS = (
    "expected_sha256",
    "expected_well_identifier",
    "expected_survey_identifier",
    "expected_coordinate_reference_system",
    "expected_wellhead_x_m",
    "expected_wellhead_y_m",
    "expected_datum_m",
    "expected_datum_reference",
    "expected_column_count",
    "expected_column_order",
    "expected_units",
    "expected_station_count",
    "expected_md_min_m",
    "expected_md_max_m",
    "azimuth_reference_for_grid_coordinates",
    "source_depth_convention",
    "header_tolerance_m",
    "residual_tolerance_tvd_m",
    "residual_tolerance_horizontal_m",
    "residual_fail_threshold_m",
    "depth_basis_policy",
    "notes",
)

_VALID_UNIT_TOKENS = {"m", "deg", "deg_per_30m"}
_VALID_AZIMUTH_REFERENCES = {"AZIM_TN", "AZIM_GN"}


def load_deviation_contract_config(yaml_path: str) -> Dict[str, DeviationFileContract]:
    """
    Load and validate `config/deviation_survey_contracts.yml`, returning a
    mapping of source filename -> `DeviationFileContract`.

    Validates, at load time (before any deviation file is opened):
    * the YAML parses, has a top-level `files` mapping, and contains no
      duplicate mapping keys anywhere (see `_NoDuplicateKeySafeLoader`);
    * every required field (`_REQUIRED_FILE_FIELDS`) is present for every
      file entry;
    * `expected_column_count` is a positive integer equal to
      `len(expected_column_order)`, and `expected_column_order` is a list
      of unique strings drawn from `REQUIRED_COLUMN_NAMES`;
    * `expected_units` covers exactly `REQUIRED_COLUMN_NAMES` with values
      drawn from `_VALID_UNIT_TOKENS`;
    * `expected_wellhead_x_m`, `expected_wellhead_y_m`, `expected_datum_m`,
      `expected_md_min_m`, `expected_md_max_m`, `header_tolerance_m`,
      `residual_tolerance_tvd_m`, `residual_tolerance_horizontal_m`, and
      `residual_fail_threshold_m` are all genuine (non-boolean) numeric
      types;
    * the three tolerance fields are non-negative, and
      `residual_fail_threshold_m` is strictly greater than both
      `residual_tolerance_tvd_m` and `residual_tolerance_horizontal_m`
      (otherwise there is no real WARNING band between "pass" and "fail");
    * `expected_md_min_m <= expected_md_max_m`;
    * `expected_station_count` is a positive integer;
    * `azimuth_reference_for_grid_coordinates` is one of
      `_VALID_AZIMUTH_REFERENCES` AND appears in `expected_column_order`;
    * `depth_basis_policy` is one of
      `p2mem.deviation_models.VALID_DEPTH_BASIS_POLICIES`.

    Raises
    ------
    DeviationContractDefinitionError
        On any of the above validation failures, or if the file is
        missing / not valid YAML.
    """
    p = Path(yaml_path)
    if not p.exists():
        raise DeviationContractDefinitionError(f"Deviation contract file not found: {yaml_path}")

    try:
        with p.open("r", encoding="utf-8") as f:
            raw = yaml.load(f, Loader=_NoDuplicateKeySafeLoader)
    except yaml.YAMLError as exc:
        raise DeviationContractDefinitionError(f"{yaml_path}: not valid YAML ({exc})") from exc

    if not raw or "files" not in raw or not isinstance(raw["files"], dict):
        raise DeviationContractDefinitionError(f"{yaml_path}: expected a top-level 'files' mapping.")

    contracts: Dict[str, DeviationFileContract] = {}
    for filename, d in raw["files"].items():
        if not isinstance(d, dict):
            raise DeviationContractDefinitionError(f"{yaml_path}: entry for {filename!r} must be a mapping.")

        missing = [f for f in _REQUIRED_FILE_FIELDS if f not in d]
        if missing:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: entry for {filename!r} is missing required field(s): {missing}"
            )

        # --- column order / count ---
        col_order = d["expected_column_order"]
        if not isinstance(col_order, list) or not all(isinstance(c, str) for c in col_order):
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_column_order must be a list of strings."
            )
        if len(set(col_order)) != len(col_order):
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_column_order contains duplicate column names."
            )
        if set(col_order) != set(REQUIRED_COLUMN_NAMES):
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_column_order must be exactly the set "
                f"{REQUIRED_COLUMN_NAMES}; got {tuple(col_order)}."
            )
        col_count = d["expected_column_count"]
        if isinstance(col_count, bool) or not isinstance(col_count, int) or col_count <= 0:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_column_count must be a positive integer."
            )
        if col_count != len(col_order):
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_column_count ({col_count}) does not match "
                f"len(expected_column_order) ({len(col_order)})."
            )

        # --- units ---
        units = d["expected_units"]
        if not isinstance(units, dict) or set(units.keys()) != set(REQUIRED_COLUMN_NAMES):
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_units must declare exactly the columns "
                f"{REQUIRED_COLUMN_NAMES}."
            )
        bad_units = {k: v for k, v in units.items() if v not in _VALID_UNIT_TOKENS}
        if bad_units:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_units has unsupported unit token(s): {bad_units} "
                f"(supported: {_VALID_UNIT_TOKENS})."
            )

        # --- numeric fields ---
        def _num(field: str) -> float:
            v = d[field]
            if isinstance(v, bool) or not isinstance(v, (int, float)):
                raise DeviationContractDefinitionError(
                    f"{yaml_path}: {filename!r}.{field} must be a genuine number, got {v!r}."
                )
            return float(v)

        wellhead_x = _num("expected_wellhead_x_m")
        wellhead_y = _num("expected_wellhead_y_m")
        datum = _num("expected_datum_m")
        md_min = _num("expected_md_min_m")
        md_max = _num("expected_md_max_m")
        header_tol = _num("header_tolerance_m")
        tvd_tol = _num("residual_tolerance_tvd_m")
        horiz_tol = _num("residual_tolerance_horizontal_m")
        fail_thresh = _num("residual_fail_threshold_m")

        if header_tol < 0 or tvd_tol < 0 or horiz_tol < 0 or fail_thresh < 0:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r} tolerances must be non-negative."
            )
        if fail_thresh <= max(tvd_tol, horiz_tol):
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.residual_fail_threshold_m ({fail_thresh}) must exceed "
                f"both residual_tolerance_tvd_m ({tvd_tol}) and residual_tolerance_horizontal_m "
                f"({horiz_tol}) - otherwise there is no WARNING band between pass and fail."
            )
        if md_min > md_max:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_md_min_m ({md_min}) exceeds "
                f"expected_md_max_m ({md_max})."
            )

        station_count = d["expected_station_count"]
        if isinstance(station_count, bool) or not isinstance(station_count, int) or station_count <= 0:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.expected_station_count must be a positive integer."
            )

        azim_ref = d["azimuth_reference_for_grid_coordinates"]
        if azim_ref not in _VALID_AZIMUTH_REFERENCES:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.azimuth_reference_for_grid_coordinates must be one of "
                f"{_VALID_AZIMUTH_REFERENCES}, got {azim_ref!r}."
            )
        if azim_ref not in col_order:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.azimuth_reference_for_grid_coordinates ({azim_ref!r}) "
                f"is not one of its own declared expected_column_order."
            )

        depth_basis = d["depth_basis_policy"]
        if depth_basis not in VALID_DEPTH_BASIS_POLICIES:
            raise DeviationContractDefinitionError(
                f"{yaml_path}: {filename!r}.depth_basis_policy must be one of "
                f"{VALID_DEPTH_BASIS_POLICIES}, got {depth_basis!r}."
            )

        for str_field in (
            "expected_sha256",
            "expected_well_identifier",
            "expected_survey_identifier",
            "expected_coordinate_reference_system",
            "expected_datum_reference",
            "source_depth_convention",
            "notes",
        ):
            if not isinstance(d[str_field], str) or not d[str_field].strip():
                raise DeviationContractDefinitionError(
                    f"{yaml_path}: {filename!r}.{str_field} must be a non-empty string."
                )

        contracts[filename] = DeviationFileContract(
            source_filename=filename,
            expected_sha256=d["expected_sha256"].strip().lower(),
            expected_well_identifier=d["expected_well_identifier"],
            expected_survey_identifier=d["expected_survey_identifier"],
            expected_coordinate_reference_system=d["expected_coordinate_reference_system"],
            expected_wellhead_x_m=wellhead_x,
            expected_wellhead_y_m=wellhead_y,
            expected_datum_m=datum,
            expected_datum_reference=d["expected_datum_reference"],
            expected_column_count=col_count,
            expected_column_order=tuple(col_order),
            expected_units=dict(units),
            expected_station_count=station_count,
            expected_md_min_m=md_min,
            expected_md_max_m=md_max,
            azimuth_reference_for_grid_coordinates=azim_ref,
            source_depth_convention=d["source_depth_convention"],
            header_tolerance_m=header_tol,
            residual_tolerance_tvd_m=tvd_tol,
            residual_tolerance_horizontal_m=horiz_tol,
            residual_fail_threshold_m=fail_thresh,
            depth_basis_policy=depth_basis,
            notes=d["notes"],
        )

    return contracts


# ---------------------------------------------------------------------------
# Header parsing
# ---------------------------------------------------------------------------
def parse_deviation_header(path: str) -> Tuple[DeviationHeaderInfo, int, Tuple[str, ...]]:
    """
    Parse a Petrel deviation-survey file's header block (everything up to
    and including the second "#===...===" separator line).

    Returns `(header_info, data_start_line_index, column_names)`, where
    `data_start_line_index` is the zero-based index of the first station
    DATA line (i.e. the line immediately after the second separator), and
    `column_names` is the file's own literal, order-preserving column
    name tuple as read from its column-header line (NOT yet checked
    against any contract's `expected_column_order`).

    Raises `DeviationFileNotFoundError` if the path does not exist, and
    `DeviationParsingError` for any structural header defect: an
    unrecognized banner line, a missing required header field, fewer or
    more than exactly two "#===" separator lines before the data begins,
    a missing/duplicated column name, or a malformed WELL HEAD /
    WELL DATUM numeric-and-unit line.
    """
    p = Path(path)
    if not p.exists():
        raise DeviationFileNotFoundError(f"Deviation-survey file not found: {path}")

    sha256 = hashlib.sha256(p.read_bytes()).hexdigest()
    lines = p.read_text(encoding="utf-8").splitlines()

    if not lines or not _BANNER_RE.match(lines[0].strip()):
        raise DeviationParsingError(
            f"{path}: first line does not match the expected Petrel well-trace banner "
            f"('# WELL TRACE FROM PETREL ...'); got {lines[0] if lines else '<empty file>'!r}."
        )

    well_name: Optional[str] = None
    survey_name: Optional[str] = None
    wellhead_x: Optional[float] = None
    wellhead_x_unit: Optional[str] = None
    wellhead_y: Optional[float] = None
    wellhead_y_unit: Optional[str] = None
    datum_value: Optional[float] = None
    datum_reference: Optional[str] = None
    datum_unit: Optional[str] = None
    well_type: Optional[str] = None
    depth_reference_statement: Optional[str] = None
    angle_unit_statement: Optional[str] = None
    crs_text: Optional[str] = None
    dx_dy_statement: Optional[str] = None
    z_statement: Optional[str] = None
    azim_tn_desc_found = False
    azim_gn_desc_found = False

    separator_positions: List[int] = []
    column_names: Optional[Tuple[str, ...]] = None
    data_start_index: Optional[int] = None

    i = 0
    n = len(lines)
    while i < n:
        raw_line = lines[i]
        line = raw_line.rstrip("\n")
        stripped = line.strip()

        if not stripped:
            i += 1
            continue

        if _SEPARATOR_RE.match(stripped):
            separator_positions.append(i)
            if len(separator_positions) == 2:
                data_start_index = i + 1
                i += 1
                break
            i += 1
            continue

        if not stripped.startswith("#"):
            if len(separator_positions) != 1:
                raise DeviationParsingError(
                    f"{path}: line {i + 1} ({stripped!r}) is not a '#' header line, but was found "
                    f"{'before any' if not separator_positions else 'after the'} '#===' separator; "
                    f"expected exactly one non-'#' column-header line strictly between the two "
                    f"'#===' separator lines."
                )
            if column_names is not None:
                raise DeviationParsingError(
                    f"{path}: more than one column-header line found between the two '#===' "
                    f"separators (line {i + 1}: {stripped!r})."
                )
            tokens = stripped.split()
            if len(set(tokens)) != len(tokens):
                dupes = sorted({t for t in tokens if tokens.count(t) > 1})
                raise DeviationParsingError(
                    f"{path}: column-header line contains duplicate column name(s): {dupes}."
                )
            missing_cols = set(REQUIRED_COLUMN_NAMES) - set(tokens)
            if missing_cols:
                raise DeviationParsingError(
                    f"{path}: column-header line is missing required column(s): {sorted(missing_cols)}."
                )
            column_names = tuple(tokens)
            i += 1
            continue

        # A '#'-prefixed header/info line - dispatch to a recognized field.
        m = _WELL_NAME_RE.match(stripped)
        if m:
            well_name = m.group(1)
            i += 1
            continue
        m = _SURVEY_RE.match(stripped)
        if m:
            survey_name = m.group(1)
            i += 1
            continue
        m = _WELLHEAD_X_RE.match(stripped)
        if m:
            wellhead_x = float(m.group(1))
            wellhead_x_unit = m.group(2)
            i += 1
            continue
        m = _WELLHEAD_Y_RE.match(stripped)
        if m:
            wellhead_y = float(m.group(1))
            wellhead_y_unit = m.group(2)
            i += 1
            continue
        m = _DATUM_RE.match(stripped)
        if m:
            datum_reference = m.group(1)
            datum_value = float(m.group(2))
            datum_unit = m.group(3)
            i += 1
            continue
        m = _WELL_TYPE_RE.match(stripped)
        if m:
            well_type = m.group(1)
            i += 1
            continue
        m = _DEPTH_REF_RE.match(stripped)
        if m:
            depth_reference_statement = m.group(1)
            i += 1
            continue
        m = _ANGLE_UNIT_RE.match(stripped)
        if m:
            angle_unit_statement = m.group(1)
            i += 1
            continue
        m = _CRS_RE.match(stripped)
        if m:
            crs_text = m.group(1)
            i += 1
            continue
        if _AZIM_TN_DESC_RE.match(stripped):
            azim_tn_desc_found = True
            i += 1
            continue
        if _AZIM_GN_DESC_RE.match(stripped):
            azim_gn_desc_found = True
            i += 1
            continue
        m = _DXDY_RE.match(stripped)
        if m:
            dx_dy_statement = m.group(1)
            i += 1
            continue
        m = _Z_RE.match(stripped)
        if m:
            z_statement = m.group(1)
            i += 1
            continue
        if _BANNER_RE.match(stripped):
            i += 1
            continue

        # An unrecognized '#' line before the first separator is tolerated
        # (a vendor may add an extra informational comment) but every
        # REQUIRED field below is still checked for presence afterward.
        i += 1

    if len(separator_positions) != 2:
        raise DeviationParsingError(
            f"{path}: expected exactly two '#===...===' header separator lines; "
            f"found {len(separator_positions)}."
        )
    if column_names is None:
        raise DeviationParsingError(
            f"{path}: no column-header line found between the two '#===' separators."
        )

    required_fields = {
        "WELL NAME": well_name,
        "DEFINITIVE SURVEY": survey_name,
        "WELL HEAD X-COORDINATE": wellhead_x,
        "WELL HEAD Y-COORDINATE": wellhead_y,
        "WELL DATUM": datum_value,
        "WELL TYPE": well_type,
        "MD/TVD depth-reference statement": depth_reference_statement,
        "ANGLES unit statement": angle_unit_statement,
        "coordinate-reference-system statement": crs_text,
        "AZIM_TN description line": azim_tn_desc_found or None,
        "AZIM_GN description line": azim_gn_desc_found or None,
        "DX/DY unit statement": dx_dy_statement,
        "Z/TVD unit statement": z_statement,
    }
    missing = [name for name, value in required_fields.items() if value is None]
    if missing:
        raise DeviationParsingError(f"{path}: missing required header field(s): {missing}")

    if wellhead_x_unit != "m" or wellhead_y_unit != "m":
        raise DeviationParsingError(
            f"{path}: unsupported wellhead-coordinate unit ({wellhead_x_unit!r}/{wellhead_y_unit!r}); "
            f"only metres ('m') is implemented."
        )
    if datum_unit != "m":
        raise DeviationParsingError(
            f"{path}: unsupported well-datum unit ({datum_unit!r}); only metres ('m') is implemented."
        )
    if angle_unit_statement.strip().rstrip(".").lower() != "degrees":
        raise DeviationParsingError(
            f"{path}: unsupported angle-unit convention {angle_unit_statement!r}; "
            f"only 'DEGREES' is implemented."
        )
    if "m-units" not in dx_dy_statement.lower().replace(" ", ""):
        raise DeviationParsingError(
            f"{path}: unsupported DX/DY unit convention {dx_dy_statement!r}; "
            f"only metre ('m-UNITS') is implemented."
        )
    if "m-units" not in z_statement.lower().replace(" ", ""):
        raise DeviationParsingError(
            f"{path}: unsupported Z/TVD unit convention {z_statement!r}; "
            f"only metre ('m-UNITS') is implemented."
        )

    header = DeviationHeaderInfo(
        source_path=str(path),
        source_filename=p.name,
        sha256=sha256,
        well_name=well_name,
        survey_name=survey_name,
        wellhead_x_m=wellhead_x,
        wellhead_y_m=wellhead_y,
        datum_elevation_m=datum_value,
        datum_reference=datum_reference,
        well_type=well_type,
        coordinate_reference_system=crs_text,
        depth_reference_statement=depth_reference_statement,
        angle_unit_statement=angle_unit_statement,
        dx_dy_statement=dx_dy_statement,
        z_statement=z_statement,
        column_names=column_names,
        header_line_count=data_start_index - 1,  # everything before the data rows, incl. both '#===' lines
        data_line_offset=data_start_index,
    )
    return header, data_start_index, column_names


# ---------------------------------------------------------------------------
# Station-data parsing
# ---------------------------------------------------------------------------
def _parse_float_strict(token: str, *, path: str, line_no: int, col_name: str) -> float:
    try:
        value = float(token)
    except ValueError as exc:
        raise DeviationParsingError(
            f"{path}: line {line_no}: non-numeric token {token!r} in column {col_name!r}."
        ) from exc
    if not math.isfinite(value):
        raise DeviationParsingError(
            f"{path}: line {line_no}: non-finite literal token {token!r} in column {col_name!r} "
            f"(NaN/Inf tokens are never accepted as station data)."
        )
    return value


def read_deviation_stations(
    path: str, data_start_index: int, column_names: Tuple[str, ...]
) -> DeviationStationData:
    """
    Parse every station data row starting at `data_start_index` (as
    returned by `parse_deviation_header`), using `column_names` (the
    file's own literal column order) to place each numeric token.

    Raises `DeviationParsingError` for a row whose token count does not
    match `len(column_names)` (row-width mismatch), or any non-numeric or
    non-finite (NaN/Inf) token.
    """
    p = Path(path)
    lines = p.read_text(encoding="utf-8").splitlines()

    rows: List[List[float]] = []
    for idx in range(data_start_index, len(lines)):
        raw_line = lines[idx]
        stripped = raw_line.strip()
        if not stripped:
            continue
        if stripped.startswith("#"):
            raise DeviationParsingError(
                f"{path}: line {idx + 1}: unexpected '#' comment line found after station data begins "
                f"(comments are only recognized in the header block, before the second '#===' separator)."
            )
        tokens = stripped.split()
        if len(tokens) != len(column_names):
            raise DeviationParsingError(
                f"{path}: line {idx + 1}: expected {len(column_names)} column(s), found {len(tokens)} "
                f"(row-width mismatch)."
            )
        row = [
            _parse_float_strict(tok, path=path, line_no=idx + 1, col_name=col)
            for tok, col in zip(tokens, column_names)
        ]
        rows.append(row)

    if not rows:
        raise DeviationParsingError(f"{path}: no station data rows found after the header.")

    arr = np.array(rows, dtype=np.float64)
    col_index = {name: k for k, name in enumerate(column_names)}
    kwargs = {
        _COLUMN_TO_FIELD[name]: arr[:, col_index[name]] for name in REQUIRED_COLUMN_NAMES
    }
    return DeviationStationData(**kwargs)


# ---------------------------------------------------------------------------
# Contract resolution
# ---------------------------------------------------------------------------
def resolve_deviation_contract(
    header: DeviationHeaderInfo,
    column_names: Tuple[str, ...],
    stations: DeviationStationData,
    contract: DeviationFileContract,
) -> Tuple[DeviationIngestionIssue, ...]:
    """
    Compare a file's actual parsed header and station data against its
    `DeviationFileContract`, returning the full tuple of
    `DeviationIngestionIssue` (ERROR and WARNING). Every check below is
    performed independently (no short-circuiting on the first mismatch),
    so a caller sees every problem in one pass, not just the first one
    encountered.

    ERROR-severity checks: filename, SHA-256, well identifier, survey
    identifier, coordinate-reference-system text, wellhead X/Y (within
    `header_tolerance_m`), datum value and reference text (within
    `header_tolerance_m` for the value), column order (exact match),
    column count, station count, MD coverage (min/max, within
    `header_tolerance_m`), duplicate/non-monotonic MD, and
    physically-invalid inclination (outside [0, 180] degrees) - the
    inclination check is ERROR-severity here (blocking), distinct from
    `p2mem.trajectory`'s own independent, second-gate rejection of the
    same condition at trajectory-computation time.

    WARNING-severity checks: the disclosed MD-unit inference (see module
    docstring). The disclosed DLS-normalization inference is added
    separately, in `load_deviation_file`, after a successful minimum-
    curvature computation is available to independently verify it against
    (see that function's docstring) - it is not one of the issues
    returned directly by this function.

    Every issue's `context` field is set to `header.source_filename`
    (the file's basename only, e.g. "Poseidon 2_dev.txt") rather than
    `header.source_path` (the full filesystem path, which is
    environment-dependent - a Colab Drive mount path, a local
    Increment-3.1-corrective-patch build path, or a CI temp directory
    would each differ for byte-identical data). This keeps every exported
    CSV/JSON deliverable free of build-environment-specific absolute
    paths (Increment 3.1 correction - see the Increment 3.1 manifest);
    `header.source_path` remains available on the `DeviationHeaderInfo`
    object itself for interactive debugging.
    """
    issues: List[DeviationIngestionIssue] = []

    def err(code: str, message: str, context: str = "") -> None:
        issues.append(DeviationIngestionIssue("ERROR", code, message, context))

    def warn(code: str, message: str, context: str = "") -> None:
        issues.append(DeviationIngestionIssue("WARNING", code, message, context))

    if header.source_filename != contract.source_filename:
        err(
            "FILENAME_MISMATCH",
            f"Loaded file basename {header.source_filename!r} does not match contract's "
            f"declared filename {contract.source_filename!r}.",
            header.source_filename,
        )
    if header.sha256.lower() != contract.expected_sha256.lower():
        err(
            "SHA256_MISMATCH",
            f"File SHA-256 {header.sha256} does not match contract's expected "
            f"{contract.expected_sha256}.",
            header.source_filename,
        )
    if header.well_name != contract.expected_well_identifier:
        err(
            "WELL_IDENTIFIER_MISMATCH",
            f"Parsed WELL NAME {header.well_name!r} does not match contract's expected "
            f"well identifier {contract.expected_well_identifier!r}.",
            header.source_filename,
        )
    if header.survey_name != contract.expected_survey_identifier:
        err(
            "SURVEY_IDENTIFIER_MISMATCH",
            f"Parsed DEFINITIVE SURVEY {header.survey_name!r} does not match contract's "
            f"expected {contract.expected_survey_identifier!r}.",
            header.source_filename,
        )
    if header.coordinate_reference_system != contract.expected_coordinate_reference_system:
        err(
            "CRS_MISMATCH",
            f"Parsed coordinate-reference-system text {header.coordinate_reference_system!r} "
            f"does not match contract's expected {contract.expected_coordinate_reference_system!r}.",
            header.source_filename,
        )
    if abs(header.wellhead_x_m - contract.expected_wellhead_x_m) > contract.header_tolerance_m:
        err(
            "WELLHEAD_X_MISMATCH",
            f"Parsed wellhead X ({header.wellhead_x_m}) differs from contract's expected "
            f"({contract.expected_wellhead_x_m}) by more than tolerance ({contract.header_tolerance_m} m).",
            header.source_filename,
        )
    if abs(header.wellhead_y_m - contract.expected_wellhead_y_m) > contract.header_tolerance_m:
        err(
            "WELLHEAD_Y_MISMATCH",
            f"Parsed wellhead Y ({header.wellhead_y_m}) differs from contract's expected "
            f"({contract.expected_wellhead_y_m}) by more than tolerance ({contract.header_tolerance_m} m).",
            header.source_filename,
        )
    if abs(header.datum_elevation_m - contract.expected_datum_m) > contract.header_tolerance_m:
        err(
            "DATUM_MISMATCH",
            f"Parsed well datum ({header.datum_elevation_m}) differs from contract's expected "
            f"({contract.expected_datum_m}) by more than tolerance ({contract.header_tolerance_m} m).",
            header.source_filename,
        )
    if header.datum_reference != contract.expected_datum_reference:
        err(
            "DATUM_REFERENCE_MISMATCH",
            f"Parsed datum-reference text {header.datum_reference!r} does not match contract's "
            f"expected {contract.expected_datum_reference!r}.",
            header.source_filename,
        )
    if column_names != contract.expected_column_order:
        err(
            "COLUMN_ORDER_MISMATCH",
            f"Parsed column order {column_names} does not match contract's required exact order "
            f"{contract.expected_column_order}.",
            header.source_filename,
        )
    if len(column_names) != contract.expected_column_count:
        err(
            "COLUMN_COUNT_MISMATCH",
            f"Parsed column count ({len(column_names)}) does not match contract's expected "
            f"({contract.expected_column_count}).",
            header.source_filename,
        )

    n_stations = stations.MD_source_m.size
    if n_stations != contract.expected_station_count:
        err(
            "STATION_COUNT_MISMATCH",
            f"Parsed station count ({n_stations}) does not match contract's expected "
            f"({contract.expected_station_count}).",
            header.source_filename,
        )
    md_min = float(np.min(stations.MD_source_m))
    md_max = float(np.max(stations.MD_source_m))
    if abs(md_min - contract.expected_md_min_m) > contract.header_tolerance_m:
        err(
            "MD_MIN_MISMATCH",
            f"Parsed minimum MD ({md_min}) differs from contract's expected "
            f"({contract.expected_md_min_m}) by more than tolerance ({contract.header_tolerance_m} m).",
            header.source_filename,
        )
    if abs(md_max - contract.expected_md_max_m) > contract.header_tolerance_m:
        err(
            "MD_MAX_MISMATCH",
            f"Parsed maximum MD ({md_max}) differs from contract's expected "
            f"({contract.expected_md_max_m}) by more than tolerance ({contract.header_tolerance_m} m).",
            header.source_filename,
        )

    # --- station-level QC (ERROR-severity structural checks) ---
    n_dup = int(np.sum(np.diff(stations.MD_source_m) == 0.0)) if n_stations > 1 else 0
    if n_dup > 0:
        err(
            "DUPLICATE_MD",
            f"{n_dup} duplicate consecutive measured-depth value(s) found.",
            header.source_filename,
        )
    n_non_monotonic = (
        int(np.sum(np.diff(stations.MD_source_m) < 0.0)) if n_stations > 1 else 0
    )
    if n_non_monotonic > 0:
        err(
            "NON_MONOTONIC_MD",
            f"{n_non_monotonic} decreasing measured-depth step(s) found; MD must be "
            f"non-decreasing.",
            header.source_filename,
        )
    invalid_incl_mask = (stations.INCL_source_deg < 0.0) | (stations.INCL_source_deg > 180.0)
    if np.any(invalid_incl_mask):
        bad_idx = np.where(invalid_incl_mask)[0].tolist()
        err(
            "INVALID_INCLINATION",
            f"{len(bad_idx)} station(s) with inclination outside the physically valid range "
            f"[0, 180] degrees at row index(es) {bad_idx}.",
            header.source_filename,
        )

    # --- WARNING-severity disclosures ---
    warn(
        MD_UNIT_NOT_EXPLICITLY_DECLARED_CODE,
        "This file's header does not include an explicit unit statement for the MD column "
        "(only X/Y/DX/DY/Z/TVD are explicitly declared in metres, and AZIM_TN/INCL/AZIM_GN in "
        "degrees). MD is treated as metres by inference from its numeric consistency with this "
        "file's own explicitly-metric TVD column and the already-validated LAS MD_m convention "
        "for the same well - not from an explicit per-file statement.",
        header.source_filename,
    )

    return tuple(issues)


# ---------------------------------------------------------------------------
# Trajectory validation (source-vs-computed comparison)
# ---------------------------------------------------------------------------
def _status_for(max_abs_residual: float, tolerance: float, fail_threshold: float) -> str:
    if max_abs_residual <= tolerance:
        return STATUS_PASS
    if max_abs_residual <= fail_threshold:
        return STATUS_WARNING
    return STATUS_FAIL


_STATUS_RANK = {STATUS_PASS: 0, STATUS_WARNING: 1, STATUS_FAIL: 2}


def compute_well_trajectory_validation(
    well_key: str,
    header: DeviationHeaderInfo,
    stations: DeviationStationData,
    contract: DeviationFileContract,
) -> Tuple["_MCBundle", TrajectoryValidationResult]:
    """
    Compute the independent minimum-curvature trajectory (using the
    contract-declared `azimuth_reference_for_grid_coordinates`) and the
    full `TrajectoryValidationResult` comparing it against the Petrel-
    supplied source trajectory, plus the three internal Petrel-source
    self-consistency checks (X ~= X_wellhead + DX, Y ~= Y_wellhead + DY,
    Z ~= Datum - TVD).

    Raises `p2mem.trajectory.TrajectoryComputationError` if the station
    data cannot be safely passed to minimum-curvature computation (this
    should already be impossible for a file whose station-level QC in
    `resolve_deviation_contract` reported no ERROR, since that function
    checks MD monotonicity and inclination range first - this is a
    second, independent gate, not the primary one).
    """
    azim_ref = contract.azimuth_reference_for_grid_coordinates
    azim_deg = (
        stations.AZIM_GN_source_deg if azim_ref == "AZIM_GN" else stations.AZIM_TN_source_deg
    )

    mc = compute_minimum_curvature_trajectory(
        stations.MD_source_m,
        stations.INCL_source_deg,
        azim_deg,
        tvd_origin_m=float(stations.TVD_source_m[0]),
        northing_origin_m=float(stations.DY_source_m[0]),
        easting_origin_m=float(stations.DX_source_m[0]),
    )

    tvd_res = mc.tvd_mc_m - stations.TVD_source_m
    easting_res = mc.easting_offset_mc_m - stations.DX_source_m
    northing_res = mc.northing_offset_mc_m - stations.DY_source_m

    def _stats(res: np.ndarray) -> Tuple[float, float, float, float]:
        return (
            float(np.max(np.abs(res))),
            float(np.mean(res)),
            float(np.sqrt(np.mean(res**2))),
            float(res[-1]),
        )

    tvd_max, tvd_mean, tvd_rmse, tvd_end = _stats(tvd_res)
    e_max, e_mean, e_rmse, e_end = _stats(easting_res)
    n_max, n_mean, n_rmse, n_end = _stats(northing_res)

    tvd_status = _status_for(tvd_max, contract.residual_tolerance_tvd_m, contract.residual_fail_threshold_m)
    e_status = _status_for(e_max, contract.residual_tolerance_horizontal_m, contract.residual_fail_threshold_m)
    n_status = _status_for(n_max, contract.residual_tolerance_horizontal_m, contract.residual_fail_threshold_m)

    # Internal Petrel-source self-consistency: X ~= X_wellhead + DX, etc.
    x_check = stations.X_source_m - (header.wellhead_x_m + stations.DX_source_m)
    y_check = stations.Y_source_m - (header.wellhead_y_m + stations.DY_source_m)
    z_check = stations.Z_source_m - (header.datum_elevation_m - stations.TVD_source_m)
    x_max = float(np.max(np.abs(x_check)))
    y_max = float(np.max(np.abs(y_check)))
    z_max = float(np.max(np.abs(z_check)))
    x_status = _status_for(x_max, contract.residual_tolerance_horizontal_m, contract.residual_fail_threshold_m)
    y_status = _status_for(y_max, contract.residual_tolerance_horizontal_m, contract.residual_fail_threshold_m)
    z_status = _status_for(z_max, contract.residual_tolerance_tvd_m, contract.residual_fail_threshold_m)

    overall_status = max(
        [tvd_status, e_status, n_status, x_status, y_status, z_status],
        key=lambda s: _STATUS_RANK[s],
    )

    validation = TrajectoryValidationResult(
        well_key=well_key,
        comparison_basis=(
            f"TVD_mc_m vs TVD_source_m; EASTING_offset_mc_m vs DX_source_m; "
            f"NORTHING_offset_mc_m vs DY_source_m (grid-coordinate azimuth reference: {azim_ref}); "
            f"plus internal Petrel-source consistency X~=X_wellhead+DX, Y~=Y_wellhead+DY, "
            f"Z~=Datum-TVD."
        ),
        tvd_max_abs_residual_m=tvd_max,
        tvd_mean_residual_m=tvd_mean,
        tvd_rmse_m=tvd_rmse,
        tvd_endpoint_residual_m=tvd_end,
        tvd_tolerance_m=contract.residual_tolerance_tvd_m,
        tvd_status=tvd_status,
        easting_max_abs_residual_m=e_max,
        easting_mean_residual_m=e_mean,
        easting_rmse_m=e_rmse,
        easting_endpoint_residual_m=e_end,
        easting_tolerance_m=contract.residual_tolerance_horizontal_m,
        easting_status=e_status,
        northing_max_abs_residual_m=n_max,
        northing_mean_residual_m=n_mean,
        northing_rmse_m=n_rmse,
        northing_endpoint_residual_m=n_end,
        northing_tolerance_m=contract.residual_tolerance_horizontal_m,
        northing_status=n_status,
        x_consistency_max_abs_residual_m=x_max,
        x_consistency_status=x_status,
        y_consistency_max_abs_residual_m=y_max,
        y_consistency_status=y_status,
        z_consistency_max_abs_residual_m=z_max,
        z_consistency_status=z_status,
        overall_status=overall_status,
        origin_initialization_note=(
            f"Minimum-curvature trajectory initialized at station 0 using this well's own "
            f"source TVD ({float(stations.TVD_source_m[0]):.9g} m), DX "
            f"({float(stations.DX_source_m[0]):.9g} m), and DY "
            f"({float(stations.DY_source_m[0]):.9g} m) as the tie-on origin - never a "
            f"hard-coded (0, 0, 0)."
        ),
    )
    return mc, validation


# ---------------------------------------------------------------------------
# Per-file and batch loaders
# ---------------------------------------------------------------------------
def load_deviation_file(path: str, contract: DeviationFileContract) -> DeviationWellResult:
    """
    Parse, contract-resolve, and trajectory-validate one deviation-survey
    file, returning a complete `DeviationWellResult`.

    Raises `DeviationFileNotFoundError`, `DeviationParsingError`, or
    `DeviationContractError` (never returns a partially valid result -
    only a successfully resolved file is returned).

    DLS-normalization disclosure (Increment 3.1 addition)
    -------------------------------------------------------
    None of the four approved files' headers explicitly states that the
    supplied `DLS` column is normalized as degrees-per-30-metres (the
    header declares angles are in degrees, but says nothing about DLS's
    own length normalization). `deg/30m` is the standard oilfield
    convention and is scientifically defensible here, but it is an
    INFERENCE, not a header-declared unit - exactly like the pre-existing
    `MD_UNIT_NOT_EXPLICITLY_DECLARED` disclosure for the MD column. This
    function verifies that inference independently for every successfully
    loaded file (not merely asserts it): it recomputes each station's
    dogleg severity from that same file's own MD/inclination/azimuth
    columns via `p2mem.trajectory` (already computed a moment earlier, as
    part of `mc`, for the trajectory-validation comparison above) and
    compares it, station by station, against the file's own supplied
    `DLS_source_deg_per_30m` column. The resulting `DeviationIngestionIssue`
    (code `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M`, always WARNING
    severity, never blocking) reports the ACTUAL maximum discrepancy found
    for THIS file - never a hard-coded or assumed number - and explicitly
    states that the raw `DLS_source_deg_per_30m` array itself is never
    altered by this check (it is a read-only comparison).
    """
    header, data_start, column_names = parse_deviation_header(path)
    stations = read_deviation_stations(path, data_start, column_names)
    issues = resolve_deviation_contract(header, column_names, stations, contract)

    error_issues = tuple(i for i in issues if i.severity == "ERROR")
    if error_issues:
        raise DeviationContractError(
            f"{path}: {len(error_issues)} contract-resolution ERROR(s): "
            + "; ".join(f"[{i.code}] {i.message}" for i in error_issues),
            issues,
        )

    mc, validation = compute_well_trajectory_validation(
        well_key=contract.expected_well_identifier, header=header, stations=stations, contract=contract
    )

    dls_recomputed_vs_source_abs_diff = np.abs(
        stations.DLS_source_deg_per_30m - mc.dls_deg_per_30m
    )
    dls_max_abs_diff = float(np.max(dls_recomputed_vs_source_abs_diff))
    dls_issue = DeviationIngestionIssue(
        "WARNING",
        DLS_NORMALIZATION_INFERRED_CODE,
        "This file's header declares angles (AZIM_TN/INCL/AZIM_GN) are in degrees but does not "
        "explicitly state the length-normalization of the supplied DLS column. 'degrees per 30 "
        "metres' was inferred (the standard oilfield dogleg-severity convention) and independently "
        "verified, not merely assumed: this file's own supplied DLS_source_deg_per_30m column was "
        "compared, station by station, against dogleg severity recomputed from this same file's own "
        f"MD/inclination/azimuth columns (p2mem.trajectory), giving a maximum absolute discrepancy of "
        f"{dls_max_abs_diff:.6e} deg/30m across {stations.MD_source_m.size} stations. The raw "
        "DLS_source_deg_per_30m values are never altered by this check.",
        header.source_filename,
    )
    issues = issues + (dls_issue,)

    depth_basis = DepthBasisSelection(
        well_key=contract.expected_well_identifier,
        selected_basis=contract.depth_basis_policy,
        rationale=(
            f"Per config/deviation_survey_contracts.yml (declared uniformly for all four wells): "
            f"'{contract.depth_basis_policy}' is used as the downstream MD-to-TVD/TVDSS mapping "
            f"basis. The independently computed minimum-curvature trajectory "
            f"(overall_status={validation.overall_status}) is retained as a QC comparison but is "
            f"not itself used for depth mapping."
        )
        if contract.depth_basis_policy == DEPTH_BASIS_PETREL_SOURCE
        else (
            f"Per config/deviation_survey_contracts.yml: 'minimum_curvature_computed' is used as "
            f"the downstream MD-to-TVD/TVDSS mapping basis for this well "
            f"(overall_status={validation.overall_status})."
        ),
    )

    return DeviationWellResult(
        header=header,
        contract=contract,
        raw=stations,
        mc=mc,
        validation=validation,
        depth_basis=depth_basis,
        issues=issues,
        contract_status="PASSED",
    )


def load_deviation_surveys(
    file_paths: Dict[str, str], contracts: Dict[str, DeviationFileContract]
) -> Tuple[Dict[str, DeviationWellResult], Dict[str, DeviationIngestionFailure]]:
    """
    Load a batch of deviation-survey files keyed by well key (e.g.
    "Poseidon_2"), isolating expected per-well ingestion failures
    (file-not-found, structural-parsing, contract-resolution, or
    trajectory-computation) as typed `DeviationIngestionFailure` records.
    Any other exception (a programming error, not an expected ingestion-
    failure mode) propagates uncaught.

    `contracts` is keyed by SOURCE FILENAME (as in
    `config/deviation_survey_contracts.yml`), matching
    `load_deviation_contract_config`'s return type; `file_paths` is keyed
    by WELL KEY (e.g. "Poseidon_2") and gives that well's file path. The
    contract used for a given well is looked up by the path's basename.
    """
    results: Dict[str, DeviationWellResult] = {}
    failures: Dict[str, DeviationIngestionFailure] = {}

    for well_key, path in file_paths.items():
        basename = Path(path).name
        contract = contracts.get(basename)
        if contract is None:
            raise DeviationContractDefinitionError(
                f"No deviation contract found for file {basename!r} (well key {well_key!r}); "
                f"contracts are keyed by source filename and must be authored before ingestion."
            )
        try:
            results[well_key] = load_deviation_file(path, contract)
        except DeviationFileNotFoundError as exc:
            failures[well_key] = DeviationIngestionFailure(
                well_key=well_key, source_path=path, error_type="file_not_found",
                message=str(exc), exception=exc,
            )
        except DeviationParsingError as exc:
            failures[well_key] = DeviationIngestionFailure(
                well_key=well_key, source_path=path, error_type="parsing_failure",
                message=str(exc), exception=exc,
            )
        except DeviationContractError as exc:
            failures[well_key] = DeviationIngestionFailure(
                well_key=well_key, source_path=path, error_type="contract_failure",
                message=str(exc), exception=exc,
            )
        except TrajectoryComputationError as exc:
            failures[well_key] = DeviationIngestionFailure(
                well_key=well_key, source_path=path, error_type="trajectory_failure",
                message=str(exc), exception=exc,
            )

    return results, failures


#### `p2mem/depth_mapping.py` — MD-to-TVD/TVDSS interpolation

Maps the locked LAS `MD_m` array onto TVD/TVDSS using the explicitly selected depth basis for that well, via a documented, deterministic piecewise-linear interpolation of the validated survey-station trajectory. Extrapolation beyond surveyed MD coverage is rejected outright, never silently clamped.

In [ ]:
%%writefile p2mem/depth_mapping.py
"""
p2mem.depth_mapping - MD-to-TVD/TVDSS interpolation and coverage checks
(Increment 3).

Scope
-----
This module maps the locked Increment 2.1.1 LAS loader's canonical
`MD_m` array onto true vertical depth (TVD) and true vertical depth
subsea (TVDSS), using ONE explicitly selected well-trajectory basis (see
`p2mem.io.deviation.DepthBasisSelection` - either the Petrel-supplied
source trajectory or the independently computed minimum-curvature
trajectory) as the MD-TVD relationship to interpolate against.

It does not read LAS files, does not read deviation files, and does not
decide which basis to use - it is a pure, typed numerical mapping step
that takes (survey MD array, survey TVD array, datum elevation, LAS MD
array) and returns TVD/TVDSS at every LAS sample.

Depth-reference convention
----------------------------
For these wells, MD and TVD are both referenced to zero at the well
datum (rotary table) and increase downward; the well datum elevation
itself is referenced to mean sea level (MSL), positive upward. Therefore:

    TVDSS_m = TVD_m - DatumElevation_m

which is consistent with the Petrel-supplied elevation coordinate
Z_m = DatumElevation_m - TVD_m (positive upward), i.e. TVDSS_m == -Z_m.
Both relationships are independently exercised in `tests/test_depth_
mapping.py`.

Interpolation method
----------------------
A transparent, deterministic PIECEWISE-LINEAR interpolation of the
validated station trajectory (`numpy.interp`) is used - explicitly named
and documented as depth interpolation between discrete survey stations,
NOT a minimum-curvature recomputation at every log sample (minimum
curvature is a station-to-station method; re-deriving it at thousands of
LAS sample depths would not add trajectory information beyond what is
already captured by the station-level TVD values themselves, and would
silently blur the distinction between "the validated station trajectory"
and "a depth grid interpolated from it"). This method:

* is deterministic (no randomness, no fitted/optimized parameters);
* is numerically stable (linear interpolation has no ill-conditioning);
* preserves exact survey-station values (`numpy.interp` returns the
  station's own TVD exactly at an input MD that exactly equals a station
  MD - the underlying data structure is not resampled or smoothed);
* introduces no new dependency (`numpy.interp` is already a project
  dependency);
* never extrapolates silently - see `DepthMappingError` /
  `ExtrapolationRejectedError` below.
"""

from __future__ import annotations

import numpy as np

from p2mem.deviation_models import (
    DEPTH_BASIS_MINIMUM_CURVATURE,
    DEPTH_BASIS_PETREL_SOURCE,
    DeviationWellResult,
    LasDepthMappingResult,
)

__all__ = [
    "DepthMappingError",
    "ExtrapolationRejectedError",
    "INTERPOLATION_METHOD",
    "select_survey_trajectory_for_mapping",
    "map_las_md_to_tvd_tvdss",
]

INTERPOLATION_METHOD = "piecewise_linear_station_interpolation"


class DepthMappingError(RuntimeError):
    """
    Raised when LAS MD cannot be safely mapped to TVD/TVDSS: the survey
    trajectory for the selected basis is not strictly increasing in MD
    (which would make a well-defined piecewise-linear function
    impossible), or any other structural precondition for interpolation
    is not met.
    """


class ExtrapolationRejectedError(DepthMappingError):
    """
    Raised when one or more LAS MD samples fall outside the selected
    survey trajectory's MD coverage. This module never extrapolates
    silently (e.g. by clamping to the nearest station or holding the
    boundary TVD constant) - a LAS MD sample beyond survey coverage is a
    real data-coverage gap that must be reported, not hidden.
    """


def select_survey_trajectory_for_mapping(
    well_result: DeviationWellResult,
) -> tuple[np.ndarray, np.ndarray, str]:
    """
    Return `(survey_md_m, survey_tvd_m, basis_used)` for the depth basis
    explicitly selected for this well
    (`well_result.depth_basis.selected_basis`) - either the Petrel-
    supplied source TVD (`DeviationStationData.TVD_source_m`) or the
    independently computed minimum-curvature TVD
    (`MinimumCurvatureResult.tvd_mc_m`). The MD array is always the
    survey's own source MD (`DeviationStationData.MD_source_m` - MD is
    never recomputed, only TVD differs between the two bases).
    """
    basis = well_result.depth_basis.selected_basis
    if basis == DEPTH_BASIS_PETREL_SOURCE:
        return well_result.raw.MD_source_m, well_result.raw.TVD_source_m, basis
    if basis == DEPTH_BASIS_MINIMUM_CURVATURE:
        return well_result.raw.MD_source_m, well_result.mc.tvd_mc_m, basis
    raise DepthMappingError(
        f"Unrecognized depth_basis_policy {basis!r} for well {well_result.depth_basis.well_key!r}."
    )


def map_las_md_to_tvd_tvdss(
    well_key: str,
    las_md_source_m: np.ndarray,
    well_result: DeviationWellResult,
) -> LasDepthMappingResult:
    """
    Map a well's canonical LAS `MD_m` array onto TVD and TVDSS using the
    explicitly selected depth basis for that well.

    Preconditions checked, in order:
    1. The selected survey trajectory's MD array is strictly increasing
       (`DepthMappingError` if not - this should already be guaranteed
       for the Petrel-source basis by `resolve_deviation_contract`'s
       duplicate/non-monotonic-MD checks, and for the minimum-curvature
       basis by construction, but this is re-verified independently here
       rather than assumed).
    2. Every LAS MD sample lies within `[survey_md_min, survey_md_max]`
       (`ExtrapolationRejectedError` naming the offending sample count and
       the exact out-of-coverage margin if not).

    `las_md_source_m` is returned unmodified as `LasDepthMappingResult
    .las_md_source_m` - this function never overwrites or resamples the
    original LAS MD array, only computes TVD/TVDSS at each of its
    existing sample depths.
    """
    survey_md, survey_tvd, basis_used = select_survey_trajectory_for_mapping(well_result)

    if survey_md.size < 2 or not np.all(np.diff(survey_md) > 0.0):
        raise DepthMappingError(
            f"{well_key}: selected survey trajectory (basis={basis_used!r}) MD array is not "
            f"strictly increasing; cannot construct a well-defined piecewise-linear MD->TVD map."
        )

    las_md = np.asarray(las_md_source_m, dtype=np.float64)
    if las_md.ndim != 1 or las_md.size == 0:
        raise DepthMappingError(f"{well_key}: LAS MD array must be a non-empty 1-D array.")
    if not np.all(np.isfinite(las_md)):
        raise DepthMappingError(f"{well_key}: LAS MD array contains non-finite value(s).")

    survey_md_min = float(survey_md[0])
    survey_md_max = float(survey_md[-1])
    las_md_min = float(np.min(las_md))
    las_md_max = float(np.max(las_md))

    below = las_md < survey_md_min
    above = las_md > survey_md_max
    n_extrapolated_would_be = int(np.sum(below) + np.sum(above))
    if n_extrapolated_would_be > 0:
        margin_below = survey_md_min - las_md_min if las_md_min < survey_md_min else 0.0
        margin_above = las_md_max - survey_md_max if las_md_max > survey_md_max else 0.0
        raise ExtrapolationRejectedError(
            f"{well_key}: {n_extrapolated_would_be} LAS MD sample(s) fall outside the selected "
            f"survey trajectory's MD coverage [{survey_md_min:.4f}, {survey_md_max:.4f}] m "
            f"(basis={basis_used!r}); LAS MD range is [{las_md_min:.4f}, {las_md_max:.4f}] m "
            f"(below-coverage margin {margin_below:.6f} m, above-coverage margin "
            f"{margin_above:.6f} m). Extrapolation is rejected by default - this function never "
            f"silently extends the trajectory beyond its surveyed range."
        )

    tvd_mapped = np.interp(las_md, survey_md, survey_tvd)
    tvdss_mapped = tvd_mapped - well_result.header.datum_elevation_m

    return LasDepthMappingResult(
        well_key=well_key,
        depth_basis_used=basis_used,
        interpolation_method=INTERPOLATION_METHOD,
        las_md_source_m=las_md,
        tvd_mapped_m=tvd_mapped,
        tvdss_mapped_m=tvdss_mapped,
        n_samples=int(las_md.size),
        survey_md_min_m=survey_md_min,
        survey_md_max_m=survey_md_max,
        las_md_min_m=las_md_min,
        las_md_max_m=las_md_max,
        coverage_margin_lower_m=las_md_min - survey_md_min,
        coverage_margin_upper_m=survey_md_max - las_md_max,
        n_extrapolated=0,
        datum_elevation_m=well_result.header.datum_elevation_m,
    )


#### `p2mem/io/deviation_inventory.py` — deterministic inventory-table builders

Mirrors `p2mem/io/inventory.py` (the locked LAS-layer inventory builder) for the Increment 3 outputs, as a separate module so that locked file is never touched.

In [ ]:
%%writefile p2mem/io/deviation_inventory.py
"""
p2mem.io.deviation_inventory - Deterministic, metadata-only inventory-table
builders for the Increment 3 deviation-survey / depth-mapping layer.

Mirrors the design of `p2mem.io.inventory` (the locked LAS-layer inventory
builder) but is a separate module so that file is never modified. Every
function here returns a list of plain dicts (one per output row), ready
for `csv.DictWriter` - never raw per-sample station or LAS arrays (those
stay out of the packaged deliverable; see the Increment 3 manifest).

Environment-independent output (Increment 3.1 correction)
-------------------------------------------------------------
Every row built here uses only a file's BASENAME (e.g.
"Poseidon 2_dev.txt") for `source_filename` and issue `context` fields -
never `DeviationIngestionFailure.source_path` or any other full
filesystem path verbatim. A full path is environment-dependent (a Colab
Drive mount path, a local development build path, a CI temp directory)
even for byte-identical source data, so embedding one in an exported
CSV/JSON deliverable would make that deliverable non-reproducible across
environments. `DeviationIngestionFailure.source_path` remains available
on the underlying typed object for interactive debugging; it is only the
row-builders here (the functions that feed the packaged, deterministic
outputs) that deliberately reduce it to a basename.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List

from p2mem.deviation_models import DeviationIngestionFailure, DeviationWellResult, LasDepthMappingResult

__all__ = [
    "build_deviation_file_inventory_rows",
    "build_trajectory_validation_rows",
    "build_depth_reference_register_rows",
    "build_las_depth_mapping_rows",
    "build_deviation_issues_rows",
    "build_deviation_depth_manifest",
]


def _sanitize_message(message: str, source_path: str) -> str:
    """
    Replace a literal occurrence of `source_path` (a full filesystem path,
    potentially environment-dependent - a Colab Drive mount path, a local
    build path, a CI temp directory) inside an exception/failure `message`
    string with just that path's basename, so a failure row's exported
    `error_message` never embeds an absolute, environment-specific path
    (Increment 3.1 correction). This only ever matters for a well that
    actually failed to load - every well in this project's real four-well
    delivery loads successfully, so `failed_wells`/failure rows are empty
    in the actual shipped outputs; this function exists so that remains
    true even if a future run does encounter a load failure.
    """
    basename = Path(source_path).name
    return message.replace(source_path, basename)


def build_deviation_file_inventory_rows(
    results: Dict[str, DeviationWellResult], failures: Dict[str, DeviationIngestionFailure]
) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(set(results) | set(failures)):
        if well_key in results:
            r = results[well_key]
            n_errors = sum(1 for i in r.issues if i.severity == "ERROR")
            n_warnings = sum(1 for i in r.issues if i.severity == "WARNING")
            rows.append(
                {
                    "well_key": well_key,
                    "source_filename": r.header.source_filename,
                    "sha256": r.header.sha256,
                    "well_name": r.header.well_name,
                    "survey_name": r.header.survey_name,
                    "well_type": r.header.well_type,
                    "wellhead_x_m": r.header.wellhead_x_m,
                    "wellhead_y_m": r.header.wellhead_y_m,
                    "datum_elevation_m": r.header.datum_elevation_m,
                    "datum_reference": r.header.datum_reference,
                    "coordinate_reference_system": r.header.coordinate_reference_system,
                    "n_stations": int(r.raw.MD_source_m.size),
                    "md_min_m": float(r.raw.MD_source_m.min()),
                    "md_max_m": float(r.raw.MD_source_m.max()),
                    "max_inclination_deg": float(r.raw.INCL_source_deg.max()),
                    "contract_status": r.contract_status,
                    "n_errors": n_errors,
                    "n_warnings": n_warnings,
                    "error_type": "",
                    "error_message": "",
                }
            )
        else:
            f = failures[well_key]
            rows.append(
                {
                    "well_key": well_key,
                    "source_filename": Path(f.source_path).name,
                    "sha256": "",
                    "well_name": "",
                    "survey_name": "",
                    "well_type": "",
                    "wellhead_x_m": "",
                    "wellhead_y_m": "",
                    "datum_elevation_m": "",
                    "datum_reference": "",
                    "coordinate_reference_system": "",
                    "n_stations": "",
                    "md_min_m": "",
                    "md_max_m": "",
                    "max_inclination_deg": "",
                    "contract_status": "FAILED",
                    "n_errors": "",
                    "n_warnings": "",
                    "error_type": f.error_type,
                    "error_message": _sanitize_message(f.message, f.source_path),
                }
            )
    return rows


def build_trajectory_validation_rows(results: Dict[str, DeviationWellResult]) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(results):
        v = results[well_key].validation
        rows.append(
            {
                "well_key": well_key,
                "comparison_basis": v.comparison_basis,
                "tvd_max_abs_residual_m": v.tvd_max_abs_residual_m,
                "tvd_mean_residual_m": v.tvd_mean_residual_m,
                "tvd_rmse_m": v.tvd_rmse_m,
                "tvd_endpoint_residual_m": v.tvd_endpoint_residual_m,
                "tvd_tolerance_m": v.tvd_tolerance_m,
                "tvd_status": v.tvd_status,
                "easting_max_abs_residual_m": v.easting_max_abs_residual_m,
                "easting_mean_residual_m": v.easting_mean_residual_m,
                "easting_rmse_m": v.easting_rmse_m,
                "easting_endpoint_residual_m": v.easting_endpoint_residual_m,
                "easting_tolerance_m": v.easting_tolerance_m,
                "easting_status": v.easting_status,
                "northing_max_abs_residual_m": v.northing_max_abs_residual_m,
                "northing_mean_residual_m": v.northing_mean_residual_m,
                "northing_rmse_m": v.northing_rmse_m,
                "northing_endpoint_residual_m": v.northing_endpoint_residual_m,
                "northing_tolerance_m": v.northing_tolerance_m,
                "northing_status": v.northing_status,
                "x_consistency_max_abs_residual_m": v.x_consistency_max_abs_residual_m,
                "x_consistency_status": v.x_consistency_status,
                "y_consistency_max_abs_residual_m": v.y_consistency_max_abs_residual_m,
                "y_consistency_status": v.y_consistency_status,
                "z_consistency_max_abs_residual_m": v.z_consistency_max_abs_residual_m,
                "z_consistency_status": v.z_consistency_status,
                "overall_status": v.overall_status,
                "origin_initialization_note": v.origin_initialization_note,
            }
        )
    return rows


def build_depth_reference_register_rows(results: Dict[str, DeviationWellResult]) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(results):
        r = results[well_key]
        rows.append(
            {
                "well_key": well_key,
                "datum_elevation_m": r.header.datum_elevation_m,
                "datum_reference": r.header.datum_reference,
                "md_reference_convention": (
                    "MD referenced to zero at well datum (rotary table), positive downward"
                ),
                "tvd_reference_convention": (
                    "TVD referenced to zero at well datum (rotary table), positive downward"
                ),
                "tvdss_formula": "TVDSS_m = TVD_m - DatumElevation_m",
                "z_formula": "Z_m = DatumElevation_m - TVD_m  (equivalently, TVDSS_m = -Z_m)",
                "depth_basis_selected": r.depth_basis.selected_basis,
                "depth_basis_rationale": r.depth_basis.rationale,
                "trajectory_validation_overall_status": r.validation.overall_status,
            }
        )
    return rows


def build_las_depth_mapping_rows(mappings: Dict[str, LasDepthMappingResult]) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(mappings):
        m = mappings[well_key]
        rows.append(
            {
                "well_key": well_key,
                "depth_basis_used": m.depth_basis_used,
                "interpolation_method": m.interpolation_method,
                "n_samples": m.n_samples,
                "survey_md_min_m": m.survey_md_min_m,
                "survey_md_max_m": m.survey_md_max_m,
                "las_md_min_m": m.las_md_min_m,
                "las_md_max_m": m.las_md_max_m,
                "coverage_margin_lower_m": m.coverage_margin_lower_m,
                "coverage_margin_upper_m": m.coverage_margin_upper_m,
                "n_extrapolated": m.n_extrapolated,
                "tvd_at_final_las_md_m": float(m.tvd_mapped_m[-1]),
                "tvdss_at_final_las_md_m": float(m.tvdss_mapped_m[-1]),
                "datum_elevation_m": m.datum_elevation_m,
            }
        )
    return rows


def build_deviation_issues_rows(
    results: Dict[str, DeviationWellResult], failures: Dict[str, DeviationIngestionFailure]
) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(results):
        r = results[well_key]
        for issue in r.issues:
            rows.append(
                {
                    "well_key": well_key,
                    "source_filename": r.header.source_filename,
                    "severity": issue.severity,
                    "code": issue.code,
                    "message": issue.message,
                    "context": issue.context,
                }
            )
    for well_key in sorted(failures):
        f = failures[well_key]
        rows.append(
            {
                "well_key": well_key,
                "source_filename": Path(f.source_path).name,
                "severity": "ERROR",
                "code": f.error_type.upper(),
                "message": _sanitize_message(f.message, f.source_path),
                "context": Path(f.source_path).name,
            }
        )
    return rows


def build_deviation_depth_manifest(
    results: Dict[str, DeviationWellResult],
    failures: Dict[str, DeviationIngestionFailure],
    mappings: Dict[str, LasDepthMappingResult],
) -> dict:
    wells = {}
    for well_key in sorted(results):
        r = results[well_key]
        m = mappings.get(well_key)
        wells[well_key] = {
            "source_filename": r.header.source_filename,
            "sha256": r.header.sha256,
            "well_name": r.header.well_name,
            "well_type": r.header.well_type,
            "n_stations": int(r.raw.MD_source_m.size),
            "md_range_m": [float(r.raw.MD_source_m.min()), float(r.raw.MD_source_m.max())],
            "max_inclination_deg": float(r.raw.INCL_source_deg.max()),
            "contract_status": r.contract_status,
            "n_issue_errors": sum(1 for i in r.issues if i.severity == "ERROR"),
            "n_issue_warnings": sum(1 for i in r.issues if i.severity == "WARNING"),
            "trajectory_validation": {
                "overall_status": r.validation.overall_status,
                "tvd_max_abs_residual_m": r.validation.tvd_max_abs_residual_m,
                "tvd_status": r.validation.tvd_status,
                "easting_max_abs_residual_m": r.validation.easting_max_abs_residual_m,
                "easting_status": r.validation.easting_status,
                "northing_max_abs_residual_m": r.validation.northing_max_abs_residual_m,
                "northing_status": r.validation.northing_status,
            },
            "depth_basis_selected": r.depth_basis.selected_basis,
            "las_depth_mapping": (
                {
                    "n_samples": m.n_samples,
                    "las_md_range_m": [m.las_md_min_m, m.las_md_max_m],
                    "survey_md_range_m": [m.survey_md_min_m, m.survey_md_max_m],
                    "n_extrapolated": m.n_extrapolated,
                    "tvd_at_final_las_md_m": float(m.tvd_mapped_m[-1]),
                    "tvdss_at_final_las_md_m": float(m.tvdss_mapped_m[-1]),
                }
                if m is not None
                else None
            ),
        }

    failed = {
        well_key: {"error_type": f.error_type, "message": _sanitize_message(f.message, f.source_path)}
        for well_key, f in failures.items()
    }

    return {
        "increment": "3",
        "n_wells_loaded": len(results),
        "n_wells_failed": len(failures),
        "total_stations_all_wells": sum(int(r.raw.MD_source_m.size) for r in results.values()),
        "contracts_passed": sum(1 for r in results.values() if r.contract_status == "PASSED"),
        "trajectory_validation_pass_count": sum(
            1 for r in results.values() if r.validation.overall_status == "PASS"
        ),
        "trajectory_validation_warning_count": sum(
            1 for r in results.values() if r.validation.overall_status == "WARNING"
        ),
        "trajectory_validation_fail_count": sum(
            1 for r in results.values() if r.validation.overall_status == "FAIL"
        ),
        "wells": wells,
        "failed_wells": failed,
    }


#### Step 7 — Write the per-file deviation-survey contracts (`config/deviation_survey_contracts.yml`)

**Technical objective:** author a human-reviewable, per-file contract for each of the four wells, built by directly inspecting each file's own header text (well/survey identity, wellhead X/Y, datum and its MSL reference, coordinate-reference-system statement, column order) — never guessed, never copied from another well's contract.

**Validation logic:** `load_deviation_contract_config` rejects the contract file itself (before any deviation file is opened) for a duplicate top-level key, a missing required field, an invalid type, an unsupported azimuth-reference or depth-basis-policy value, or an internally inconsistent tolerance/threshold pair.

**Tolerance and depth-basis-policy design note:** `residual_tolerance_tvd_m` (0.01 m), `residual_tolerance_horizontal_m` (0.01 m), and `residual_fail_threshold_m` (5.0 m) are declared identically across all four wells - not tuned per well to force a pass/fail outcome. `depth_basis_policy: petrel_source_trace` is likewise declared uniformly for all four wells (see the Depth-Reference Convention section below for why).

In [ ]:
%%writefile config/deviation_survey_contracts.yml
# config/deviation_survey_contracts.yml
#
# Increment 3 human-authored, human-reviewable per-file deviation-survey
# contract for each of the four approved Petrel well-trace files
# (Poseidon 2, Boreas 1, Poseidon North 1, Proteus 1ST2). Built by directly
# inspecting each file's actual header block and station data - every
# `expected_*` value below was independently read from the file itself
# (never assumed, never copied from a prior increment's prose), and is
# re-verified against the file's ACTUAL parsed header/data by
# `p2mem.io.deviation.resolve_deviation_contract` at load time. A mismatch
# in any `expected_*` field is a blocking ERROR - ingestion does not
# silently proceed with "close enough" values.
#
# Increment 3.1 correction: the four top-level filename keys below are the
# EXACT, literal source filenames, which contain spaces (e.g.
# "Poseidon 2_dev.txt", not "Poseidon_2_dev.txt"). Increment 3 originally
# keyed this file with underscores substituted for spaces, which an audit
# found would fail to resolve against the actual files as they exist in
# Google Drive (contract lookup is an exact filename match - see
# `p2mem.io.deviation.load_deviation_surveys`). Internal WELL KEYS (used
# everywhere else in this project - notebook DEV_FILES dict values,
# output row well_key columns, Python dict keys) remain underscored
# (Poseidon_2, Boreas_1, Poseidon_North_1, Proteus_1ST2) for Python-
# identifier friendliness; only the FILENAME strings below carry the
# literal, space-containing names.
#
# Tolerance rationale (identical across all four wells, not tuned per
# well): `header_tolerance_m` (1 mm) allows for floating-point round-trip
# noise in a re-typed/re-exported header value without masking a
# genuinely different wellhead/datum. `residual_tolerance_tvd_m` and
# `residual_tolerance_horizontal_m` (1 cm each) are the PASS threshold for
# the independent minimum-curvature-vs-Petrel-source trajectory
# comparison - generous enough that ordinary floating-point/interpolation
# noise (observed at ~1 mm or better for three of the four wells) passes
# cleanly, but tight enough that a real discrepancy (observed at ~0.19 m
# TVD / ~1.6 m horizontal for Proteus 1ST2) is flagged. `residual_fail_
# threshold_m` (5 m) is an outer bound: a residual beyond this would
# suggest a parsing/contract error (wrong file, wrong wellhead, corrupted
# station data) rather than a legitimate trajectory-reconstruction
# discrepancy, and is reported as FAIL rather than WARNING. These three
# values are deliberately identical for every well in this file - they
# were chosen once, from first principles, and are not re-tuned per well
# to force a particular pass/fail outcome for any specific file.
#
# `depth_basis_policy: petrel_source_trace` is set uniformly for all four
# wells. Given the unresolved Proteus 1ST2 trajectory discrepancy (see the
# Increment 3 manifest), the conservative, auditable choice is to use the
# Petrel-supplied source TVD as every well's downstream MD-to-TVD/TVDSS
# mapping basis, while retaining the independently computed minimum-
# curvature trajectory as an always-reported QC comparison. This is not
# cherry-picked per well (e.g. using minimum-curvature for the three
# wells that agree closely and source-trace only for Proteus) - the same
# policy value is declared for every well, so the choice is visible and
# uniform rather than silently tuned to make the output look better.
files:
  "Poseidon 2_dev.txt":
    expected_sha256: "862e3d765edb8ecda6a4ea45ac1cdf82b1588fa2999c49368dd24fd1953fafdf"
    expected_well_identifier: "Poseidon 2"
    expected_survey_identifier: "Explicit survey 1"
    expected_coordinate_reference_system: 'GDA94 / MGA Zone 51 ("") [null,null]'
    expected_wellhead_x_m: 421661.80000001
    expected_wellhead_y_m: 8488800.70000000
    expected_datum_m: 21.79999924
    expected_datum_reference: "RT, Rotary table, from MSL"
    expected_column_count: 11
    expected_column_order: [MD, X, Y, Z, TVD, DX, DY, AZIM_TN, INCL, DLS, AZIM_GN]
    expected_units:
      MD: m
      X: m
      Y: m
      Z: m
      TVD: m
      DX: m
      DY: m
      AZIM_TN: deg
      INCL: deg
      DLS: deg_per_30m
      AZIM_GN: deg
    expected_station_count: 124
    expected_md_min_m: 0.0
    expected_md_max_m: 5356.0
    azimuth_reference_for_grid_coordinates: AZIM_GN
    source_depth_convention: "MD and TVD referenced to zero (=0) at well datum (rotary table), increasing downward"
    header_tolerance_m: 0.001
    residual_tolerance_tvd_m: 0.01
    residual_tolerance_horizontal_m: 0.01
    residual_fail_threshold_m: 5.0
    depth_basis_policy: petrel_source_trace
    notes: >
      GAS well type. Independent minimum-curvature reconstruction is
      expected to agree with the Petrel-supplied TVD/DX/DY to
      approximately millimetre scale (see Increment 3 manifest Section 5
      for the actual re-verified residuals). No known trajectory anomaly.

  "Boreas 1_dev.txt":
    expected_sha256: "8dec7c18b39d5aa9b4dfc608c8a165ce6e066a85112e38905eaeb241c1932c64"
    expected_well_identifier: "Boreas 1"
    expected_survey_identifier: "Explicit survey 1"
    expected_coordinate_reference_system: 'GDA94 / MGA Zone 51 ("") [null,null]'
    expected_wellhead_x_m: 424077.27700000
    expected_wellhead_y_m: 8490107.52400000
    expected_datum_m: 21.79999924
    expected_datum_reference: "RT, Rotary table, from MSL"
    expected_column_count: 11
    expected_column_order: [MD, X, Y, Z, TVD, DX, DY, AZIM_TN, INCL, DLS, AZIM_GN]
    expected_units:
      MD: m
      X: m
      Y: m
      Z: m
      TVD: m
      DX: m
      DY: m
      AZIM_TN: deg
      INCL: deg
      DLS: deg_per_30m
      AZIM_GN: deg
    expected_station_count: 134
    expected_md_min_m: 0.0
    expected_md_max_m: 5210.0
    azimuth_reference_for_grid_coordinates: AZIM_GN
    source_depth_convention: "MD and TVD referenced to zero (=0) at well datum (rotary table), increasing downward"
    header_tolerance_m: 0.001
    residual_tolerance_tvd_m: 0.01
    residual_tolerance_horizontal_m: 0.01
    residual_fail_threshold_m: 5.0
    depth_basis_policy: petrel_source_trace
    notes: >
      GAS well type. This well's build section reaches the dataset's
      second-highest maximum inclination (~9.56 deg). Independent
      minimum-curvature reconstruction is expected to agree with the
      Petrel-supplied TVD/DX/DY to approximately millimetre scale (see
      Increment 3 manifest Section 5). No known trajectory anomaly.

  "Poseidon North 1_dev.txt":
    expected_sha256: "f8bea90b59bcfe72fd5e6d22d5a27b6fc2fe622c6b2df877065c206bb04c84e1"
    expected_well_identifier: "Poseidon North 1"
    expected_survey_identifier: "Explicit survey 1"
    expected_coordinate_reference_system: 'GDA94 / MGA Zone 51 ("") [null,null]'
    expected_wellhead_x_m: 428611.91000000
    expected_wellhead_y_m: 8499338.27000000
    expected_datum_m: 22.00000000
    expected_datum_reference: "RT, Rotary table, from MSL"
    expected_column_count: 11
    expected_column_order: [MD, X, Y, Z, TVD, DX, DY, AZIM_TN, INCL, DLS, AZIM_GN]
    expected_units:
      MD: m
      X: m
      Y: m
      Z: m
      TVD: m
      DX: m
      DY: m
      AZIM_TN: deg
      INCL: deg
      DLS: deg_per_30m
      AZIM_GN: deg
    expected_station_count: 147
    expected_md_min_m: 0.0
    expected_md_max_m: 5287.5180664
    azimuth_reference_for_grid_coordinates: AZIM_GN
    source_depth_convention: "MD and TVD referenced to zero (=0) at well datum (rotary table), increasing downward"
    header_tolerance_m: 0.001
    residual_tolerance_tvd_m: 0.01
    residual_tolerance_horizontal_m: 0.01
    residual_fail_threshold_m: 5.0
    depth_basis_policy: petrel_source_trace
    notes: >
      WELL TYPE is declared UNDEFINED in this file's own header (not
      GAS, unlike the other three wells) - preserved and reported as-is,
      never silently assumed to be GAS. Its datum (22.0 m) differs
      slightly from the other three wells' shared datum (21.79999924 m);
      both are used exactly as declared, never rounded to match. No known
      trajectory anomaly; independent minimum-curvature reconstruction is
      expected to agree with the Petrel-supplied TVD/DX/DY to
      approximately millimetre scale (see Increment 3 manifest Section 5).

  "Proteus 1ST2_dev.txt":
    expected_sha256: "16c656b14067b8df1af084332547e1bc1f6f0316e0bb438c4b56e00f7ebcd23e"
    expected_well_identifier: "Proteus 1ST2"
    expected_survey_identifier: "Explicit survey 1"
    expected_coordinate_reference_system: 'GDA94 / MGA Zone 51 ("") [null,null]'
    expected_wellhead_x_m: 428152.83000000
    expected_wellhead_y_m: 8481148.02000000
    expected_datum_m: 21.79999924
    expected_datum_reference: "RT, Rotary table, from MSL"
    expected_column_count: 11
    expected_column_order: [MD, X, Y, Z, TVD, DX, DY, AZIM_TN, INCL, DLS, AZIM_GN]
    expected_units:
      MD: m
      X: m
      Y: m
      Z: m
      TVD: m
      DX: m
      DY: m
      AZIM_TN: deg
      INCL: deg
      DLS: deg_per_30m
      AZIM_GN: deg
    expected_station_count: 155
    expected_md_min_m: -0.000000763
    expected_md_max_m: 5249.7102107
    azimuth_reference_for_grid_coordinates: AZIM_GN
    source_depth_convention: "MD and TVD referenced to zero (=0) at well datum (rotary table), increasing downward"
    header_tolerance_m: 0.001
    residual_tolerance_tvd_m: 0.01
    residual_tolerance_horizontal_m: 0.01
    residual_fail_threshold_m: 5.0
    depth_basis_policy: petrel_source_trace
    notes: >
      KNOWN DATA-QUALITY FINDING (see Increment 3 manifest for full
      discussion - not silently resolved or hidden here): this file's
      first station carries a source MD/TVD of approximately -7.63e-7 m
      (preserved exactly, never rounded to 0.0 or treated as an error).
      Independent minimum-curvature reconstruction of this well's TVD/
      easting/northing disagrees with the Petrel-supplied source trace by
      substantially more than the other three wells (approximately 0.19 m
      TVD, 1.6 m easting, 0.9 m northing at maximum, versus millimetre-
      scale for the other three wells), even though this well's own
      supplied DLS column is internally consistent with a degrees-per-
      30-m recomputation from its own inclination/azimuth. This is
      reported as a visible QC WARNING (not FAILED ingestion, and not a
      "corrected" trajectory) - see `depth_basis_policy` above, which
      uses the Petrel source trace (not the disagreeing minimum-curvature
      trajectory) as this well's downstream depth-mapping basis.


#### Step 7b — Write the synthetic test fixtures and the new test suites

**Technical objective:** write eleven small synthetic Petrel-format deviation-survey fixtures (none require the real project files) and the four new test modules (`tests/test_trajectory.py`, `tests/test_deviation.py`, `tests/test_depth_mapping.py`, `tests/test_deviation_inventory.py`), verbatim from the tested files on disk. The locked `tests/test_units.py` and `tests/test_las.py` are NOT rewritten here. `tests/test_deviation_inventory.py` is new in Increment 3.1 — it guards the absolute-path-leakage correction (audit finding 2) with synthetic fixtures only.

In [ ]:
%%writefile tests/fixtures/dev_valid.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 600.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 20.0000000000 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_near_zero_negative_origin.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 -7.63e-07 100000.0000020000 199999.9999990000 10.0000007630 -0.0000007630 0.0000020000 -0.0000010000 0.0000000000 0.0000000000 0.0000000000 0.0000000000
 100 100000.0000020000 199999.9999990000 -90.0000000000 100.0000000000 0.0000020000 -0.0000010000 0.0000000000 0.0000000000 0.0000000000 0.0000000000
 300 100020.5504818924 200003.6236030716 -288.5410399214 298.5410399214 20.5504818924 3.6236030715 80.0000000000 12.0000000000 1.8000000000 80.0000000000
 600 100081.9763954102 200014.4546489394 -581.9853201416 591.9853201416 81.9763954102 14.4546489394 80.0000000000 12.0000000000 0.0000000000 80.0000000000


In [ ]:
%%writefile tests/fixtures/dev_missing_column.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z            TVD            DX            DY            AZIM_TN            INCL            AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 600.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 20.0000000000 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_duplicate_column.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        MD
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 600.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 20.0000000000 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_malformed_numeric_row.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 N/A 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 600.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 20.0000000000 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_row_width_mismatch.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000
 600.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 20.0000000000 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_nonfinite_token.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 600.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 nan 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_duplicate_md.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 300.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 20.0000000000 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_non_monotonic_md.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 250.0 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 20.0000000000 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_invalid_inclination.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN DEGREES
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 600.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 200.0 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/fixtures/dev_unsupported_angle_convention.txt
# WELL TRACE FROM PETREL 
# WELL NAME:              Test Well 1
# DEFINITIVE SURVEY:      Test survey 1
# WELL HEAD X-COORDINATE: 100000.00000000 (m)
# WELL HEAD Y-COORDINATE: 200000.00000000 (m)
# WELL DATUM (RT, Rotary table, from MSL): 10.00000000 (m)
# WELL TYPE:              GAS
# MD AND TVD ARE REFERENCED (=0) AT WELL DATUM AND INCREASE DOWNWARDS
# ANGLES ARE GIVEN IN RADIANS
# XYZ TRACE IS GIVEN IN COORDINATE SYSTEM GDA94 / MGA Zone 51 ("") [null,null]
# AZIM_TN: azimuth in True North 
# AZIM_GN: azimuth in Grid North 
# DX DY ARE GIVEN IN GRID NORTH IN m-UNITS
# DEPTH (Z, tvd_z) GIVEN IN m-UNITS
#===============================================================================================================================================
      MD            X            Y            Z           TVD           DX          DY        AZIM_TN        INCL         DLS        AZIM_GN
#===============================================================================================================================================
 0.0000000000 100000.00000 200000.00000 10.000000 0.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 100.0000000000 100000.00000 200000.00000 -90.000000 100.000000 0.0000000000 0.0000000000 5.0000000000 0.0000000000 0.0000000000 0.0000000000
 300.0000000000 100012.31005 200012.31005 -288.986154 298.986154 12.3100450580 12.3100450580 51.0000000000 10.0000000000 1.5000000000 45.0000000000
 600.0000000000 100068.39164 200065.85310 -578.398005 588.398005 68.3916448333 65.8531039780 53.5000000000 20.0000000000 1.0011930499 47.0000000000
 900.0000000000 100154.53148 200141.64036 -855.483094 865.483094 154.5314765140 141.6403560923 56.2000000000 25.0000000000 0.5128576192 50.0000000000
 1200.0000000000 100251.65479 200223.13649 -1127.375431 1137.375431 251.6547878042 223.1364907746 56.2000000000 25.0000000000 0.0000000000 50.0000000000


In [ ]:
%%writefile tests/test_trajectory.py
"""
tests/test_trajectory.py - Validation suite for p2mem.trajectory
(Increment 3: minimum-curvature trajectory computation).

These are PORTABLE, synthetic-data unit tests: no real project deviation
file is required. Real four-well integration (which DOES require the
private/raw project deviation files) is a separate notebook/script run.
"""

import numpy as np
import pytest

from p2mem.trajectory import (
    MinimumCurvatureResult,
    TrajectoryComputationError,
    compute_minimum_curvature_trajectory,
    dogleg_angle_rad,
    minimum_curvature_intervals,
    ratio_factor,
)
from p2mem.units import degrees_to_radians


# ---------------------------------------------------------------------------
# Ratio-factor limit / numerical stability
# ---------------------------------------------------------------------------
def test_ratio_factor_exact_zero_dogleg_returns_one():
    rf = ratio_factor(np.array([0.0]))
    assert rf[0] == pytest.approx(1.0, abs=0.0)


def test_ratio_factor_taylor_and_direct_branches_agree_at_threshold():
    # Just below and just above the small-dogleg threshold should agree
    # to high precision (continuity of the piecewise definition).
    from p2mem.trajectory import _SMALL_DOGLEG_THRESHOLD_RAD as THRESH

    below = ratio_factor(np.array([THRESH * 0.5]))[0]
    above = ratio_factor(np.array([THRESH * 2.0]))[0]
    assert below == pytest.approx(1.0, abs=1e-12)
    assert above == pytest.approx(1.0, abs=1e-12)
    assert abs(below - above) < 1e-12


def test_ratio_factor_extremely_small_dogleg_is_numerically_stable():
    beta = np.array([1e-12, 1e-10, 1e-9, 1e-8])
    rf = ratio_factor(beta)
    assert np.all(np.isfinite(rf))
    assert np.allclose(rf, 1.0, atol=1e-10)


def test_ratio_factor_known_value_at_moderate_dogleg():
    # RF at beta = 1 radian: (2/1)*tan(0.5) computed independently via math.
    import math

    beta = np.array([1.0])
    expected = (2.0 / 1.0) * math.tan(0.5)
    assert ratio_factor(beta)[0] == pytest.approx(expected, rel=1e-12)


# ---------------------------------------------------------------------------
# Dogleg angle: analytical / independent cross-checks
# ---------------------------------------------------------------------------
def test_dogleg_angle_zero_when_stations_identical_direction():
    beta = dogleg_angle_rad(
        np.array([30.0]), np.array([30.0]), np.array([45.0]), np.array([45.0])
    )
    assert beta[0] == pytest.approx(0.0, abs=1e-12)


def test_dogleg_angle_identical_nonvertical_direction_is_exact_zero_machine_precision():
    # Increment 3.1 correction: the arctan2(|u1 x u2|, u1 . u2) formulation
    # must report EXACTLY 0.0 (bit-for-bit, not merely "very small") for
    # two stations with identical, nonvertical inclination and azimuth,
    # across a range of angle values - not just the one value spot-checked
    # above. The previous arccos(cos_beta) formulation could not meet this
    # bar (it left a ~1e-6-degree noise floor even for identical stations).
    for incl_deg, azim_deg in [
        (5.0, 0.0), (30.0, 45.0), (60.0, 123.4), (89.9, 270.0), (10.0, 359.999),
    ]:
        beta = dogleg_angle_rad(
            np.array([incl_deg]), np.array([incl_deg]), np.array([azim_deg]), np.array([azim_deg])
        )
        assert beta[0] == 0.0, f"expected exact 0.0 for incl={incl_deg}, azim={azim_deg}, got {beta[0]!r}"


def test_dogleg_angle_preserves_genuinely_tiny_nonzero_dogleg():
    # A genuinely tiny (but real) inclination change must NOT be erased by
    # the numerically stable formulation - it must be resolved accurately,
    # not rounded down to 0.0 the way a naive "if very small, call it zero"
    # implementation might.
    tiny_deg = 1.0e-6
    beta = dogleg_angle_rad(
        np.array([30.0]), np.array([30.0 + tiny_deg]), np.array([45.0]), np.array([45.0])
    )
    beta_deg = np.rad2deg(beta[0])
    assert beta_deg > 0.0
    assert beta_deg == pytest.approx(tiny_deg, rel=1e-6)


def test_dogleg_angle_from_vertical_equals_inclination_independent_of_azimuth():
    # I1 = 0 (vertical): sin(I1) = 0, so the azimuth cross-term vanishes
    # regardless of A1 or A2 - dogleg from vertical to (I2, A2) must equal
    # I2 exactly, for ANY A1/A2 pair.
    for a1, a2 in [(0.0, 0.0), (0.0, 90.0), (123.4, 987.6), (0.0, 359.999)]:
        beta = dogleg_angle_rad(
            np.array([0.0]), np.array([17.5]), np.array([a1]), np.array([a2])
        )
        assert beta[0] == pytest.approx(np.deg2rad(17.5), abs=1e-10)


def test_azimuth_undefined_at_zero_inclination_does_not_inflate_dogleg():
    # A vertical station (I=0) carrying an arbitrary placeholder azimuth
    # must not itself generate a spurious "azimuth discontinuity" dogleg.
    # Two consecutive vertical stations with wildly different recorded
    # azimuths must show dogleg == 0 (both inclinations are 0).
    beta = dogleg_angle_rad(
        np.array([0.0]), np.array([0.0]), np.array([12.3]), np.array([321.9])
    )
    assert beta[0] == pytest.approx(0.0, abs=1e-12)


def test_dogleg_angle_matches_independent_direction_cosine_dot_product():
    # Independent cross-check: the tangent unit vector at a station is
    # (sin I cos A, sin I sin A, cos I) in (N, E, Down); the dogleg angle
    # between two stations is the angle between their tangent vectors,
    # i.e. arccos(t1 . t2). This is mathematically the same quantity as
    # the trig identity used in dogleg_angle_rad, computed via a
    # completely independent expression (a 3-vector dot product) - a
    # coding bug in the trig implementation (index swap, sign error,
    # degree/radian mixup) would not, in general, also satisfy this
    # independent check.
    rng = np.random.default_rng(42)
    incl1 = rng.uniform(0, 60, size=25)
    incl2 = rng.uniform(0, 60, size=25)
    azim1 = rng.uniform(0, 360, size=25)
    azim2 = rng.uniform(0, 360, size=25)

    beta = dogleg_angle_rad(incl1, incl2, azim1, azim2)

    i1r = np.deg2rad(incl1)
    i2r = np.deg2rad(incl2)
    a1r = np.deg2rad(azim1)
    a2r = np.deg2rad(azim2)
    t1 = np.stack([np.sin(i1r) * np.cos(a1r), np.sin(i1r) * np.sin(a1r), np.cos(i1r)], axis=-1)
    t2 = np.stack([np.sin(i2r) * np.cos(a2r), np.sin(i2r) * np.sin(a2r), np.cos(i2r)], axis=-1)
    dot = np.clip(np.sum(t1 * t2, axis=-1), -1.0, 1.0)
    beta_independent = np.arccos(dot)

    assert np.allclose(beta, beta_independent, atol=1e-10)


def test_dogleg_angle_azimuth_wraparound_gives_small_dogleg():
    # 359 deg -> 1 deg is a true azimuth change of only 2 degrees, not 358.
    beta_wrap = dogleg_angle_rad(
        np.array([30.0]), np.array([30.0]), np.array([359.0]), np.array([1.0])
    )
    beta_direct_2deg = dogleg_angle_rad(
        np.array([30.0]), np.array([30.0]), np.array([0.0]), np.array([2.0])
    )
    assert beta_wrap[0] == pytest.approx(beta_direct_2deg[0], abs=1e-12)
    assert np.rad2deg(beta_wrap[0]) < 5.0  # small, not ~358 deg


def test_ratio_factor_stable_for_dogleg_computed_near_vertical_wraparound_case():
    # End-to-end check that a tiny dogleg produced by the (Increment 3.1)
    # numerically stable dogleg computation still feeds cleanly into
    # ratio_factor's small-angle Taylor branch, for a near-vertical survey
    # with an azimuth wraparound (azimuth is poorly defined near vertical,
    # so this exercises both edge cases at once).
    beta = dogleg_angle_rad(
        np.array([0.05]), np.array([0.05]), np.array([359.5]), np.array([0.5])
    )
    rf = ratio_factor(beta)
    assert np.all(np.isfinite(rf))
    assert rf[0] == pytest.approx(1.0, abs=1e-9)


# ---------------------------------------------------------------------------
# Minimum-curvature displacement: closed-form / independent checks
# ---------------------------------------------------------------------------
def test_fully_vertical_well_tvd_equals_md_zero_horizontal_offset():
    md = np.array([0.0, 500.0, 1000.0, 1500.0])
    incl = np.zeros_like(md)
    azim = np.zeros_like(md)  # undefined but harmless at incl == 0
    result = compute_minimum_curvature_trajectory(
        md, incl, azim, tvd_origin_m=0.0, northing_origin_m=0.0, easting_origin_m=0.0
    )
    assert np.allclose(result.tvd_mc_m, md, atol=1e-9)
    assert np.allclose(result.northing_offset_mc_m, 0.0, atol=1e-9)
    assert np.allclose(result.easting_offset_mc_m, 0.0, atol=1e-9)
    assert np.allclose(result.dogleg_deg, 0.0, atol=1e-9)


def test_constant_inclination_straight_trajectory_matches_closed_form():
    # Straight tangent hold section (I, A constant): dogleg == 0 exactly,
    # RF == 1 exactly, so minimum curvature reduces to the simple
    # closed-form tangential displacement:
    #   dTVD = dMD cos(I); dN = dMD sin(I) cos(A); dE = dMD sin(I) sin(A)
    md = np.array([0.0, 100.0, 250.0, 400.0])
    incl = np.full_like(md, 40.0)
    azim = np.full_like(md, 70.0)
    result = compute_minimum_curvature_trajectory(
        md, incl, azim, tvd_origin_m=0.0, northing_origin_m=0.0, easting_origin_m=0.0
    )
    i = np.deg2rad(40.0)
    a = np.deg2rad(70.0)
    expected_tvd = md * np.cos(i)
    expected_n = md * np.sin(i) * np.cos(a)
    expected_e = md * np.sin(i) * np.sin(a)
    assert np.allclose(result.tvd_mc_m, expected_tvd, atol=1e-9)
    assert np.allclose(result.northing_offset_mc_m, expected_n, atol=1e-9)
    assert np.allclose(result.easting_offset_mc_m, expected_e, atol=1e-9)
    # Increment 3.1: dogleg_deg is now computed via the numerically stable
    # arctan2(|u1 x u2|, u1 . u2) formulation (see p2mem.trajectory module
    # docstring, "Numerical stability of the dogleg angle"), which reports
    # EXACTLY 0.0 for identical (I, A) stations - no residual noise floor
    # remains (the previous arccos(cos_beta) formulation left a ~1e-6-deg
    # spurious noise floor here, tolerated at the time via atol=1e-5; that
    # tolerance is tightened back down now that the root cause is fixed).
    assert np.array_equal(result.dogleg_deg, np.zeros_like(result.dogleg_deg))
    assert np.array_equal(result.dls_deg_per_30m, np.zeros_like(result.dls_deg_per_30m))


def test_correct_north_east_sign_convention_by_azimuth_quadrant():
    # Azimuth 0 (north): pure +N, ~0 E. Azimuth 90 (east): pure +E, ~0 N.
    # Azimuth 180 (south): pure -N. Azimuth 270 (west): pure -E.
    md = np.array([0.0, 100.0])
    incl = np.array([0.0, 30.0])
    for azim_val, expect_n_sign, expect_e_sign in [
        (0.0, +1, 0),
        (90.0, 0, +1),
        (180.0, -1, 0),
        (270.0, 0, -1),
    ]:
        azim = np.array([azim_val, azim_val])
        result = compute_minimum_curvature_trajectory(
            md, incl, azim, tvd_origin_m=0.0, northing_origin_m=0.0, easting_origin_m=0.0
        )
        n_final, e_final = result.northing_offset_mc_m[-1], result.easting_offset_mc_m[-1]
        if expect_n_sign > 0:
            assert n_final > 1.0
        elif expect_n_sign < 0:
            assert n_final < -1.0
        else:
            assert abs(n_final) < 1e-6
        if expect_e_sign > 0:
            assert e_final > 1.0
        elif expect_e_sign < 0:
            assert e_final < -1.0
        else:
            assert abs(e_final) < 1e-6


def test_dls_degrees_per_30m_known_value():
    # Two stations 30 m apart with a dogleg of exactly 3 degrees (achieved
    # via a pure inclination build at constant azimuth, so beta == the
    # inclination change) should report DLS == 3.0 deg/30m exactly.
    md = np.array([1000.0, 1030.0])
    incl = np.array([10.0, 13.0])
    azim = np.array([0.0, 0.0])
    beta_deg, rf, d_tvd, d_n, d_e = minimum_curvature_intervals(md, incl, azim)
    assert beta_deg[0] == pytest.approx(3.0, abs=1e-9)
    dls = (beta_deg[0] / (md[1] - md[0])) * 30.0
    assert dls == pytest.approx(3.0, abs=1e-9)


def test_dls_scales_inversely_with_interval_length_for_fixed_dogleg():
    # Same 3-degree dogleg over 15 m instead of 30 m -> DLS doubles.
    md = np.array([1000.0, 1015.0])
    incl = np.array([10.0, 13.0])
    azim = np.array([0.0, 0.0])
    beta_deg, *_ = minimum_curvature_intervals(md, incl, azim)
    dls = (beta_deg[0] / (md[1] - md[0])) * 30.0
    assert dls == pytest.approx(6.0, abs=1e-9)


# ---------------------------------------------------------------------------
# Origin initialization
# ---------------------------------------------------------------------------
def test_origin_values_are_used_as_station_zero_not_hard_coded_zero():
    # Proteus-style tiny nonzero first-station origin must be preserved
    # exactly as the trajectory's starting condition.
    md = np.array([-7.63e-7, 500.0])
    incl = np.array([0.0, 5.0])
    azim = np.array([0.0, 45.0])
    result = compute_minimum_curvature_trajectory(
        md,
        incl,
        azim,
        tvd_origin_m=-7.63e-7,
        northing_origin_m=-1e-6,
        easting_origin_m=2e-6,
    )
    assert result.tvd_mc_m[0] == -7.63e-7
    assert result.northing_offset_mc_m[0] == -1e-6
    assert result.easting_offset_mc_m[0] == 2e-6


def test_scalar_result_shapes_and_types_are_ndarrays():
    md = np.array([0.0, 100.0, 200.0])
    incl = np.array([0.0, 10.0, 20.0])
    azim = np.array([0.0, 30.0, 30.0])
    result = compute_minimum_curvature_trajectory(
        md, incl, azim, tvd_origin_m=0.0, northing_origin_m=0.0, easting_origin_m=0.0
    )
    assert isinstance(result, MinimumCurvatureResult)
    for arr in (
        result.dogleg_deg,
        result.dls_deg_per_30m,
        result.tvd_mc_m,
        result.northing_offset_mc_m,
        result.easting_offset_mc_m,
    ):
        assert isinstance(arr, np.ndarray)
        assert arr.shape == (3,)
    assert result.dogleg_deg[0] == 0.0
    assert result.dls_deg_per_30m[0] == 0.0


# ---------------------------------------------------------------------------
# Rejection of nonphysical / structurally invalid input
# ---------------------------------------------------------------------------
def test_rejects_inclination_above_180():
    with pytest.raises(TrajectoryComputationError, match="physically valid range"):
        compute_minimum_curvature_trajectory(
            np.array([0.0, 100.0]),
            np.array([0.0, 181.0]),
            np.array([0.0, 0.0]),
            tvd_origin_m=0.0,
            northing_origin_m=0.0,
            easting_origin_m=0.0,
        )


def test_rejects_negative_inclination():
    with pytest.raises(TrajectoryComputationError, match="physically valid range"):
        compute_minimum_curvature_trajectory(
            np.array([0.0, 100.0]),
            np.array([0.0, -5.0]),
            np.array([0.0, 0.0]),
            tvd_origin_m=0.0,
            northing_origin_m=0.0,
            easting_origin_m=0.0,
        )


def test_rejects_non_increasing_md():
    with pytest.raises(TrajectoryComputationError, match="strictly increasing"):
        compute_minimum_curvature_trajectory(
            np.array([0.0, 100.0, 100.0]),
            np.array([0.0, 5.0, 6.0]),
            np.array([0.0, 10.0, 10.0]),
            tvd_origin_m=0.0,
            northing_origin_m=0.0,
            easting_origin_m=0.0,
        )


def test_rejects_decreasing_md():
    with pytest.raises(TrajectoryComputationError, match="strictly increasing"):
        compute_minimum_curvature_trajectory(
            np.array([0.0, 200.0, 100.0]),
            np.array([0.0, 5.0, 6.0]),
            np.array([0.0, 10.0, 10.0]),
            tvd_origin_m=0.0,
            northing_origin_m=0.0,
            easting_origin_m=0.0,
        )


def test_rejects_nan_policy_non_finite_md():
    with pytest.raises(TrajectoryComputationError, match="non-finite"):
        compute_minimum_curvature_trajectory(
            np.array([0.0, np.nan, 200.0]),
            np.array([0.0, 5.0, 6.0]),
            np.array([0.0, 10.0, 10.0]),
            tvd_origin_m=0.0,
            northing_origin_m=0.0,
            easting_origin_m=0.0,
        )


def test_rejects_infinite_azimuth():
    with pytest.raises(TrajectoryComputationError, match="non-finite"):
        compute_minimum_curvature_trajectory(
            np.array([0.0, 100.0]),
            np.array([0.0, 5.0]),
            np.array([0.0, np.inf]),
            tvd_origin_m=0.0,
            northing_origin_m=0.0,
            easting_origin_m=0.0,
        )


def test_rejects_mismatched_array_shapes():
    with pytest.raises(TrajectoryComputationError, match="shape"):
        compute_minimum_curvature_trajectory(
            np.array([0.0, 100.0, 200.0]),
            np.array([0.0, 5.0]),
            np.array([0.0, 10.0, 10.0]),
            tvd_origin_m=0.0,
            northing_origin_m=0.0,
            easting_origin_m=0.0,
        )


def test_single_station_trajectory_returns_origin_only():
    md = np.array([500.0])
    incl = np.array([12.0])
    azim = np.array([45.0])
    result = compute_minimum_curvature_trajectory(
        md, incl, azim, tvd_origin_m=500.0, northing_origin_m=1.0, easting_origin_m=2.0
    )
    assert result.tvd_mc_m[0] == 500.0
    assert result.northing_offset_mc_m[0] == 1.0
    assert result.easting_offset_mc_m[0] == 2.0
    assert result.dogleg_deg[0] == 0.0


In [ ]:
%%writefile tests/test_deviation.py
"""
tests/test_deviation.py - Validation suite for p2mem.io.deviation
(Increment 3: Petrel deviation-survey ingestion and per-file contract
resolution).

These are PORTABLE unit tests: they use only small synthetic fixtures
under tests/fixtures/ (dev_*.txt) and never require the four real project
deviation files. Real four-well integration (which DOES require the
private/raw project deviation files) is a separate notebook/script run.
"""

import hashlib
import math
from pathlib import Path

import numpy as np
import pytest
import yaml

from p2mem.deviation_models import (
    DEPTH_BASIS_MINIMUM_CURVATURE,
    DEPTH_BASIS_PETREL_SOURCE,
    STATUS_PASS,
    STATUS_WARNING,
    DeviationFileContract,
)
from p2mem.io.deviation import (
    DLS_NORMALIZATION_INFERRED_CODE,
    DeviationContractDefinitionError,
    DeviationContractError,
    DeviationFileNotFoundError,
    DeviationParsingError,
    load_deviation_contract_config,
    load_deviation_file,
    load_deviation_surveys,
    parse_deviation_header,
    read_deviation_stations,
    resolve_deviation_contract,
)

FIXTURES = Path(__file__).parent / "fixtures"


def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _base_contract(filename: str, **overrides) -> DeviationFileContract:
    fixture_path = FIXTURES / filename
    default_sha256 = _sha256(fixture_path) if fixture_path.exists() else "0" * 64
    defaults = dict(
        source_filename=filename,
        expected_sha256=default_sha256,
        expected_well_identifier="Test Well 1",
        expected_survey_identifier="Test survey 1",
        expected_coordinate_reference_system='GDA94 / MGA Zone 51 ("") [null,null]',
        expected_wellhead_x_m=100000.0,
        expected_wellhead_y_m=200000.0,
        expected_datum_m=10.0,
        expected_datum_reference="RT, Rotary table, from MSL",
        expected_column_count=11,
        expected_column_order=("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
        expected_units={
            "MD": "m", "X": "m", "Y": "m", "Z": "m", "TVD": "m", "DX": "m", "DY": "m",
            "AZIM_TN": "deg", "INCL": "deg", "DLS": "deg_per_30m", "AZIM_GN": "deg",
        },
        expected_station_count=6,
        expected_md_min_m=0.0,
        expected_md_max_m=1200.0,
        azimuth_reference_for_grid_coordinates="AZIM_GN",
        source_depth_convention="MD and TVD referenced to zero at well datum, increasing downward",
        header_tolerance_m=0.001,
        residual_tolerance_tvd_m=0.01,
        residual_tolerance_horizontal_m=0.01,
        residual_fail_threshold_m=5.0,
        depth_basis_policy=DEPTH_BASIS_PETREL_SOURCE,
        notes="Synthetic fixture for Increment 3 unit tests.",
    )
    defaults.update(overrides)
    return DeviationFileContract(**defaults)


def _near_zero_contract(**overrides) -> DeviationFileContract:
    return _base_contract(
        "dev_near_zero_negative_origin.txt",
        expected_station_count=4,
        expected_md_min_m=-7.63e-7,
        expected_md_max_m=600.0,
        **overrides,
    )


# ---------------------------------------------------------------------------
# Header / structural parsing
# ---------------------------------------------------------------------------
def test_valid_file_parses_full_header():
    header, data_start, cols = parse_deviation_header(str(FIXTURES / "dev_valid.txt"))
    assert header.well_name == "Test Well 1"
    assert header.survey_name == "Test survey 1"
    assert header.wellhead_x_m == pytest.approx(100000.0)
    assert header.wellhead_y_m == pytest.approx(200000.0)
    assert header.datum_elevation_m == pytest.approx(10.0)
    assert header.well_type == "GAS"
    assert header.coordinate_reference_system == 'GDA94 / MGA Zone 51 ("") [null,null]'
    assert cols == ("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN")
    assert header.sha256 == _sha256(FIXTURES / "dev_valid.txt")


def test_exact_column_order_preserved():
    _, _, cols = parse_deviation_header(str(FIXTURES / "dev_valid.txt"))
    assert cols == ("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN")


def test_missing_column_rejected():
    with pytest.raises(DeviationParsingError, match="missing required column"):
        parse_deviation_header(str(FIXTURES / "dev_missing_column.txt"))


def test_duplicate_column_rejected():
    with pytest.raises(DeviationParsingError, match="duplicate column name"):
        parse_deviation_header(str(FIXTURES / "dev_duplicate_column.txt"))


def test_malformed_numeric_row_rejected():
    header, data_start, cols = parse_deviation_header(str(FIXTURES / "dev_malformed_numeric_row.txt"))
    with pytest.raises(DeviationParsingError, match="non-numeric token"):
        read_deviation_stations(str(FIXTURES / "dev_malformed_numeric_row.txt"), data_start, cols)


def test_row_width_mismatch_rejected():
    header, data_start, cols = parse_deviation_header(str(FIXTURES / "dev_row_width_mismatch.txt"))
    with pytest.raises(DeviationParsingError, match="row-width mismatch"):
        read_deviation_stations(str(FIXTURES / "dev_row_width_mismatch.txt"), data_start, cols)


def test_nonfinite_literal_token_rejected():
    header, data_start, cols = parse_deviation_header(str(FIXTURES / "dev_nonfinite_token.txt"))
    with pytest.raises(DeviationParsingError, match="non-finite literal token"):
        read_deviation_stations(str(FIXTURES / "dev_nonfinite_token.txt"), data_start, cols)


def test_file_not_found_raises_typed_error():
    with pytest.raises(DeviationFileNotFoundError):
        parse_deviation_header(str(FIXTURES / "does_not_exist_dev.txt"))


def test_unsupported_angle_convention_rejected():
    with pytest.raises(DeviationParsingError, match="unsupported angle-unit convention"):
        parse_deviation_header(str(FIXTURES / "dev_unsupported_angle_convention.txt"))


def test_proteus_style_near_zero_negative_md_is_preserved_not_rejected():
    path = str(FIXTURES / "dev_near_zero_negative_origin.txt")
    header, data_start, cols = parse_deviation_header(path)
    stations = read_deviation_stations(path, data_start, cols)
    assert stations.MD_source_m[0] == pytest.approx(-7.63e-7, abs=1e-12)
    assert stations.MD_source_m[0] < 0.0


# ---------------------------------------------------------------------------
# Contract resolution
# ---------------------------------------------------------------------------
def _load_valid(contract=None):
    contract = contract or _base_contract("dev_valid.txt")
    return load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)


def test_valid_file_and_contract_resolves_cleanly():
    result = _load_valid()
    assert result.contract_status == "PASSED"
    error_issues = [i for i in result.issues if i.severity == "ERROR"]
    assert error_issues == []
    assert result.raw.MD_source_m.size == 6


def test_wrong_filename_is_blocking():
    contract = _base_contract("dev_valid.txt", source_filename="some_other_file.txt")
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "FILENAME_MISMATCH" for i in excinfo.value.issues)


def test_wrong_sha256_is_blocking():
    contract = _base_contract("dev_valid.txt", expected_sha256="0" * 64)
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "SHA256_MISMATCH" for i in excinfo.value.issues)


def test_wrong_well_identifier_is_blocking():
    contract = _base_contract("dev_valid.txt", expected_well_identifier="Wrong Well")
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "WELL_IDENTIFIER_MISMATCH" for i in excinfo.value.issues)


def test_wrong_datum_is_blocking():
    contract = _base_contract("dev_valid.txt", expected_datum_m=999.0)
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "DATUM_MISMATCH" for i in excinfo.value.issues)


def test_wrong_wellhead_coordinates_is_blocking():
    contract = _base_contract("dev_valid.txt", expected_wellhead_x_m=1.0)
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "WELLHEAD_X_MISMATCH" for i in excinfo.value.issues)


def test_wrong_crs_is_blocking():
    contract = _base_contract("dev_valid.txt", expected_coordinate_reference_system="WGS84")
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "CRS_MISMATCH" for i in excinfo.value.issues)


def test_wrong_station_count_is_blocking():
    contract = _base_contract("dev_valid.txt", expected_station_count=999)
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "STATION_COUNT_MISMATCH" for i in excinfo.value.issues)


def test_wrong_md_coverage_is_blocking():
    contract = _base_contract("dev_valid.txt", expected_md_max_m=1.0)
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "MD_MAX_MISMATCH" for i in excinfo.value.issues)


def test_duplicate_md_is_blocking():
    contract = _base_contract("dev_duplicate_md.txt")
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_duplicate_md.txt"), contract)
    assert any(i.code == "DUPLICATE_MD" for i in excinfo.value.issues)


def test_non_monotonic_md_is_blocking():
    contract = _base_contract("dev_non_monotonic_md.txt")
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_non_monotonic_md.txt"), contract)
    assert any(i.code == "NON_MONOTONIC_MD" for i in excinfo.value.issues)


def test_invalid_inclination_is_blocking():
    contract = _base_contract("dev_invalid_inclination.txt")
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_invalid_inclination.txt"), contract)
    assert any(i.code == "INVALID_INCLINATION" for i in excinfo.value.issues)


def test_md_unit_not_declared_warning_always_present():
    result = _load_valid()
    codes = [i.code for i in result.issues]
    assert "MD_UNIT_NOT_EXPLICITLY_DECLARED" in codes
    warning = next(i for i in result.issues if i.code == "MD_UNIT_NOT_EXPLICITLY_DECLARED")
    assert warning.severity == "WARNING"


# ---------------------------------------------------------------------------
# Increment 3.1: DLS-normalization-inferred disclosure
# ---------------------------------------------------------------------------
def test_dls_normalization_inferred_warning_always_present_on_success():
    result = _load_valid()
    codes = [i.code for i in result.issues]
    assert DLS_NORMALIZATION_INFERRED_CODE in codes
    warning = next(i for i in result.issues if i.code == DLS_NORMALIZATION_INFERRED_CODE)
    assert warning.severity == "WARNING"
    # The message must actually state a computed discrepancy figure (not a
    # boilerplate "trust me" claim) and must state the raw column is
    # untouched.
    assert "deg/30m" in warning.message
    assert "never altered" in warning.message
    assert warning.context == result.header.source_filename


def test_dls_normalization_check_reports_actual_computed_discrepancy_not_hardcoded():
    # dev_valid.txt was built using the project's own trusted minimum-
    # curvature function, so its own supplied DLS column should already be
    # self-consistent with an independent recomputation to within a tiny,
    # genuinely computed (not assumed) discrepancy.
    result = _load_valid()
    warning = next(i for i in result.issues if i.code == DLS_NORMALIZATION_INFERRED_CODE)
    # Independently recompute the same comparison from the typed result's
    # own public fields, and confirm the message's reported figure is
    # consistent with it (not a different, hard-coded number).
    independent_max_diff = float(
        np.max(np.abs(result.raw.DLS_source_deg_per_30m - result.mc.dls_deg_per_30m))
    )
    assert f"{independent_max_diff:.6e}" in warning.message
    # Raw source DLS values are untouched by the check.
    assert result.raw.DLS_source_deg_per_30m is not result.mc.dls_deg_per_30m


def test_dls_normalization_warning_present_even_when_trajectory_validation_warns():
    # The DLS-normalization disclosure is about the DLS column's own
    # normalization convention, not about trajectory agreement - it must
    # still be present (and still WARNING, not escalated) even for a well
    # whose independent trajectory reconstruction disagrees with the
    # source trajectory (mirroring the real Proteus 1ST2 finding).
    contract = _base_contract(
        "dev_valid.txt",
        residual_tolerance_tvd_m=1e-12,
        residual_tolerance_horizontal_m=1e-12,
    )
    result = load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert result.validation.overall_status == STATUS_WARNING
    codes = [i.code for i in result.issues]
    assert DLS_NORMALIZATION_INFERRED_CODE in codes
    warning = next(i for i in result.issues if i.code == DLS_NORMALIZATION_INFERRED_CODE)
    assert warning.severity == "WARNING"


def test_column_order_mismatch_is_blocking():
    # Same physical columns, declared in a different required order.
    contract = _base_contract(
        "dev_valid.txt",
        expected_column_order=("X", "MD", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
    )
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(FIXTURES / "dev_valid.txt"), contract)
    assert any(i.code == "COLUMN_ORDER_MISMATCH" for i in excinfo.value.issues)


# ---------------------------------------------------------------------------
# Increment 3.1: filenames containing spaces (the real Petrel deviation
# files' actual names, e.g. "Poseidon 2_dev.txt") must resolve correctly,
# and a file renamed to substitute underscores for spaces must NOT
# silently match a contract declared for the space-containing name - this
# is the exact defect the Increment 3.1 corrective patch fixed (Increment
# 3 had silently renamed these to underscore variants internally).
# ---------------------------------------------------------------------------
REAL_DEVIATION_FILENAMES_WITH_SPACES = (
    "Poseidon 2_dev.txt",
    "Boreas 1_dev.txt",
    "Poseidon North 1_dev.txt",
    "Proteus 1ST2_dev.txt",
)


@pytest.mark.parametrize("spaced_name", REAL_DEVIATION_FILENAMES_WITH_SPACES)
def test_real_filenames_with_spaces_resolve_successfully(tmp_path, spaced_name):
    spaced_path = tmp_path / spaced_name
    spaced_path.write_bytes((FIXTURES / "dev_valid.txt").read_bytes())
    contract = _base_contract(spaced_name, expected_sha256=_sha256(spaced_path))
    result = load_deviation_file(str(spaced_path), contract)
    assert result.contract_status == "PASSED"
    assert result.header.source_filename == spaced_name
    assert " " in result.header.source_filename


@pytest.mark.parametrize("spaced_name", REAL_DEVIATION_FILENAMES_WITH_SPACES)
def test_underscore_renamed_filename_does_not_silently_match_contract(tmp_path, spaced_name):
    # Same byte content, but the file on disk has been renamed to
    # substitute underscores for spaces (exactly the Increment 3 defect).
    # A contract declared for the space-containing name must reject this
    # as a FILENAME_MISMATCH, never silently accept it as a match.
    underscored_name = spaced_name.replace(" ", "_")
    underscored_path = tmp_path / underscored_name
    underscored_path.write_bytes((FIXTURES / "dev_valid.txt").read_bytes())
    contract = _base_contract(spaced_name, expected_sha256=_sha256(underscored_path))
    with pytest.raises(DeviationContractError) as excinfo:
        load_deviation_file(str(underscored_path), contract)
    assert any(i.code == "FILENAME_MISMATCH" for i in excinfo.value.issues)


def test_proteus_style_near_zero_origin_contract_resolves_cleanly():
    result = load_deviation_file(
        str(FIXTURES / "dev_near_zero_negative_origin.txt"), _near_zero_contract()
    )
    assert result.contract_status == "PASSED"
    assert result.raw.MD_source_m[0] == pytest.approx(-7.63e-7, abs=1e-12)
    # Preserved exactly as the trajectory's own tie-on origin, not zeroed.
    assert result.mc.tvd_mc_m[0] == pytest.approx(-7.63e-7, abs=1e-12)


# ---------------------------------------------------------------------------
# Deviation-contract YAML authoring validation
# (load_deviation_contract_config)
# ---------------------------------------------------------------------------
def _write_yaml(tmp_path, text: str) -> str:
    p = tmp_path / "contracts.yml"
    p.write_text(text)
    return str(p)


def _minimal_yaml_entry(sha: str) -> str:
    return f"""
files:
  dev_valid.txt:
    expected_sha256: "{sha}"
    expected_well_identifier: "Test Well 1"
    expected_survey_identifier: "Test survey 1"
    expected_coordinate_reference_system: 'GDA94 / MGA Zone 51 ("") [null,null]'
    expected_wellhead_x_m: 100000.0
    expected_wellhead_y_m: 200000.0
    expected_datum_m: 10.0
    expected_datum_reference: "RT, Rotary table, from MSL"
    expected_column_count: 11
    expected_column_order: [MD, X, Y, Z, TVD, DX, DY, AZIM_TN, INCL, DLS, AZIM_GN]
    expected_units:
      MD: m
      X: m
      Y: m
      Z: m
      TVD: m
      DX: m
      DY: m
      AZIM_TN: deg
      INCL: deg
      DLS: deg_per_30m
      AZIM_GN: deg
    expected_station_count: 6
    expected_md_min_m: 0.0
    expected_md_max_m: 1200.0
    azimuth_reference_for_grid_coordinates: AZIM_GN
    source_depth_convention: "test"
    header_tolerance_m: 0.001
    residual_tolerance_tvd_m: 0.01
    residual_tolerance_horizontal_m: 0.01
    residual_fail_threshold_m: 5.0
    depth_basis_policy: petrel_source_trace
    notes: "test"
"""


def test_valid_yaml_contract_loads(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    path = _write_yaml(tmp_path, _minimal_yaml_entry(sha))
    contracts = load_deviation_contract_config(path)
    assert "dev_valid.txt" in contracts
    assert contracts["dev_valid.txt"].expected_station_count == 6


def test_duplicate_contract_key_rejected(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    text = _minimal_yaml_entry(sha) + _minimal_yaml_entry(sha).replace(
        "dev_valid.txt:", "dev_valid.txt:  # duplicate\n"
    )
    # Force an actual duplicate top-level key by concatenating two "files:" blocks
    # is invalid YAML shape; instead duplicate the key WITHIN one files: mapping.
    dup_text = """
files:
  dev_valid.txt:
    expected_sha256: "%s"
    expected_well_identifier: "A"
    expected_survey_identifier: "S"
    expected_coordinate_reference_system: "CRS"
    expected_wellhead_x_m: 1.0
    expected_wellhead_y_m: 1.0
    expected_datum_m: 1.0
    expected_datum_reference: "RT"
    expected_column_count: 11
    expected_column_order: [MD, X, Y, Z, TVD, DX, DY, AZIM_TN, INCL, DLS, AZIM_GN]
    expected_units: {MD: m, X: m, Y: m, Z: m, TVD: m, DX: m, DY: m, AZIM_TN: deg, INCL: deg, DLS: deg_per_30m, AZIM_GN: deg}
    expected_station_count: 6
    expected_md_min_m: 0.0
    expected_md_max_m: 1.0
    azimuth_reference_for_grid_coordinates: AZIM_GN
    source_depth_convention: "test"
    header_tolerance_m: 0.001
    residual_tolerance_tvd_m: 0.01
    residual_tolerance_horizontal_m: 0.01
    residual_fail_threshold_m: 5.0
    depth_basis_policy: petrel_source_trace
    notes: "test"
  dev_valid.txt:
    expected_sha256: "%s"
    expected_well_identifier: "B"
    expected_survey_identifier: "S"
    expected_coordinate_reference_system: "CRS"
    expected_wellhead_x_m: 1.0
    expected_wellhead_y_m: 1.0
    expected_datum_m: 1.0
    expected_datum_reference: "RT"
    expected_column_count: 11
    expected_column_order: [MD, X, Y, Z, TVD, DX, DY, AZIM_TN, INCL, DLS, AZIM_GN]
    expected_units: {MD: m, X: m, Y: m, Z: m, TVD: m, DX: m, DY: m, AZIM_TN: deg, INCL: deg, DLS: deg_per_30m, AZIM_GN: deg}
    expected_station_count: 6
    expected_md_min_m: 0.0
    expected_md_max_m: 1.0
    azimuth_reference_for_grid_coordinates: AZIM_GN
    source_depth_convention: "test"
    header_tolerance_m: 0.001
    residual_tolerance_tvd_m: 0.01
    residual_tolerance_horizontal_m: 0.01
    residual_fail_threshold_m: 5.0
    depth_basis_policy: petrel_source_trace
    notes: "test"
""" % (sha, sha)
    path = _write_yaml(tmp_path, dup_text)
    with pytest.raises(DeviationContractDefinitionError, match="Duplicate key"):
        load_deviation_contract_config(path)


def test_invalid_type_rejected(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    text = _minimal_yaml_entry(sha).replace("expected_column_count: 11", 'expected_column_count: "eleven"')
    path = _write_yaml(tmp_path, text)
    with pytest.raises(DeviationContractDefinitionError, match="expected_column_count"):
        load_deviation_contract_config(path)


def test_unsupported_azimuth_reference_rejected(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    text = _minimal_yaml_entry(sha).replace(
        "azimuth_reference_for_grid_coordinates: AZIM_GN",
        "azimuth_reference_for_grid_coordinates: AZIM_MAGNETIC",
    )
    path = _write_yaml(tmp_path, text)
    with pytest.raises(DeviationContractDefinitionError, match="azimuth_reference_for_grid_coordinates"):
        load_deviation_contract_config(path)


def test_unsupported_depth_basis_policy_rejected(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    text = _minimal_yaml_entry(sha).replace(
        "depth_basis_policy: petrel_source_trace", "depth_basis_policy: made_up_policy"
    )
    path = _write_yaml(tmp_path, text)
    with pytest.raises(DeviationContractDefinitionError, match="depth_basis_policy"):
        load_deviation_contract_config(path)


def test_invalid_tolerance_rejected(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    text = _minimal_yaml_entry(sha).replace("residual_tolerance_tvd_m: 0.01", "residual_tolerance_tvd_m: -0.01")
    path = _write_yaml(tmp_path, text)
    with pytest.raises(DeviationContractDefinitionError, match="non-negative"):
        load_deviation_contract_config(path)


def test_fail_threshold_not_exceeding_tolerance_rejected(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    text = _minimal_yaml_entry(sha).replace("residual_fail_threshold_m: 5.0", "residual_fail_threshold_m: 0.005")
    path = _write_yaml(tmp_path, text)
    with pytest.raises(DeviationContractDefinitionError, match="must exceed"):
        load_deviation_contract_config(path)


def test_incompatible_column_count_rejected(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    text = _minimal_yaml_entry(sha).replace("expected_column_count: 11", "expected_column_count: 10")
    path = _write_yaml(tmp_path, text)
    with pytest.raises(DeviationContractDefinitionError, match="does not match"):
        load_deviation_contract_config(path)


def test_missing_required_field_rejected(tmp_path):
    sha = _sha256(FIXTURES / "dev_valid.txt")
    text = _minimal_yaml_entry(sha).replace('    expected_datum_reference: "RT, Rotary table, from MSL"\n', "")
    path = _write_yaml(tmp_path, text)
    with pytest.raises(DeviationContractDefinitionError, match="missing required field"):
        load_deviation_contract_config(path)


# ---------------------------------------------------------------------------
# Batch loading / error isolation
# ---------------------------------------------------------------------------
def test_batch_isolates_one_failed_well_from_successful_wells():
    contracts = {
        "dev_valid.txt": _base_contract("dev_valid.txt"),
        "dev_duplicate_md.txt": _base_contract("dev_duplicate_md.txt"),
    }
    file_paths = {
        "GOOD_WELL": str(FIXTURES / "dev_valid.txt"),
        "BAD_WELL": str(FIXTURES / "dev_duplicate_md.txt"),
    }
    results, failures = load_deviation_surveys(file_paths, contracts)
    assert "GOOD_WELL" in results
    assert "BAD_WELL" in failures
    assert failures["BAD_WELL"].error_type == "contract_failure"


def test_batch_isolates_file_not_found():
    contracts = {"does_not_exist_dev.txt": _base_contract("does_not_exist_dev.txt")}
    file_paths = {"MISSING": str(FIXTURES / "does_not_exist_dev.txt")}
    results, failures = load_deviation_surveys(file_paths, contracts)
    assert "MISSING" in failures
    assert failures["MISSING"].error_type == "file_not_found"


def test_batch_deterministic_row_ordering():
    contracts = {
        "dev_valid.txt": _base_contract("dev_valid.txt"),
    }
    file_paths = {"WELL_A": str(FIXTURES / "dev_valid.txt")}
    results1, _ = load_deviation_surveys(file_paths, contracts)
    results2, _ = load_deviation_surveys(file_paths, contracts)
    assert list(results1["WELL_A"].raw.MD_source_m) == list(results2["WELL_A"].raw.MD_source_m)


def test_batch_unexpected_programming_error_propagates(monkeypatch):
    import p2mem.io.deviation as devmod

    def _boom(*args, **kwargs):
        raise KeyError("simulated programming error")

    monkeypatch.setattr(devmod, "parse_deviation_header", _boom)
    contracts = {"dev_valid.txt": _base_contract("dev_valid.txt")}
    file_paths = {"WELL_A": str(FIXTURES / "dev_valid.txt")}
    with pytest.raises(KeyError):
        devmod.load_deviation_surveys(file_paths, contracts)


# ---------------------------------------------------------------------------
# Trajectory validation status integration (PASS vs WARNING)
# ---------------------------------------------------------------------------
def test_trajectory_validation_status_pass_for_self_consistent_fixture():
    result = _load_valid()
    assert result.validation.overall_status == STATUS_PASS
    assert result.validation.tvd_status == STATUS_PASS


def test_source_and_computed_trajectories_never_overwritten():
    result = _load_valid()
    # Source arrays are untouched by the MC computation.
    assert result.raw.TVD_source_m is not result.mc.tvd_mc_m
    assert not np.array_equal(result.raw.TVD_source_m, np.zeros_like(result.raw.TVD_source_m))
    # Both remain independently accessible on the same result object.
    assert result.raw.TVD_source_m.shape == result.mc.tvd_mc_m.shape


def test_azim_tn_never_used_for_grid_coordinate_residuals():
    # dev_valid.txt was generated with AZIM_TN deliberately offset from
    # AZIM_GN. If the implementation mistakenly used AZIM_TN for the
    # grid-coordinate (northing/easting) computation, residuals against
    # the (AZIM_GN-derived) source DX/DY would be large, not near-zero.
    result = _load_valid()
    assert result.validation.easting_max_abs_residual_m < 1e-6
    assert result.validation.northing_max_abs_residual_m < 1e-6


In [ ]:
%%writefile tests/test_depth_mapping.py
"""
tests/test_depth_mapping.py - Validation suite for p2mem.depth_mapping
(Increment 3: MD-to-TVD/TVDSS interpolation and coverage checks).

Portable, synthetic-data unit tests only - no real project file required.
"""

import numpy as np
import pytest

from p2mem.deviation_models import (
    DEPTH_BASIS_MINIMUM_CURVATURE,
    DEPTH_BASIS_PETREL_SOURCE,
    DepthBasisSelection,
    DeviationFileContract,
    DeviationHeaderInfo,
    DeviationStationData,
    DeviationWellResult,
    TrajectoryValidationResult,
)
from p2mem.depth_mapping import (
    DepthMappingError,
    ExtrapolationRejectedError,
    INTERPOLATION_METHOD,
    map_las_md_to_tvd_tvdss,
    select_survey_trajectory_for_mapping,
)
from p2mem.trajectory import compute_minimum_curvature_trajectory


def _make_well_result(
    md, incl, azim_gn, datum_elevation_m=20.0, depth_basis=DEPTH_BASIS_PETREL_SOURCE
) -> DeviationWellResult:
    md = np.asarray(md, dtype=np.float64)
    incl = np.asarray(incl, dtype=np.float64)
    azim_gn = np.asarray(azim_gn, dtype=np.float64)

    mc = compute_minimum_curvature_trajectory(
        md, incl, azim_gn, tvd_origin_m=float(md[0]), northing_origin_m=0.0, easting_origin_m=0.0
    )
    # Use the MC trajectory itself as the "source" TVD too, for a
    # self-consistent synthetic fixture (depth-mapping tests do not need
    # to exercise the trajectory-residual comparison - that's covered in
    # test_deviation.py).
    stations = DeviationStationData(
        MD_source_m=md,
        X_source_m=np.zeros_like(md),
        Y_source_m=np.zeros_like(md),
        Z_source_m=datum_elevation_m - mc.tvd_mc_m,
        TVD_source_m=mc.tvd_mc_m.copy(),
        DX_source_m=mc.easting_offset_mc_m.copy(),
        DY_source_m=mc.northing_offset_mc_m.copy(),
        AZIM_TN_source_deg=azim_gn.copy(),
        INCL_source_deg=incl,
        DLS_source_deg_per_30m=mc.dls_deg_per_30m,
        AZIM_GN_source_deg=azim_gn,
    )
    header = DeviationHeaderInfo(
        source_path="synthetic",
        source_filename="synthetic_dev.txt",
        sha256="0" * 64,
        well_name="Synthetic Well",
        survey_name="Synthetic survey",
        wellhead_x_m=0.0,
        wellhead_y_m=0.0,
        datum_elevation_m=datum_elevation_m,
        datum_reference="RT, Rotary table, from MSL",
        well_type="GAS",
        coordinate_reference_system="TEST",
        depth_reference_statement="test",
        angle_unit_statement="DEGREES",
        dx_dy_statement="m-UNITS",
        z_statement="m-UNITS",
        column_names=("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
        header_line_count=16,
        data_line_offset=17,
    )
    contract = DeviationFileContract(
        source_filename="synthetic_dev.txt",
        expected_sha256="0" * 64,
        expected_well_identifier="Synthetic Well",
        expected_survey_identifier="Synthetic survey",
        expected_coordinate_reference_system="TEST",
        expected_wellhead_x_m=0.0,
        expected_wellhead_y_m=0.0,
        expected_datum_m=datum_elevation_m,
        expected_datum_reference="RT, Rotary table, from MSL",
        expected_column_count=11,
        expected_column_order=("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
        expected_units={},
        expected_station_count=int(md.size),
        expected_md_min_m=float(md[0]),
        expected_md_max_m=float(md[-1]),
        azimuth_reference_for_grid_coordinates="AZIM_GN",
        source_depth_convention="test",
        header_tolerance_m=0.001,
        residual_tolerance_tvd_m=0.01,
        residual_tolerance_horizontal_m=0.01,
        residual_fail_threshold_m=5.0,
        depth_basis_policy=depth_basis,
        notes="synthetic",
    )
    validation = TrajectoryValidationResult(
        well_key="Synthetic Well",
        comparison_basis="synthetic self-consistent fixture",
        tvd_max_abs_residual_m=0.0, tvd_mean_residual_m=0.0, tvd_rmse_m=0.0,
        tvd_endpoint_residual_m=0.0, tvd_tolerance_m=0.01, tvd_status="PASS",
        easting_max_abs_residual_m=0.0, easting_mean_residual_m=0.0, easting_rmse_m=0.0,
        easting_endpoint_residual_m=0.0, easting_tolerance_m=0.01, easting_status="PASS",
        northing_max_abs_residual_m=0.0, northing_mean_residual_m=0.0, northing_rmse_m=0.0,
        northing_endpoint_residual_m=0.0, northing_tolerance_m=0.01, northing_status="PASS",
        x_consistency_max_abs_residual_m=0.0, x_consistency_status="PASS",
        y_consistency_max_abs_residual_m=0.0, y_consistency_status="PASS",
        z_consistency_max_abs_residual_m=0.0, z_consistency_status="PASS",
        overall_status="PASS",
        origin_initialization_note="synthetic",
    )
    depth_basis_sel = DepthBasisSelection(
        well_key="Synthetic Well", selected_basis=depth_basis, rationale="test"
    )
    return DeviationWellResult(
        header=header, contract=contract, raw=stations, mc=mc,
        validation=validation, depth_basis=depth_basis_sel,
    )


# ---------------------------------------------------------------------------
# Basis selection
# ---------------------------------------------------------------------------
def test_select_petrel_source_basis_returns_source_arrays():
    well = _make_well_result([0.0, 100.0, 500.0], [0.0, 5.0, 10.0], [0.0, 30.0, 30.0])
    md, tvd, basis = select_survey_trajectory_for_mapping(well)
    assert basis == DEPTH_BASIS_PETREL_SOURCE
    assert np.array_equal(md, well.raw.MD_source_m)
    assert np.array_equal(tvd, well.raw.TVD_source_m)


def test_select_minimum_curvature_basis_returns_mc_arrays():
    well = _make_well_result(
        [0.0, 100.0, 500.0], [0.0, 5.0, 10.0], [0.0, 30.0, 30.0],
        depth_basis=DEPTH_BASIS_MINIMUM_CURVATURE,
    )
    md, tvd, basis = select_survey_trajectory_for_mapping(well)
    assert basis == DEPTH_BASIS_MINIMUM_CURVATURE
    assert np.array_equal(tvd, well.mc.tvd_mc_m)


# ---------------------------------------------------------------------------
# Mapping correctness
# ---------------------------------------------------------------------------
def test_exact_preservation_at_survey_stations():
    well = _make_well_result([0.0, 200.0, 400.0, 600.0], [0.0, 10.0, 15.0, 15.0], [0.0, 45.0, 45.0, 45.0])
    las_md = well.raw.MD_source_m.copy()  # sample exactly at the survey stations
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert np.allclose(mapping.tvd_mapped_m, well.raw.TVD_source_m, atol=1e-9)


def test_deterministic_interpolation_repeated_calls_agree():
    well = _make_well_result([0.0, 200.0, 400.0, 600.0], [0.0, 10.0, 15.0, 15.0], [0.0, 45.0, 45.0, 45.0])
    las_md = np.linspace(10.0, 590.0, 50)
    m1 = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    m2 = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert np.array_equal(m1.tvd_mapped_m, m2.tvd_mapped_m)


def test_monotonic_mapped_tvd_for_monotonic_survey():
    well = _make_well_result([0.0, 200.0, 400.0, 600.0], [0.0, 10.0, 15.0, 15.0], [0.0, 45.0, 45.0, 45.0])
    las_md = np.linspace(1.0, 599.0, 100)
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert np.all(np.diff(mapping.tvd_mapped_m) >= 0.0)


def test_exact_sample_count_preservation():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    las_md = np.linspace(5.0, 395.0, 137)
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert mapping.n_samples == 137
    assert mapping.tvd_mapped_m.shape == (137,)
    assert mapping.tvdss_mapped_m.shape == (137,)


def test_selected_depth_basis_recorded_in_result():
    well = _make_well_result(
        [0.0, 200.0], [0.0, 10.0], [0.0, 45.0], depth_basis=DEPTH_BASIS_MINIMUM_CURVATURE
    )
    mapping = map_las_md_to_tvd_tvdss("WELL_A", np.array([100.0]), well)
    assert mapping.depth_basis_used == DEPTH_BASIS_MINIMUM_CURVATURE
    assert mapping.interpolation_method == INTERPOLATION_METHOD


def test_las_md_source_never_modified():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    las_md = np.array([50.0, 150.0, 350.0])
    original = las_md.copy()
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert np.array_equal(las_md, original)
    assert np.array_equal(mapping.las_md_source_m, original)


# ---------------------------------------------------------------------------
# Extrapolation rejection
# ---------------------------------------------------------------------------
def test_upper_bound_extrapolation_rejected():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    las_md = np.array([100.0, 450.0])  # 450 > survey max of 400
    with pytest.raises(ExtrapolationRejectedError, match="above-coverage margin"):
        map_las_md_to_tvd_tvdss("WELL_A", las_md, well)


def test_lower_bound_extrapolation_rejected():
    well = _make_well_result([50.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    las_md = np.array([10.0, 100.0])  # 10 < survey min of 50
    with pytest.raises(ExtrapolationRejectedError, match="below-coverage margin"):
        map_las_md_to_tvd_tvdss("WELL_A", las_md, well)


def test_no_extrapolation_when_fully_inside_coverage():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    las_md = np.array([50.0, 150.0, 350.0])
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert mapping.n_extrapolated == 0
    assert mapping.coverage_margin_lower_m >= 0.0
    assert mapping.coverage_margin_upper_m >= 0.0


def test_exact_boundary_md_is_not_extrapolation():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    las_md = np.array([0.0, 400.0])  # exactly at survey bounds
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert mapping.n_extrapolated == 0


# ---------------------------------------------------------------------------
# Depth-reference sign convention (TVDSS)
# ---------------------------------------------------------------------------
def test_tvdss_sign_convention_tvd_minus_datum():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0], datum_elevation_m=25.0)
    las_md = np.array([100.0, 300.0])
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert np.allclose(mapping.tvdss_mapped_m, mapping.tvd_mapped_m - 25.0, atol=1e-9)


def test_tvdss_equals_negative_z_for_source_basis():
    # Z_m = DatumElevation_m - TVD_m  =>  TVDSS_m = TVD_m - DatumElevation_m = -Z_m
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0], datum_elevation_m=18.5)
    las_md = well.raw.MD_source_m.copy()
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    z_at_stations = well.raw.Z_source_m
    assert np.allclose(mapping.tvdss_mapped_m, -z_at_stations, atol=1e-9)


def test_positive_and_negative_tvdss_near_wellhead():
    # A shallow LAS sample above the datum-referenced sea-level crossing
    # point yields negative TVDSS (above MSL); a deep sample yields
    # positive TVDSS (below MSL).
    datum = 20.0  # 20 m above MSL
    well = _make_well_result([0.0, 15.0, 100.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], datum_elevation_m=datum)
    las_md = np.array([5.0, 50.0])  # TVD == MD here (vertical well)
    mapping = map_las_md_to_tvd_tvdss("WELL_A", las_md, well)
    assert mapping.tvdss_mapped_m[0] < 0.0  # 5 m TVD - 20 m datum = -15 m (above MSL)
    assert mapping.tvdss_mapped_m[1] > 0.0  # 50 m TVD - 20 m datum = +30 m (below MSL)


# ---------------------------------------------------------------------------
# Structural failure handling
# ---------------------------------------------------------------------------
def test_non_increasing_survey_md_raises_depth_mapping_error():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    # Corrupt the survey MD post-construction to simulate a non-increasing basis.
    import dataclasses

    bad_stations = dataclasses.replace(well.raw, MD_source_m=np.array([0.0, 400.0, 200.0]))
    bad_well = dataclasses.replace(well, raw=bad_stations)
    with pytest.raises(DepthMappingError, match="strictly increasing"):
        map_las_md_to_tvd_tvdss("WELL_A", np.array([100.0]), bad_well)


def test_empty_las_md_rejected():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    with pytest.raises(DepthMappingError, match="non-empty"):
        map_las_md_to_tvd_tvdss("WELL_A", np.array([]), well)


def test_non_finite_las_md_rejected():
    well = _make_well_result([0.0, 200.0, 400.0], [0.0, 10.0, 10.0], [0.0, 45.0, 45.0])
    with pytest.raises(DepthMappingError, match="non-finite"):
        map_las_md_to_tvd_tvdss("WELL_A", np.array([100.0, np.nan]), well)


In [ ]:
%%writefile tests/test_deviation_inventory.py
"""
tests/test_deviation_inventory.py - Validation suite for
p2mem.io.deviation_inventory (Increment 3.1 addition).

Portable, synthetic-data unit tests only - no real project file required.

These tests exist specifically to guard the Increment 3.1 corrective-patch
fix: no row built by this module (for a successful well OR a failed one)
may embed a full, environment-dependent filesystem path (a Colab Drive
mount path, a local development build path, a CI temp directory) - only
a file's basename may appear, in `source_filename` and `context` fields,
and any embedded exception message must have its own copy of the path
reduced to a basename too.
"""

import dataclasses

import numpy as np

from p2mem.deviation_models import (
    DepthBasisSelection,
    DeviationFileContract,
    DeviationHeaderInfo,
    DeviationIngestionFailure,
    DeviationIngestionIssue,
    DeviationStationData,
    DeviationWellResult,
    TrajectoryValidationResult,
)
from p2mem.io.deviation_inventory import (
    build_deviation_depth_manifest,
    build_deviation_file_inventory_rows,
    build_deviation_issues_rows,
    build_depth_reference_register_rows,
    build_trajectory_validation_rows,
)
from p2mem.trajectory import compute_minimum_curvature_trajectory

# A deliberately absolute, environment-looking path - mirrors what a real
# build (or a real Colab Drive mount) would embed if not sanitized.
_FAKE_ABSOLUTE_DIR = "/home/fake_build_env/p2mem_build/data/raw/deviation"
_FAKE_BASENAME = "Poseidon 2_dev.txt"
_FAKE_ABSOLUTE_PATH = f"{_FAKE_ABSOLUTE_DIR}/{_FAKE_BASENAME}"


def _make_well_result(source_path: str, source_filename: str) -> DeviationWellResult:
    md = np.array([0.0, 100.0, 200.0])
    incl = np.array([0.0, 5.0, 10.0])
    azim = np.array([0.0, 30.0, 30.0])
    mc = compute_minimum_curvature_trajectory(
        md, incl, azim, tvd_origin_m=0.0, northing_origin_m=0.0, easting_origin_m=0.0
    )
    stations = DeviationStationData(
        MD_source_m=md,
        X_source_m=np.zeros_like(md),
        Y_source_m=np.zeros_like(md),
        Z_source_m=20.0 - mc.tvd_mc_m,
        TVD_source_m=mc.tvd_mc_m.copy(),
        DX_source_m=mc.easting_offset_mc_m.copy(),
        DY_source_m=mc.northing_offset_mc_m.copy(),
        AZIM_TN_source_deg=azim.copy(),
        INCL_source_deg=incl,
        DLS_source_deg_per_30m=mc.dls_deg_per_30m,
        AZIM_GN_source_deg=azim,
    )
    header = DeviationHeaderInfo(
        source_path=source_path,
        source_filename=source_filename,
        sha256="0" * 64,
        well_name="Poseidon 2",
        survey_name="Explicit survey 1",
        wellhead_x_m=0.0,
        wellhead_y_m=0.0,
        datum_elevation_m=20.0,
        datum_reference="RT, Rotary table, from MSL",
        well_type="GAS",
        coordinate_reference_system="TEST",
        depth_reference_statement="test",
        angle_unit_statement="DEGREES",
        dx_dy_statement="m-UNITS",
        z_statement="m-UNITS",
        column_names=("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
        header_line_count=16,
        data_line_offset=17,
    )
    contract = DeviationFileContract(
        source_filename=source_filename,
        expected_sha256="0" * 64,
        expected_well_identifier="Poseidon 2",
        expected_survey_identifier="Explicit survey 1",
        expected_coordinate_reference_system="TEST",
        expected_wellhead_x_m=0.0,
        expected_wellhead_y_m=0.0,
        expected_datum_m=20.0,
        expected_datum_reference="RT, Rotary table, from MSL",
        expected_column_count=11,
        expected_column_order=("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
        expected_units={},
        expected_station_count=int(md.size),
        expected_md_min_m=float(md[0]),
        expected_md_max_m=float(md[-1]),
        azimuth_reference_for_grid_coordinates="AZIM_GN",
        source_depth_convention="test",
        header_tolerance_m=0.001,
        residual_tolerance_tvd_m=0.01,
        residual_tolerance_horizontal_m=0.01,
        residual_fail_threshold_m=5.0,
        depth_basis_policy="petrel_source_trace",
        notes="synthetic",
    )
    validation = TrajectoryValidationResult(
        well_key="Poseidon 2",
        comparison_basis="synthetic self-consistent fixture",
        tvd_max_abs_residual_m=0.0, tvd_mean_residual_m=0.0, tvd_rmse_m=0.0,
        tvd_endpoint_residual_m=0.0, tvd_tolerance_m=0.01, tvd_status="PASS",
        easting_max_abs_residual_m=0.0, easting_mean_residual_m=0.0, easting_rmse_m=0.0,
        easting_endpoint_residual_m=0.0, easting_tolerance_m=0.01, easting_status="PASS",
        northing_max_abs_residual_m=0.0, northing_mean_residual_m=0.0, northing_rmse_m=0.0,
        northing_endpoint_residual_m=0.0, northing_tolerance_m=0.01, northing_status="PASS",
        x_consistency_max_abs_residual_m=0.0, x_consistency_status="PASS",
        y_consistency_max_abs_residual_m=0.0, y_consistency_status="PASS",
        z_consistency_max_abs_residual_m=0.0, z_consistency_status="PASS",
        overall_status="PASS",
        origin_initialization_note="synthetic",
    )
    depth_basis_sel = DepthBasisSelection(
        well_key="Poseidon 2", selected_basis="petrel_source_trace", rationale="test"
    )
    issues = (
        DeviationIngestionIssue("WARNING", "MD_UNIT_NOT_EXPLICITLY_DECLARED", "test warning", source_filename),
    )
    return DeviationWellResult(
        header=header, contract=contract, raw=stations, mc=mc,
        validation=validation, depth_basis=depth_basis_sel, issues=issues,
    )


def _flatten_values(obj):
    """Recursively yield every string value found in a dict/list/scalar tree."""
    if isinstance(obj, dict):
        for v in obj.values():
            yield from _flatten_values(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            yield from _flatten_values(v)
    elif isinstance(obj, str):
        yield obj


# ---------------------------------------------------------------------------
# Successful wells: source_filename / context must be basenames only
# ---------------------------------------------------------------------------
def test_successful_well_file_inventory_row_has_no_absolute_path():
    well = _make_well_result(_FAKE_ABSOLUTE_PATH, _FAKE_BASENAME)
    rows = build_deviation_file_inventory_rows({"Poseidon_2": well}, {})
    assert len(rows) == 1
    assert rows[0]["source_filename"] == _FAKE_BASENAME
    for value in _flatten_values(rows):
        assert _FAKE_ABSOLUTE_DIR not in value
        assert "/home" not in value


def test_successful_well_issues_row_context_has_no_absolute_path():
    well = _make_well_result(_FAKE_ABSOLUTE_PATH, _FAKE_BASENAME)
    rows = build_deviation_issues_rows({"Poseidon_2": well}, {})
    assert len(rows) == 1
    assert rows[0]["context"] == _FAKE_BASENAME
    for value in _flatten_values(rows):
        assert _FAKE_ABSOLUTE_DIR not in value


def test_manifest_and_register_rows_have_no_absolute_path():
    well = _make_well_result(_FAKE_ABSOLUTE_PATH, _FAKE_BASENAME)
    manifest = build_deviation_depth_manifest({"Poseidon_2": well}, {}, {})
    register_rows = build_depth_reference_register_rows({"Poseidon_2": well})
    validation_rows = build_trajectory_validation_rows({"Poseidon_2": well})
    for value in _flatten_values(manifest):
        assert _FAKE_ABSOLUTE_DIR not in value
    for value in _flatten_values(register_rows):
        assert _FAKE_ABSOLUTE_DIR not in value
    for value in _flatten_values(validation_rows):
        assert _FAKE_ABSOLUTE_DIR not in value


# ---------------------------------------------------------------------------
# Failed wells: source_filename / context / error_message must all be
# sanitized to a basename, even though the underlying exception message
# embeds the full path.
# ---------------------------------------------------------------------------
def _make_failure() -> DeviationIngestionFailure:
    return DeviationIngestionFailure(
        well_key="Poseidon_2",
        source_path=_FAKE_ABSOLUTE_PATH,
        error_type="file_not_found",
        message=f"Deviation-survey file not found: {_FAKE_ABSOLUTE_PATH}",
        exception=FileNotFoundError(_FAKE_ABSOLUTE_PATH),
    )


def test_failed_well_file_inventory_row_sanitizes_path_and_message():
    failure = _make_failure()
    rows = build_deviation_file_inventory_rows({}, {"Poseidon_2": failure})
    assert len(rows) == 1
    row = rows[0]
    assert row["source_filename"] == _FAKE_BASENAME
    assert _FAKE_ABSOLUTE_DIR not in row["error_message"]
    assert _FAKE_BASENAME in row["error_message"]  # basename preserved, path scrubbed


def test_failed_well_issues_row_sanitizes_path_and_message():
    failure = _make_failure()
    rows = build_deviation_issues_rows({}, {"Poseidon_2": failure})
    assert len(rows) == 1
    row = rows[0]
    assert row["source_filename"] == _FAKE_BASENAME
    assert row["context"] == _FAKE_BASENAME
    assert _FAKE_ABSOLUTE_DIR not in row["message"]
    assert _FAKE_BASENAME in row["message"]


def test_failed_well_manifest_sanitizes_message():
    failure = _make_failure()
    manifest = build_deviation_depth_manifest({}, {"Poseidon_2": failure}, {})
    assert "Poseidon_2" in manifest["failed_wells"]
    entry = manifest["failed_wells"]["Poseidon_2"]
    assert _FAKE_ABSOLUTE_DIR not in entry["message"]
    assert _FAKE_BASENAME in entry["message"]


#### Step 8 — Install the package in editable mode

**Technical objective:** (re)install `p2mem` from the just-written source tree so the notebook's Python kernel imports the exact code just written, not a stale cached version.

**Note (Increment 3.1.1):** the `%cd` below is a defensive second confirmation of the working directory already entered and verified in Step 2b, retained because it is harmless (re-entering a directory the kernel is already in) and because `pip install -e .` must run with `PROJECT_ROOT` as the working directory regardless. It is not, and has not been since Step 2b was added, the first working-directory change in this notebook.

In [ ]:
%cd /content/drive/MyDrive/Poseidon_1D_MEM
!pip install -q -e .

#### Step 9 — Run the complete unit-test suite (Increment 1.1 + 2.1.1 + 3 together)

**Technical objective:** confirm every existing test still passes (nothing in the locked foundation was weakened) and every new Increment 3 test passes, in one combined run. The actual reported pass count is read from this cell's own output - never assumed or hard-coded ahead of time.

In [ ]:
!pytest -v

### Minimum-Curvature Method

#### Step 10 — Inspect the actual deviation-survey headers of all four wells, and validate their per-file contracts

**Technical objective:** parse each file's header block ONLY (no station data yet) and print every extracted field, so the header content driving every later step is visible and auditable, not assumed. Then validate each file's actual header/data against its contract, BEFORE computing any trajectory.

**Governing logic:** `parse_deviation_header` (structural parsing) and `resolve_deviation_contract` (contract comparison) - see `p2mem/io/deviation.py`.

**Failure behavior:** a structural defect raises `DeviationParsingError`; a contract mismatch raises `DeviationContractError` naming every ERROR-severity issue found (not just the first). Neither is caught here - a real per-file problem should stop this notebook loudly, not be silently skipped.

**Expected result:** all four wells' headers parse cleanly and their contracts resolve with zero ERROR-severity issues (one WARNING - the disclosed MD-unit inference - is expected and printed, not hidden).

In [ ]:
from p2mem.io.deviation import (
    load_deviation_contract_config,
    parse_deviation_header,
    read_deviation_stations,
    resolve_deviation_contract,
)

DEV_CONTRACT_PATH = os.path.join(PROJECT_ROOT, "config", "deviation_survey_contracts.yml")
dev_contracts = load_deviation_contract_config(DEV_CONTRACT_PATH)
print(f"Loaded {len(dev_contracts)} deviation-survey contract(s): {list(dev_contracts)}")

for key, fn in DEV_FILES.items():
    path = os.path.join(DEV_DIR, fn)
    header, data_start, cols = parse_deviation_header(path)
    stations = read_deviation_stations(path, data_start, cols)
    issues = resolve_deviation_contract(header, cols, stations, dev_contracts[fn])
    errors = [i for i in issues if i.severity == "ERROR"]
    warnings = [i for i in issues if i.severity == "WARNING"]
    print(f"\n=== {key} ({fn}) ===")
    print(f"  Well name: {header.well_name!r}  Survey: {header.survey_name!r}  Type: {header.well_type!r}")
    print(f"  Wellhead: X={header.wellhead_x_m}, Y={header.wellhead_y_m}  Datum: {header.datum_elevation_m} m ({header.datum_reference})")
    print(f"  CRS: {header.coordinate_reference_system}")
    print(f"  Columns ({len(cols)}): {cols}")
    print(f"  Stations: {stations.MD_source_m.size}  MD range: [{stations.MD_source_m.min():.6f}, {stations.MD_source_m.max():.4f}] m")
    print(f"  Max inclination: {stations.INCL_source_deg.max():.4f} deg")
    print(f"  Contract check: {len(errors)} ERROR(s), {len(warnings)} WARNING(s)")
    for w in warnings:
        print(f"    [WARNING] {w.code}: {w.message}")
    if errors:
        for e in errors:
            print(f"    [ERROR] {e.code}: {e.message}")
        raise RuntimeError(f"{key}: contract resolution failed with {len(errors)} ERROR(s).")
print("\nAll four deviation-survey contracts resolved with zero ERRORs.")

#### Step 11 — Load all four wells (full ingestion + minimum-curvature trajectory + residual validation)

**Technical objective:** run the complete per-well pipeline - parse, contract-resolve, compute the independent minimum-curvature trajectory (using `AZIM_GN`, the grid-coordinate azimuth reference declared in each well's contract), and compare it against the Petrel-supplied source trajectory.

**Governing logic:** `load_deviation_surveys` (`p2mem/io/deviation.py`) - isolates any expected per-well failure (file-not-found, parsing, contract, trajectory) as a typed `DeviationIngestionFailure`; an unexpected programming error still propagates uncaught.

**Expected result:** all four wells load successfully (zero failures); Poseidon 2, Boreas 1, and Poseidon North 1 show trajectory-validation `overall_status == "PASS"`; Proteus 1ST2 shows `overall_status == "WARNING"` - reported explicitly below, not suppressed. Each well is also expected to carry exactly two non-blocking WARNING-severity issues: the pre-existing `MD_UNIT_NOT_EXPLICITLY_DECLARED` and the Increment 3.1 `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` (see the Corrective patch note above) - both printed below with their actual per-well computed message, not summarized away.

In [ ]:
from p2mem.io.deviation import load_deviation_surveys

dev_file_paths = {key: os.path.join(DEV_DIR, fn) for key, fn in DEV_FILES.items()}
dev_results, dev_failures = load_deviation_surveys(dev_file_paths, dev_contracts)

print(f"Loaded successfully: {sorted(dev_results)}")
print(f"Failed: {sorted(dev_failures)}")
for key, failure in dev_failures.items():
    print(f"  {key}: [{failure.error_type}] {failure.message}")

print("\n=== Per-well ingestion issues (all WARNING-severity; zero ERRORs reached this point) ===")
for key in DEV_FILES:
    print(f"\n{key}:")
    for issue in dev_results[key].issues:
        print(f"  [{issue.severity}] {issue.code}: {issue.message}")

print("\n=== Trajectory validation (minimum curvature vs Petrel source) ===")
for key in DEV_FILES:
    v = dev_results[key].validation
    print(
        f"{key}: overall={v.overall_status} | "
        f"TVD max|res|={v.tvd_max_abs_residual_m:.6f} m ({v.tvd_status}, tol={v.tvd_tolerance_m} m) | "
        f"East max|res|={v.easting_max_abs_residual_m:.6f} m ({v.easting_status}) | "
        f"North max|res|={v.northing_max_abs_residual_m:.6f} m ({v.northing_status})"
    )
    print(f"    {v.origin_initialization_note}")

> **QUALITY-CONTROL NOTE:** Poseidon 2, Boreas 1, and Poseidon North 1 independently reconstruct their Petrel-supplied TVD to millimetre scale via minimum curvature - strong evidence the survey stations, azimuth-reference handling, and minimum-curvature implementation are all mutually consistent for those three wells. Proteus 1ST2 does **not** reach that level of agreement (see Step 12 below); this is reported, not smoothed over.

#### Step 12 — Investigate the Proteus 1ST2 trajectory discrepancy

**Technical objective:** characterize WHERE and HOW the Proteus 1ST2 discrepancy arises, using only evidence already computed above - never asserting a cause that isn't supported by the numbers actually produced.

**Method:** compare the file's own supplied `DLS` column (degrees per 30 m) against an independent recomputation of the same dogleg-severity metric from the same file's own inclination/azimuth columns (i.e. NOT the TVD/DX/DY residual - a completely separate consistency check on the angular data alone), station by station, alongside the accumulating TVD residual. The "degrees per 30 m" normalization used here is the same inferred-and-verified basis disclosed by the `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` warning every well raises on load (Step 11) - this cell's per-station comparison is a manual, well-specific deep-dive using the same underlying check, not a separate or conflicting computation.

**Expected result:** the supplied and recomputed DLS values agree closely at every station (confirming the inclination/azimuth data itself is internally consistent), while the TVD residual grows - concentrated below approximately MD 4200 m rather than uniformly across the whole well. This pattern (angularly consistent, but progressively diverging in accumulated position) is consistent with - but does not, on this evidence alone, prove - the possibility that this exported trace was resampled or interpolated to a coarser station density than whatever internal computation originally produced its TVD/X/Y/Z columns. No independent evidence is available in this project to confirm that specific mechanism, so it is reported as an *observed pattern*, not a proven cause.

In [ ]:
import numpy as np

proteus = dev_results["Proteus_1ST2"]
md_p = proteus.raw.MD_source_m
tvd_res_p = proteus.mc.tvd_mc_m - proteus.raw.TVD_source_m
dls_source_p = proteus.raw.DLS_source_deg_per_30m
dls_mc_p = proteus.mc.dls_deg_per_30m

dls_diff = np.abs(dls_source_p - dls_mc_p)
print(f"Proteus 1ST2: max |DLS_source - DLS_recomputed| across all {md_p.size} stations = {dls_diff.max():.6f} deg/30m")
print("  -> the file's own supplied DLS is essentially exactly reproducible from its own inclination/azimuth.")

deep_mask = md_p > 4200.0
print(f"\nTVD residual below MD 4200 m ({int(deep_mask.sum())} stations): "
      f"max={np.max(np.abs(tvd_res_p[deep_mask])):.5f} m")
print(f"TVD residual above MD 4200 m ({int((~deep_mask).sum())} stations): "
      f"max={np.max(np.abs(tvd_res_p[~deep_mask])):.5f} m")
print(f"\nOverall Proteus 1ST2 status: {proteus.validation.overall_status}  "
      f"(TVD max|res|={proteus.validation.tvd_max_abs_residual_m:.5f} m, "
      f"easting max|res|={proteus.validation.easting_max_abs_residual_m:.5f} m, "
      f"northing max|res|={proteus.validation.northing_max_abs_residual_m:.5f} m)")

> **INTERPRETATION:** the near-perfect DLS agreement combined with a TVD residual that grows almost entirely below MD ~4200 m indicates the discrepancy is not a defect in this project's minimum-curvature implementation (which reproduces the other three wells to millimetre scale using identical code) or in the parsing of Proteus 1ST2's angular data. It is consistent with a difference in how the Proteus 1ST2 trace itself was generated or exported upstream (e.g. a coarser resampling than whatever process originally produced its TVD/X/Y/Z columns) - but this project has no independent evidence (no access to the original Petrel project or its trace-generation settings) to confirm that specific mechanism, so no definitive cause is claimed.

> **LIMITATION:** the Proteus 1ST2 trajectory discrepancy (~0.19 m TVD, ~1.6 m easting, ~0.9 m northing at maximum) is an open, unresolved data-quality finding, not a validated or corrected trajectory. It is not adjudicated in this increment.

### Depth-Reference Convention

#### Step 13 — Select the downstream depth basis and map LAS MD to TVD/TVDSS

**Technical objective:** for every well, record which trajectory (`petrel_source_trace` or `minimum_curvature_computed`) is used as the downstream MD-to-TVD/TVDSS mapping basis (declared explicitly per well in `config/deviation_survey_contracts.yml`, uniformly `petrel_source_trace` for all four wells given the unresolved Proteus 1ST2 finding above), then map each well's locked Increment 2.1.1 LAS `MD_m` array onto TVD and TVDSS using that basis via deterministic piecewise-linear interpolation of the validated survey-station trajectory.

**Governing equations:** $TVDSS_m = TVD_m - DatumElevation_m$ (equivalently $-Z_m$) - see the Theory section above.

**Validation logic:** `map_las_md_to_tvd_tvdss` (`p2mem/depth_mapping.py`) first verifies every LAS MD sample lies within the well's surveyed MD coverage; any sample outside that range raises `ExtrapolationRejectedError` rather than silently extrapolating.

**Expected result:** all four wells' LAS MD coverage is confirmed fully inside its survey MD coverage (zero extrapolated samples), and TVD/TVDSS are computed at every LAS sample depth.

In [ ]:
from p2mem.io.las import load_file_contract_config as load_las_contracts, load_wells
from p2mem.depth_mapping import map_las_md_to_tvd_tvdss

LAS_CONTRACT_PATH = os.path.join(PROJECT_ROOT, "config", "las_curve_contracts.yml")
LOGS_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "logs")
las_contracts = load_las_contracts(LAS_CONTRACT_PATH)
las_file_paths = {key: os.path.join(LOGS_DIR, f"{key}_logs.las") for key in DEV_FILES}
las_results, las_errors = load_wells(las_file_paths, las_contracts)
print(f"LAS loaded: {sorted(las_results)}  LAS failed: {sorted(las_errors)}")

mappings = {}
for key in DEV_FILES:
    las_md = las_results[key].canonical_data["MD_m"]
    survey_md = dev_results[key].raw.MD_source_m
    inside = (las_md.min() >= survey_md.min()) and (las_md.max() <= survey_md.max())
    mappings[key] = map_las_md_to_tvd_tvdss(key, las_md, dev_results[key])
    m = mappings[key]
    print(
        f"{key}: basis={m.depth_basis_used}  LAS MD [{m.las_md_min_m:.4f}, {m.las_md_max_m:.4f}] "
        f"vs survey MD [{m.survey_md_min_m:.6f}, {m.survey_md_max_m:.4f}] -> "
        f"{'fully inside' if inside else 'OUT OF COVERAGE'}  n_extrapolated={m.n_extrapolated}"
    )
    print(f"    TVD @ final LAS MD = {m.tvd_mapped_m[-1]:.5f} m   TVDSS @ final LAS MD = {m.tvdss_mapped_m[-1]:.5f} m")

### Quality-Control Results

#### Step 14 — Generate the deterministic inventory outputs

**Technical objective:** write the six deterministic, metadata-only Increment 3 output files under `outputs/03_deviation_depth/` - never raw per-sample station or LAS arrays.

In [ ]:
import csv
import json

from p2mem.io.deviation_inventory import (
    build_deviation_depth_manifest,
    build_deviation_file_inventory_rows,
    build_deviation_issues_rows,
    build_depth_reference_register_rows,
    build_las_depth_mapping_rows,
    build_trajectory_validation_rows,
)

OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "03_deviation_depth")

def write_csv(path, rows, fieldnames=None):
    fieldnames = fieldnames or (list(rows[0].keys()) if rows else [])
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

write_csv(os.path.join(OUT_DIR, "deviation_file_inventory.csv"), build_deviation_file_inventory_rows(dev_results, dev_failures))
write_csv(os.path.join(OUT_DIR, "trajectory_validation_summary.csv"), build_trajectory_validation_rows(dev_results))
write_csv(os.path.join(OUT_DIR, "depth_reference_register.csv"), build_depth_reference_register_rows(dev_results))
write_csv(os.path.join(OUT_DIR, "las_depth_mapping_summary.csv"), build_las_depth_mapping_rows(mappings))
write_csv(
    os.path.join(OUT_DIR, "deviation_ingestion_issues.csv"),
    build_deviation_issues_rows(dev_results, dev_failures),
    fieldnames=["well_key", "source_filename", "severity", "code", "message", "context"],
)
manifest = build_deviation_depth_manifest(dev_results, dev_failures, mappings)
with open(os.path.join(OUT_DIR, "deviation_depth_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print("Outputs written to:", OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, fn)
    if os.path.isfile(fp):
        print(" -", fn, os.path.getsize(fp), "bytes")

#### Step 15 — Display concise summary tables

**Technical objective:** display the trajectory-validation summary and LAS depth-mapping summary directly in the notebook, so the key numbers are visible without opening the CSV files.

In [ ]:
import pandas as pd

df_validation = pd.read_csv(os.path.join(OUT_DIR, "trajectory_validation_summary.csv"))
display(df_validation[[
    "well_key", "tvd_max_abs_residual_m", "tvd_tolerance_m", "tvd_status",
    "easting_max_abs_residual_m", "northing_max_abs_residual_m", "overall_status",
]])

df_mapping = pd.read_csv(os.path.join(OUT_DIR, "las_depth_mapping_summary.csv"))
display(df_mapping[[
    "well_key", "depth_basis_used", "n_samples", "survey_md_min_m", "survey_md_max_m",
    "las_md_min_m", "las_md_max_m", "n_extrapolated", "tvd_at_final_las_md_m", "tvdss_at_final_las_md_m",
]])

#### Step 16 — Generate professional QC figures

**Technical objective:** produce five portfolio-quality QC figures (plan view, vertical sections, MD-vs-TVD, minimum-curvature-vs-Petrel residuals, and mapped LAS depth coverage), each with SI units, a clear well legend, source and computed trajectories visually distinct where both are shown, and the Proteus 1ST2 discrepancy visibly shown rather than hidden.

**Failure behavior:** none of these figures fabricate geology, formations, pressure, or lithology - they plot only quantities already computed above.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figures")
COLORS = {
    "Poseidon_2": "#1b6ca8", "Boreas_1": "#2f9e44",
    "Poseidon_North_1": "#e8590c", "Proteus_1ST2": "#c92a2a",
}
FOOTER = "Tier C - Screening-Level / Uncalibrated Educational 1D MEM (Poseidon 2 portfolio project)"

def _footer(fig):
    fig.text(0.01, 0.01, FOOTER, fontsize=7, color="#555555", ha="left", va="bottom")

# Figure 1: four-well plan view
fig, ax = plt.subplots(figsize=(8, 8))
for key in DEV_FILES:
    r = dev_results[key]
    x = r.header.wellhead_x_m + r.raw.DX_source_m
    y = r.header.wellhead_y_m + r.raw.DY_source_m
    ax.plot(x, y, color=COLORS[key], label=key.replace("_", " "), lw=1.6)
    ax.scatter([r.header.wellhead_x_m], [r.header.wellhead_y_m], color=COLORS[key], marker="^", s=60, zorder=5)
ax.set_xlabel("Easting, X (m, GDA94 / MGA Zone 51)")
ax.set_ylabel("Northing, Y (m, GDA94 / MGA Zone 51)")
ax.set_title("Four-Well Plan View (Petrel-Supplied Source Trajectory)")
ax.set_aspect("equal", adjustable="datalim")
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=9)
_footer(fig)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(os.path.join(FIG_DIR, "fig01_plan_view.png"), dpi=150)
plt.show()

# Figure 2: vertical trajectory sections
fig, axes = plt.subplots(1, 4, figsize=(18, 6), sharey=True)
for ax, key in zip(axes, DEV_FILES):
    r = dev_results[key]
    horiz = (r.raw.DX_source_m**2 + r.raw.DY_source_m**2) ** 0.5
    ax.plot(horiz, r.raw.TVD_source_m, color=COLORS[key], lw=1.8)
    ax.set_title(key.replace("_", " "))
    ax.set_xlabel("Horizontal displacement (m)")
    ax.invert_yaxis()
    ax.grid(alpha=0.3)
axes[0].set_ylabel("TVD (m, below well datum)")
fig.suptitle("Vertical Trajectory Sections (Petrel-Supplied Source TVD)")
_footer(fig)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.savefig(os.path.join(FIG_DIR, "fig02_vertical_sections.png"), dpi=150)
plt.show()

# Figure 3: MD vs TVD and MD-minus-TVD
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for key in DEV_FILES:
    r = dev_results[key]
    axes[0].plot(r.raw.MD_source_m, r.raw.TVD_source_m, color=COLORS[key], label=key.replace("_", " "), lw=1.6)
    axes[1].plot(r.raw.MD_source_m, r.raw.MD_source_m - r.raw.TVD_source_m, color=COLORS[key], lw=1.6)
axes[0].set_xlabel("MD (m)"); axes[0].set_ylabel("TVD (m)"); axes[0].set_title("MD vs TVD")
axes[0].invert_yaxis(); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
axes[1].set_xlabel("MD (m)"); axes[1].set_ylabel("MD - TVD (m)"); axes[1].set_title("MD minus TVD (departure from vertical)")
axes[1].grid(alpha=0.3)
_footer(fig)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(os.path.join(FIG_DIR, "fig03_md_vs_tvd.png"), dpi=150)
plt.show()

# Figure 4: minimum-curvature vs Petrel residual comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, key in zip(axes.flat, DEV_FILES):
    r = dev_results[key]
    tvd_res = r.mc.tvd_mc_m - r.raw.TVD_source_m
    ax.plot(r.raw.MD_source_m, tvd_res, color=COLORS[key], lw=1.4, label="TVD residual (mc - source)")
    ax.axhline(r.contract.residual_tolerance_tvd_m, color="gray", ls="--", lw=0.8, label="pass tolerance")
    ax.axhline(-r.contract.residual_tolerance_tvd_m, color="gray", ls="--", lw=0.8)
    ax.set_title(f"{key.replace('_', ' ')} - TVD residual (status: {r.validation.tvd_status})")
    ax.set_xlabel("MD (m)"); ax.set_ylabel("TVD residual (m)")
    ax.grid(alpha=0.3); ax.legend(fontsize=7, loc="best")
fig.suptitle("Minimum-Curvature vs Petrel-Supplied TVD Residual (Proteus 1ST2 discrepancy shown, not hidden)")
_footer(fig)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.savefig(os.path.join(FIG_DIR, "fig04_mc_vs_petrel_residuals.png"), dpi=150)
plt.show()

# Figure 5: mapped LAS MD-TVD/TVDSS coverage
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for key in DEV_FILES:
    m = mappings[key]
    axes[0].plot(m.las_md_source_m, m.tvd_mapped_m, color=COLORS[key], lw=1.2, label=key.replace("_", " "))
    axes[1].plot(m.las_md_source_m, m.tvdss_mapped_m, color=COLORS[key], lw=1.2, label=key.replace("_", " "))
for ax, ylabel, title in zip(axes, ["Mapped TVD (m)", "Mapped TVDSS (m)"], ["LAS MD -> TVD coverage", "LAS MD -> TVDSS coverage"]):
    ax.set_xlabel("LAS MD (m)"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.invert_yaxis(); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle("Mapped LAS MD-TVD/TVDSS Coverage (piecewise-linear station interpolation, no extrapolation)")
_footer(fig)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.savefig(os.path.join(FIG_DIR, "fig05_las_depth_mapping_coverage.png"), dpi=150)
plt.show()

print("Figures written to:", FIG_DIR)

#### Step 17 — Integration regression checks

**Technical objective:** independently re-verify, against the code and data actually run in this notebook, every regression figure this increment was designed against - never assumed, always recomputed here.

In [ ]:
EXPECTED_STATIONS = {"Poseidon_2": 124, "Boreas_1": 134, "Poseidon_North_1": 147, "Proteus_1ST2": 155}
print("=== INTEGRATION REGRESSION CHECKS (Increment 3) ===")
for key, expected in EXPECTED_STATIONS.items():
    actual = dev_results[key].raw.MD_source_m.size
    print(f"{key}: {actual} stations (expected {expected}) -> {'MATCH' if actual == expected else 'MISMATCH'}")

total_stations = sum(dev_results[k].raw.MD_source_m.size for k in DEV_FILES)
print(f"Total stations across 4 wells: {total_stations} (expected 560) -> {'MATCH' if total_stations == 560 else 'MISMATCH'}")
print(f"Contracts passed: {sum(1 for k in DEV_FILES if dev_results[k].contract_status == 'PASSED')}/4")
print("Trajectory validation status:", {k: dev_results[k].validation.overall_status for k in DEV_FILES})
print(f"Total LAS extrapolated samples (expected 0): {sum(mappings[k].n_extrapolated for k in DEV_FILES)}")
for key in DEV_FILES:
    inside = (mappings[key].las_md_min_m >= mappings[key].survey_md_min_m) and (mappings[key].las_md_max_m <= mappings[key].survey_md_max_m)
    print(f"{key}: LAS MD fully inside survey MD coverage -> {inside}")

### Interpretation

Three of the four wells (Poseidon 2, Boreas 1, Poseidon North 1) show excellent (millimetre-scale) agreement between the Petrel-supplied source trajectory and this project's independently implemented minimum-curvature computation - strong, independent evidence that both the survey-station data and this project's trajectory engine are correct and mutually consistent for those wells. Proteus 1ST2 does not reach that agreement in its deeper section; the evidence gathered (DLS self-consistency intact, TVD residual concentrated below ~MD 4200 m) points toward a trace-generation or resampling difference upstream of this project's own code, but this is reported as an observed, unresolved pattern - not a proven root cause, and not something this increment corrects.

### Limitations

- This increment's minimum-curvature engine and depth-mapping layer are, like the rest of this project, screening-level and uncalibrated - no independently surveyed check-shot or gyro re-run exists to adjudicate the Proteus 1ST2 discrepancy.
- The MD-unit-not-explicitly-declared inference (documented as a WARNING on every well) is disclosed, not eliminated - no header line in any of the four files states MD's unit directly.
- `depth_basis_policy: petrel_source_trace` is a conservative, auditable default given the unresolved Proteus 1ST2 finding, not a claim that the Petrel source trajectory is independently validated as correct.
- Downstream phases (checkshot/time-depth conversion, formation-top correction, petrophysics, pore pressure, elastic properties, rock strength, stresses, wellbore stability) all remain explicitly out of scope and unimplemented.

### Completion Gate

> **PHASE COMPLETION GATE:** Increment 3 (Deviation-Survey Ingestion, Minimum-Curvature Validation, and MD-TVD-TVDSS Depth Framework) is complete when: (1) all four deviation-survey files load with zero contract-resolution ERRORs; (2) the combined test suite passes in full; (3) three of the four wells show trajectory-validation PASS and the fourth (Proteus 1ST2) shows its WARNING explicitly, not hidden; (4) LAS MD is confirmed fully inside survey MD coverage for all four wells with zero extrapolated samples; (5) all six deterministic output files and five QC figures are generated. All five conditions are verified programmatically below, not asserted.

In [ ]:
print("=" * 78)
print("INCREMENT 3 COMPLETION GATE")
print("=" * 78)

gate_checks = {
    "All 4 deviation files loaded (0 failures)": len(dev_failures) == 0,
    "All 4 contracts PASSED": all(dev_results[k].contract_status == "PASSED" for k in DEV_FILES),
    "3 wells PASS, Proteus_1ST2 WARNING (disclosed, not hidden)": (
        dev_results["Poseidon_2"].validation.overall_status == "PASS"
        and dev_results["Boreas_1"].validation.overall_status == "PASS"
        and dev_results["Poseidon_North_1"].validation.overall_status == "PASS"
        and dev_results["Proteus_1ST2"].validation.overall_status == "WARNING"
    ),
    "LAS MD fully inside survey MD coverage, 0 extrapolated (all 4 wells)": (
        sum(mappings[k].n_extrapolated for k in DEV_FILES) == 0
    ),
    "6 deterministic output files present": all(
        os.path.exists(os.path.join(OUT_DIR, fn)) for fn in [
            "deviation_file_inventory.csv", "trajectory_validation_summary.csv",
            "depth_reference_register.csv", "las_depth_mapping_summary.csv",
            "deviation_ingestion_issues.csv", "deviation_depth_manifest.json",
        ]
    ),
    "5 QC figures present": all(
        os.path.exists(os.path.join(FIG_DIR, fn)) for fn in [
            "fig01_plan_view.png", "fig02_vertical_sections.png", "fig03_md_vs_tvd.png",
            "fig04_mc_vs_petrel_residuals.png", "fig05_las_depth_mapping_coverage.png",
        ]
    ),
}
for label, passed in gate_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")

if all(gate_checks.values()):
    print("\nIncrement 3 is complete. Stopping here per the approved scope.")
    print("Not implemented (explicitly out of scope): checkshot processing, formation-top")
    print("correction, petrophysics, pore pressure, elastic properties, rock strength,")
    print("stresses, wellbore stability. Increment 4 has NOT been started.")
else:
    raise RuntimeError("Increment 3 completion gate FAILED - see failed check(s) above.")